In [1]:
from pathlib import Path
import os
print(os.getcwd())
print('\n'.join(str(p) for p in Path('/workspace').iterdir()))

/workspace
/workspace/.config
/workspace/funders_audit_sample.csv
/workspace/deal_rules.md
/workspace/.kernel_llm_logs_1.jsonl
/workspace/memory
/workspace/lost+found
/workspace/.skills
/workspace/.prompts


In [2]:
print(Path('deal_rules.md').read_text())
print(Path('funders_audit_sample.csv').read_text())
print(Path('.prompts/MEMORY.md').read_text())

# Who funded each venture: research rules for a deal pass

Part of issue #380 (see `PROTOCOL.md`). Each pass gets a list of open-source-minded ventures. For each venture, find
**every disclosed outside funding event and who put the money in**, with a source URL for each. This is about the
names of the funders, more than the amounts.

## For each venture, record

- `open_basis`: what is open and under which licence, ≤ 20 words (e.g. "Grafana, AGPLv3 since 2021"). For U1 rows
  the basis is already given; copy it. For U2/U3 rows check it; if the main product turns out not to be open, say
  so here and set `open_ok` to `no`.
- `open_url`: the repository, licence or design-files page that shows it (U2/U3 only; optional for U1).
- `country`, `founded` (year), `status` (`private`, `acquired`, `listed`, `shut_down`, `nonprofit`, `unknown`).
- `funding_found`: `yes` if any outside funding was found; `none_found` if you looked and found none;
  `bootstrapped_stated` if the company says it took 

In [3]:
import pandas as pd, json, asyncio, datetime
from e14c.subagents import submit_task, check_task
sample=pd.read_csv('funders_audit_sample.csv').fillna('')
Path('audit_work').mkdir(exist_ok=True)
groups=[sample.iloc[i:i+5] for i in range(0,40,5)]
prompt='''Independently research disclosed outside funding events for ONLY the ventures below. Read /workspace/deal_rules.md and /workspace/funders_audit_sample.csv. User needs a funding audit, not licensing or company metadata. Search web freshly (no facts from memory); about 3-6 searches/fetches each, more for many rounds. Include ALL disclosed events: seed, angel, accelerator, series, grants, crowdfunding, token, debt, secondary etc. One row per event with columns exactly key,venture,type,label,date,amount_usd,lead,investors,source_url,confidence. Types from rules, or none_found only after reasonable searches. Date YYYY-MM; blank when month unknown, explain in notes. Amount digits USD, blank undisclosed; avoid unsubstantiated exchange conversion. People 'Full Name (person)', firms usual names, lead excluded from investors, separators ' | '. Each event source must actually name investors or amount. Prefer company/investor/filings/reputable news; fetch sources, don't infer investors across rounds. Acquisition/IPO not ordinary funding; include IPO only if outside capital issued and clear amount. Keep distinct secondary deals; don't duplicate total cumulative amounts as separate rounds. If no funding, none_found row and note search scope; distinguish bootstrapped if sourced. Need one line per venture on searches and uncertainties. Explicitly report which ventures NOT fully researched (reasonable search with none found counts researched; unresolved material event histories count incomplete). Save CSV to /workspace/audit_work/group{n}.csv and notes JSON to /workspace/audit_work/group{n}_notes.json, JSON list of objects with key, note, incomplete(boolean), sources(list URLs checked). Save full BibTeX for EVERY used web source using e14c.web.save_citation (load citation-integrity skill). Persist search/fetch evidence in /workspace/audit_work/group{n}_evidence.json or .md. Do NOT overwrite root .deliverables.json or other groups files. Final answer short with local file paths and incomplete count. You have ~35 min; prioritize full coverage and trustworthy sources. Ventures:\n{rows}'''
async def launch(n,g):
    return await submit_task(prompt.format(n=n,rows=g.to_csv(index=False)),agent_type='data-retrieval',project_id=os.environ.get('PROJECT_ID'))
task_ids=await asyncio.gather(*(launch(i+1,g) for i,g in enumerate(groups)))
print(list(enumerate(task_ids,1)))
Path('audit_work/tasks.json').write_text(json.dumps(dict(enumerate(task_ids,1)),indent=2))

[(1, '84dea63a-6319-43c0-a0ff-0a9d45ebc0ba'), (2, 'd837b2d6-cfbe-4d5f-96db-7a65294f01ba'), (3, 'b167ec30-2689-496e-b683-3d6e527b2778'), (4, 'cb65c732-d0ce-4369-860f-29e5528ac62b'), (5, 'f1d49db0-fc7f-4bb4-885e-8c31cc9553f8'), (6, '78387c12-5338-4c51-b039-365853670889'), (7, '65955bd0-9605-41d7-80dd-9e036c8f6d8f'), (8, '09c9f518-f5d8-4501-9671-a3f2ada07c89')]


378

In [4]:
from e14c.web import web_search, save_citation, fetch_parsed_page
print(datetime.date.today())
r=await web_search('VA Linux funding 1998 1999 Sequoia Intel series investors',num_results=6)
print(json.dumps(r,indent=2)[:25000])

2026-11-26


[
  {
    "title": "Books: VA Linux",
    "url": "https://edwardbetts.com/monograph/VA_Linux",
    "publishedDate": null,
    "full_text": "Books: VA Linux\n\n# VA Linux\n\n Rebel Code: Linux and the Open Source Revolution by Glyn Moody \n\n barriers to entry, business logic, commoditize, Compatible Time-Sharing System, Debian, Dennis Ritchie, Donald Knuth, Eben Moglen, Free Software Foundation, ghettoisation, Guido van Rossum, history of Unix, hypertext link, Johann Wolfgang von Goethe, John Gilmore, Ken Thompson, Kickstarter, Larry Ellison, Larry Wall, Marc Andreessen, MITM: man-in-the-middle, Multics, Network effects, new economy, packet switching, RFC: Request For Comment, Richard Stallman, Silicon Valley, skunkworks, slashdot, SoftBank, Steve Ballmer, Steve Jobs, Steven Levy, the Cathedral and the Bazaar, thinkpad, VA Linux \n\n The four leading distributions were not the only established players in the GNU/Linux world who benefited from the powerful backing of mainstream computer

In [5]:
own_evidence=[]
def escbib(s):
    return str(s).replace('&',r'\&').replace('%',r'\%').replace('_',r'\_').replace('#',r'\#').replace('{','').replace('}','')
def retain(result,author=None):
    url=result['url']; title=result.get('title',url); author=author or url.split('/')[2]
    key='audit'+__import__('hashlib').md5(url.encode()).hexdigest()[:12]
    date=(result.get('publishedDate') or '')[:10]
    bib='@online{'+key+',\n author = {{'+escbib(author)+'}},\n title = {'+escbib(title)+'},\n url = {'+url+'},\n urldate = {'+str(datetime.date.today())+'},\n'+(' date = {'+date+'},\n' if date else '')+'}\n'
    save_citation(bib)
    own_evidence.append(result)
    Path('audit_work/independent_evidence.json').write_text(json.dumps(own_evidence,indent=2))
    return key
print([(x['title'],x['url']) for x in r])

[('Books: VA Linux', 'https://edwardbetts.com/monograph/VA_Linux'), ('Dell plus Sun equals VA Research', 'https://www.forbes.com/1999/05/03/feat.html'), ('VA Linux Systems', 'https://www.vasoftware.com/about/aboutus_pr_all.php'), ('PRNewswire: VA Linux Systems Secures $25 Million in Equity Funding | Linux Today', 'https://www.linuxtoday.com/it-management/prnewswire-va-linux-systems-secures-25-million-in-equity-funding/'), ('Sequoia Capital invests in Linux start-up VA Research | Linux Today', 'https://www.linuxtoday.com/infrastructure/sequoia-capital-invests-in-linux-start-up-va-research/'), ('VA new investment telegraphs another Linux IPO?', 'https://www.theregister.com/on-prem/1999/06/22/va-new-investment-telegraphs-another-linux-ipo/898595')]


In [6]:
for x in r:
    if 'linuxtoday' in x['url'] or 'theregister' in x['url']:
        retain(x)
print([ (x['title'],x.get('full_text','')) for x in r if 'prnewswire' in x['url']])

[('PRNewswire: VA Linux Systems Secures $25 Million in Equity Funding | Linux Today', 'PRNewswire: VA Linux Systems Secures $25 Million in Equity Funding | Linux Today\n\n# PRNewswire: VA Linux Systems Secures $25 Million in Equity Funding\n\nWritten By\n\n Web Webster\n\n Jun 22, 1999 \n\n“Second Round of Financing Fuels Rapid Growth of Promising Linux Start-Up.”\n\n“VA Linux Systems… has completed a $25 million round of private equity funding in a second round of financing. Companies participating in this round of financing include the Capital Research Group, W.R. Hambrecht + Co., Intel Corporation, IDG Ventures, Lehman Brothers, Sequoia Capital, SGI, Sumitomo Corporation and Presidio Venture Partners.\n\n“Since its inception, VA has been leading the growing migration of computer users from Windows NT, NetWare and Unix operating systems to Linux, the award- winning open source operating system, by providing complete, pre-installed Linux computer systems and nationwide service and sup

In [7]:
rr=await web_search('"VA Research" "Intel" "February" "1999" investment',num_results=5)
for x in rr:
 print(x['title'],x['url'], '\n',x.get('full_text','')[:6000],'\n')

Books: VA Linux https://edwardbetts.com/monograph/VA_Linux 
 Books: VA Linux

# VA Linux

 Rebel Code: Linux and the Open Source Revolution by Glyn Moody 

 barriers to entry, business logic, commoditize, Compatible Time-Sharing System, Debian, Dennis Ritchie, Donald Knuth, Eben Moglen, Free Software Foundation, ghettoisation, Guido van Rossum, history of Unix, hypertext link, Johann Wolfgang von Goethe, John Gilmore, Ken Thompson, Kickstarter, Larry Ellison, Larry Wall, Marc Andreessen, MITM: man-in-the-middle, Multics, Network effects, new economy, packet switching, RFC: Request For Comment, Richard Stallman, Silicon Valley, skunkworks, slashdot, SoftBank, Steve Ballmer, Steve Jobs, Steven Levy, the Cathedral and the Bazaar, thinkpad, VA Linux 

 The four leading distributions were not the only established players in the GNU/Linux world who benefited from the powerful backing of mainstream computer giants and the financial markets during 1998 and 1999. Another was the GNU/Linux hardw

In [8]:
print([(x['title'],x['url']) for x in rr])
statuses=await asyncio.gather(*(check_task(t) for t in task_ids))
print([(i+1,s.status,s.runtime_minutes) for i,s in enumerate(statuses)])

[('Books: VA Linux', 'https://edwardbetts.com/monograph/VA_Linux'), ('', 'https://condor.depaul.edu/elliott/513/projects-archive/DS420Spring99/California/Documents/133.htm'), ('Preview - Los Angeles Times', 'https://www.latimes.com/archives/la-xpm-1999-mar-01-fi-12890-story.html'), ('VA Linux Systems', 'https://www.vasoftware.com/about/aboutus_pr_all.php'), ('In Context: Follow the Money; Cathedral Investments in the Bazaar | Linux Today', 'https://www.linuxtoday.com/it-management/in-context-follow-the-money-cathedral-investments-in-the-bazaar/')]


Task 65955bd0-9605-41d7-80dd-9e036c8f6d8f still running (status: in progress). Call check_task(task_id) again later.
Task d837b2d6-cfbe-4d5f-96db-7a65294f01ba still running (status: in progress). Call check_task(task_id) again later.
Task f1d49db0-fc7f-4bb4-885e-8c31cc9553f8 still running (status: in progress). Call check_task(task_id) again later.
Task b167ec30-2689-496e-b683-3d6e527b2778 still running (status: in progress). Call check_task(task_id) again later.
Task 09c9f518-f5d8-4501-9671-a3f2ada07c89 still running (status: in progress). Call check_task(task_id) again later.
Task cb65c732-d0ce-4369-860f-29e5528ac62b still running (status: in progress). Call check_task(task_id) again later.
Task 84dea63a-6319-43c0-a0ff-0a9d45ebc0ba still running (status: in progress). Call check_task(task_id) again later.
Task 78387c12-5338-4c51-b039-365853670889 still running (status: in progress). Call check_task(task_id) again later.
[(1, 'in progress', None), (2, 'in progress', 0.3708265333333333

In [9]:
intel_old=await fetch_parsed_page('https://condor.depaul.edu/elliott/513/projects-archive/DS420Spring99/California/Documents/133.htm')
print(intel_old[:17000])

| Intel To Invest In Linux PC Vendor VA Research Sunday February 28 1:42 PM ET By Therese Poletti SAN FRANCISCO (Reuters) - VA Research Inc., a developer of Intel-based PCs, servers and workstations running the increasingly popular Linux operating system, is expected to announce Monday that it will receive an investment from chip giant Intel Corp. (Nasdaq:INTC - news), according to industry sources. The investment, which will also include some venture capitalists, is Intel's second in a Linux-related startup company. The announcement will be made the day before Linux World -- the first big trade show devoted to Linux -- begins. Linux is an alternative version of the UNIX operating system which runs on Intel-based systems and on other computer architectures such as Compaq Computer Corp. (NYSE:CPQ - news)'s Alpha chip. Developed by Finnish programmer Linus Torvalds in 1991, the software is maintained by a group of far-flung programmers and given away over the Internet. While the number o

In [10]:
q=await web_search('"VA Research" "March 1, 1999" investment',num_results=5)
for x in q: print(x['title'],x['url'],'\n',x.get('full_text','')[:6000],'\n')

Intel Expected to Buy Stake in Linux Firm http://www.nytimes.com/library/tech/99/03/biztech/articles/01linux.html 
 Intel Expected to Buy Stake in Linux Firm

> > ##### March 1, 1999
> > 
> > 
> > 
> > ## Intel Expected to Buy Stake in Linux Firm
> > 
> > ##### By THE ASSOCIATED PRESS 
> > 
> > EW YORK -- Computer chip giant Intel Corp. is expected to be the latest company to announce its endorsement of Linux, a well-regarded version of the Unix operating system used by businesses to run computer networks and Internet sites. 
> > 
> > In a move that could be announced as soon as today, Intel is buying a multimillion dollar stake in VA Research, a Mountain View, Calif.-based maker of business computers that run on Linux, industry sources said. 
> > 
> > Officials at both the Santa Clara, Calif.-based Intel and VA Research declined comment. 
> > 
> > Linux was developed in the early 1990s by a Finnish student named Linus Torvalds. The grassroots software is edging into the mainstream as 

In [11]:
cnet=await fetch_parsed_page('https://condor.depaul.edu/elliott/513/projects-archive/DS420Spring99/California/Documents/136.htm')
print(cnet[:11000])

| Intel throws weight behind Linux By Stephen Shankland Staff Writer, CNET News.com March 1, 1999, 4:40 a.m. PT update VA Research is receiving an investment from Intel and has acquired the coveted Linux.com domain name for use as a portal site dedicated to the Unix-like operating system, industry sources said. The announcements are expected to occur this week at the LinuxWorld Conference and Expo in San Jose, California. VA is expected to help get Linux running on Merced, the first chip based on Intel's next-generation IA-64 architecture, sources said. Intel's Linux-related investment, on the heels of others, furthers the chipmaker's support of operating systems other than those developed by Microsoft. In a related announcement, database maker Oracle plans to make an unspecified equity investment in Red Hat Software, according to the Wall Street Journal. Red Hat sells the operating system along with customer support. In another boost for Linux, Silicon Graphics, Hewlett-Packard, and o

In [12]:
extra_prompt='''Supplemental audit, not main funding research: inspect /workspace/funders_audit_sample.csv and /workspace/deal_rules.md. Search specifically for grants/government/philanthropic awards/prizes and crowdfunding/token funding of sample ventures, particularly Antmicro, Bitcraze, Bela (Augmented Instruments), BCN3D, Efabless, FarmBot, Open Bionics, ResearchHub, neuromore, Coqui. Other agents research general venture rounds, so focus non-VC funding often missed. Do not include universities' own research grants as company funding unless the legal venture is recipient. Exclude preorder sales unless identifiable crowdfunding campaign. Build /workspace/audit_work/supplement_nonvc.csv same columns key,venture,type,label,date,amount_usd,lead,investors,source_url,confidence and /workspace/audit_work/supplement_nonvc_notes.md. Every included event needs a fetched source URL naming amount or money funder. No memory. Save BibTeX for all used sources with e14c.web.save_citation. Explicitly distinguish aggregate multi-year awards vs individual events and direct venture vs academic partner. Keep foreign amounts blank USD if cannot source conversion, but retain exact original amount and currency in notes. Do not write root manifest. Spend ~15-20 min. Output local paths and key caveats.'''
extra_task=await submit_task(extra_prompt,agent_type='data-retrieval',project_id=os.environ.get('PROJECT_ID'))
print(extra_task)
Path('audit_work/supplement_task.txt').write_text(extra_task)

5b9368af-2a81-4e85-98cb-9fca8dbf03fc


36

In [13]:
recent=await web_search('"RadixArk" funding investors SGLang',num_results=6)
for x in recent: print(x['title'],x['url'], '\n',x.get('full_text','')[:7500],'\n')

Our $100M Seed to Build Open Infrastructure for Frontier AI | RadixArk Blog https://www.radixark.com/blog/radixark-launches-100m-seed 
 Our $100M Seed to Build Open Infrastructure for Frontier AI | RadixArk Blog

May 5, 2026

# Our $100M Seed to Build Open Infrastructure for Frontier AI

Today, we are officially launching RadixArk with $100 million in Seed funding at a $400 million post-money valuation. The round was led by Accel and co-led by Spark Capital, with participation from NVentures, Salience Capital, A&E Investment, HOF Capital, Walden Catalyst, AMD, LDVP, WTT Fubon Family, MediaTek, Databricks and more.

## Our Mission: Making AI infra open and accessible

AI has moved quickly, but the infrastructure behind it has not yet been built the way it should be.

Frontier labs have some of the strongest internal systems in the world. But even there, infrastructure is often built under the pressure of delivering the next model, supporting the next architecture, or scaling the next wo

In [14]:
sem=asyncio.Semaphore(4)
async def latest_check(row):
    async with sem:
        name=row['venture']
        query=f'"{name.split(" (")[0]}" funding investment raised latest'
        try:
            results=await web_search(query,num_results=4)
            return {'key':row['key'],'query':query,'results':results}
        except Exception as exc:
            return {'key':row['key'],'query':query,'error':str(exc)}
latest_checks=await asyncio.gather(*(latest_check(row) for row in sample.to_dict('records')))
Path('audit_work/latest_checks.json').write_text(json.dumps(latest_checks,indent=2))
for x in latest_checks:
 print(x['key'],[(r['title'],(r.get('publishedDate') or '')[:10]) for r in x.get('results',[])])

BruceDevices [('Bruce Devices', ''), ('BruceDevices/firmware', ''), ('Browse products by Bruce Devices on Elecrow', ''), ('Contact\n – Bruce Devices', '')]
Cheshire-Labs [('Cheshire Labs Revenue, Funding & Valuation', ''), ('Cheshire Labs', ''), ('AI Lab Automation | Cheshire Labs', ''), ('Michael Salmi', '2026-11-19')]
coqui-ai [('Coqui - 2026 Funding Rounds & List of Investors - Tracxn', ''), ('Coqui - Products, Competitors, Financials, Employees, Headquarters Locations', ''), ('Coqui company information, funding & investors | Dealroom.co', ''), ('Coqui Asset Profile | Preqin | BlackRock', '')]
dbeaver [('DBeaver Funding: How Much Did They Raise & Key Investors', ''), ('DBeaver', ''), ('DBeaver', ''), ('DBeaver', '')]
e2b-dev [('E2B — Financials', '2025-08-28'), ('E2B', ''), ('Český startup E2B překročil hranici 120 milionů dolarů v anualizovaných tržbách', '2026-11-11'), ('E2B Raises a $21M Series A to Offer Cloud for AI Agents to Fortune 100 | BusinessPressWire', '2025-07-28')]
Fin

In [15]:
for x in latest_checks[18:30]:
 print('\n',x['key'])
 for rr2 in x.get('results',[]): print(rr2['title'],rr2['url'],'\n',rr2.get('snippet','')[:900])
print('files',[(p.name,p.stat().st_size) for p in Path('audit_work').iterdir()])


 d2lang
Terrastruct 2026 Company Profile: Valuation, Funding & Investors | PitchBook https://pitchbook.com/profiles/company/490983-04 
 # Terrastruct

This is a profile preview from the PitchBook Platform.

## Terrastruct Overview 

- Year Founded
- 2019

 

- Status
- Private

- Employees
- 3

 

- Latest Deal Type
- Seed

- Latest Deal Amount
- $2M

- Investors
- 3

...

Financing Status Venture Capital-Backed

...

## Terrastruct Valuation & Funding

| Deal Type | Date | Amount | Raised to Date | Post-Val | Status | Stage |
| --- | --- | --- | --- | --- | --- | --- |
| 1. Seed Round | 01-Dec-2021 | $2M | | | Completed | Generating Revenue |

...

| Investor Name | Investor Type | Holding | Investor Since | Participating Rounds |
| --- | --- | --- | --- | --- |
| Bloomberg Beta | Venture Capital | Minority | | |
| Fathom Capital | Venture Capital | Minority | | |
| Haystack Management Company | Venture Capital | Minority | | |

...

- ### How much funding has Terrastruct raised over

In [16]:
techaro=await web_search('"Techaro" funding grant investment Anubis NLnet',num_results=8)
for x in techaro: print(x['title'],x['url'],'\n',x.get('full_text','')[:4500],'\n')

TecharoHQ/anubis https://github.com/techaroHQ/anubis 
 # TecharoHQ/anubis

Weighs the soul of incoming HTTP requests to stop AI crawlers

- Stars: 24415
- Forks: 837
- Watchers: 24415
- Open issues: 425
- License: MIT License
- Homepage: https://anubis.techaro.lol/
- Default branch: main
- Created: 2025-03-17T17:35:28Z

## Languages

- BitBake
- CSS
- Dockerfile
- Go
- HCL
- HTML
- JavaScript
- Makefile
- Python
- Ruby
- Rust
- Shell
- TypeScript
- templ

## Topics

- defense
- security

## Top Contributors

- Xe (491 contributions)
- dependabot[bot] (99 contributions)
- JasonLovesDoggo (59 contributions)
- lotharsm (12 contributions)
- Earl0fPudding (12 contributions)
- SlyEcho (11 contributions)
- jvoisin (10 contributions)
- Dryusdan (9 contributions)
- p-linnane (7 contributions)
- dchandekstark (7 contributions)

---

## README

# Anubis

 
 
 

enbyware
GitHub Issues or Pull Requests by label
GitHub go.mod Go version
language count
repo size
GitHub Sponsors

## Sponsors

Anubis i

In [17]:
tea=await web_search('site:anubis.techaro.lol "Funding update"',num_results=5)
for x in tea: print(x['title'],x['url'],x.get('full_text','')[:9500])

feat(blog): add short funding update post https://github.com/TecharoHQ/anubis/pull/994 # feat(blog): add short funding update post

- State: merged
- Author: Xe
- Created: 2025-08-18T12:29:16Z
- Updated: 2025-08-18T12:42:28Z
- Repository: TecharoHQ/anubis
- Number: #994
- +62 -0 in 3 files
- Merged: 2025-08-18T12:42:28Z
- Merge commit: e8dfff635015d6c906dddd49cb0eaf591326092a
- Assignees: Xe

---

Submitted as a PR to run the spellcheck action.

## Timeline

- someone committed

- Xe was assigned

- Review by github-advanced-security[bot]: 

- someone committed

- Xe merged

- Xe closed

- Referenced by PR #236413: anubis 1.22.0
v1.22.0: Yda Hext https://github.com/TecharoHQ/anubis/releases/tag/v1.22.0 # v1.22.0: Yda Hext

- Tag: v1.22.0
- Repository: TecharoHQ/anubis
- Published: 2025-09-06T15:57:31Z
- Author: Xe

---

> Someone has to make an effort at reconciliation if these conflicts are ever going to end. 
 
In this release, we finally fix the odd number of CPU cores bug, pave the

In [18]:
techaro_url='https://anubis.techaro.lol/blog/2025/funding-update/'
try:
 techaro_page=await fetch_parsed_page(techaro_url)
 print(techaro_page[:12000])
except Exception as e: print(type(e).__name__,str(e)[:300])

Funding update | Anubis

[![A happy jackal woman with brown hair and red eyes](https://anubis.techaro.lol/img/favicon.webp)![A happy jackal woman with brown hair and red eyes](https://anubis.techaro.lol/img/favicon.webp) **Anubis**](https://anubis.techaro.lol/)[Blog](https://anubis.techaro.lol/blog)[Docs](https://anubis.techaro.lol/docs/)[Unbranded Version](https://anubis.techaro.lol/docs/admin/botstopper)

[GitHub](https://github.com/TecharoHQ/anubis)[Sponsor the Project](https://github.com/sponsors/Xe)

Recent posts

### 2026

* [It took a year to ship WebAssembly in Anubis](https://anubis.techaro.lol/blog/2026/anubis-wasm)
* [Anubis v1.27.0 is now available!](https://anubis.techaro.lol/blog/release/v1.27.0)

### 2025

* [Taking steps to end abusive traffic from cloud providers](https://anubis.techaro.lol/blog/2025/file-abuse-reports)
* [Sometimes CPU cores are odd](https://anubis.techaro.lol/blog/2025/cpu-core-odd)
* [Funding update](https://anubis.techaro.lol/blog/2025/funding-upda

In [19]:
retain({'url':techaro_url,'title':'Funding update','publishedDate':'2025-08-18','full_text':techaro_page},'Xe Iaso')
moretea=await web_search('"Techaro" "sponsors" funding',num_results=6)
for x in moretea: print(x['title'],x['url'],'\n',x.get('full_text','')[:4500],'\n')

TecharoHQ/anubis https://github.com/TecharoHQ/anubis?tab=readme-ov-file 
 # TecharoHQ/anubis

Weighs the soul of incoming HTTP requests to stop AI crawlers

- Stars: 24004
- Forks: 811
- Watchers: 24004
- Open issues: 412
- License: MIT License
- Homepage: https://anubis.techaro.lol/
- Default branch: main
- Created: 2025-03-17T17:35:28Z

## Languages

- BitBake
- CSS
- Dockerfile
- Go
- HCL
- HTML
- JavaScript
- Makefile
- Python
- Ruby
- Rust
- Shell
- TypeScript
- templ

## Topics

- defense
- security

## Top Contributors

- Xe (490 contributions)
- dependabot[bot] (99 contributions)
- JasonLovesDoggo (59 contributions)
- lotharsm (12 contributions)
- Earl0fPudding (12 contributions)
- SlyEcho (11 contributions)
- jvoisin (10 contributions)
- Dryusdan (9 contributions)
- p-linnane (7 contributions)
- dchandekstark (7 contributions)

---

## README

# Anubis

 
 
 

enbyware
GitHub Issues or Pull Requests by label
GitHub go.mod Go version
language count
repo size
GitHub Sponsors

##

In [20]:
statuses=await asyncio.gather(*(check_task(t) for t in task_ids))
print([(i+1,s.status,s.runtime_minutes) for i,s in enumerate(statuses)])
print([(p.name,p.stat().st_size) for p in Path('audit_work').iterdir()])

Task 84dea63a-6319-43c0-a0ff-0a9d45ebc0ba still running (status: in progress). Call check_task(task_id) again later.
Task 78387c12-5338-4c51-b039-365853670889 still running (status: in progress). Call check_task(task_id) again later.
Task f1d49db0-fc7f-4bb4-885e-8c31cc9553f8 still running (status: in progress). Call check_task(task_id) again later.
Task 65955bd0-9605-41d7-80dd-9e036c8f6d8f still running (status: in progress). Call check_task(task_id) again later.
Task b167ec30-2689-496e-b683-3d6e527b2778 still running (status: in progress). Call check_task(task_id) again later.
Task 09c9f518-f5d8-4501-9671-a3f2ada07c89 still running (status: in progress). Call check_task(task_id) again later.
Task cb65c732-d0ce-4369-860f-29e5528ac62b still running (status: in progress). Call check_task(task_id) again later.
Task d837b2d6-cfbe-4d5f-96db-7a65294f01ba still running (status: in progress). Call check_task(task_id) again later.
[(1, 'in progress', 2.1461689), (2, 'in progress', 2.28503723333

In [21]:
bruce=await web_search('"Bruce Devices" funding Kickstarter investment crowdfunding',num_results=6)
for x in bruce: print(x['title'],x['url'],'\n',x.get('full_text','')[:5000],'\n')

Bruce Devices https://github.com/BruceDevices 
 # Bruce Devices

- Login: BruceDevices
- Blog: bruce.computer
- Email: contact@bruce.computer
- Public repos: 17
- Followers: 747
- Verified: yes
- Created: 2025-04-23T18:49:56Z

## Top Repositories

- BruceDevices/firmware - Predatory ESP32 Firmware (7306 stars)
- BruceDevices/App - Bruce App for Predatory Firmware (239 stars)
- BruceDevices/bruce-js-tooling - The Bruce JavaScript tooling provides tools and type definitions to help you build applications for the Bruce platform. (39 stars)
- BruceDevices/App-IOS - IOS Application to interact with Bruce Devices (39 stars)
- BruceDevices/Wiki - Wiki for Bruce Devices (37 stars)
- BruceDevices/PCBs - (18 stars) 

BruceDevices/firmware https://github.com/BruceDevices/firmware/ 
 # BruceDevices/firmware

Predatory ESP32 Firmware

- Stars: 7306
- Forks: 2477
- Watchers: 7306
- Open issues: 271
- License: GNU Affero General Public License v3.0
- Homepage: https://bruce.computer
- Default branch:

In [22]:
db=await web_search('"DBeaver" "funding" investors seed round',num_results=6)
for x in db: print(x['title'],x['url'],'\n',x.get('full_text','')[:5000],'\n')

DBeaver's Next Step: our first round of funding https://dbeaver.com/2023/04/11/dbeavers-next-step-our-first-round-of-funding/ 
 DBeaver's Next Step: our first round of funding

# DBeaver’s Next Step: our first round of funding

 Apr 11, 2023 

Our team is happy to celebrate important milestones with you, and today we want to share that our company has raised $6 million in funding in the seed round led by Headline with additional investment from Prospective Technologies. It’s a good moment to take a small step back, look at the path DBeaver has already taken, and a little peek into future plans.

“We started small” – this phrase accurately describes our journey’s beginning. Back in 2011, DBeaver was just a pet project, millions of which never became recognizable enough. But we thought our idea could make life easier for many people, and we were not mistaken.

Years passed, DBeaver changed and grew, and we kept up with it. Since 2017, we have launched several enterprise products, and the

In [23]:
db2=await web_search('"DBeaver" "2021" "Prospective" investment',num_results=4)
for x in db2: print(x['title'],x['url'],'\n',x.get('full_text','')[:4500],'\n')

Prospective Technologies Venture Capital https://linkedin.com/company/pt-vc 
 # Prospective Technologies Venture Capital

Prospective Technologies Venture Capital is a Venture Capital and Private Equity Principals company. PTV is a specialist VC fund focused on IT infrastructure software, investing in B2B tools, DevOps, databases, and cloud-native technologies with global potential. Prospective Technologies Venture Capital employs 7 people, founded in 2021. Headquartered in George Town, Grand Cayman, Cayman Islands, with presence in United States. Its workforce is distributed across Cyprus, United States, France, and United Kingdom.

## About

Prospective Technologies is an investment and venture capital firm focused on B2B and SMB tech solutions with global expansion potential in order to build a thriving international company. 
In addition to years of both operational and investing experience, we have expertise in engineering, business development, operations, sales and marketing, ma

In [24]:
p=Path('.prompts/MEMORY.md')
p.write_text('''# Project Memory

## Context
- This project independently audits disclosed outside funding for open-source-minded ventures using the supplied sample and `deal_rules.md`.

## Workflow conventions
- Deliver `funders_audit.csv` (one row per funding event, requested schema) and `funders_audit_notes.md` (one line per venture); preserve input keys and venture names exactly.
- Use fresh web evidence for funding events, normalize investor firms and person names per the rules, and explicitly count ventures whose research remains incomplete.
- Treat a reasonable unsuccessful search as `none_found`, not proof of no outside funding; document search scope and uncertainty.
''')
print('Saved procedural conventions.')

Saved procedural conventions.


In [25]:
# Retain cross-check sources for eventual reconciliation.
for result in recent[:1]+[db[0],db[-2],db2[0]]:
 retain(result)
# Search-name collisions to explicitly guard against in the combined audit.
Path('audit_work/reconciliation_flags.json').write_text(json.dumps({
 'TecharoHQ': {'source':techaro_url,'issue':'Disclosed recurring donations on GitHub Sponsors/Patreon/Liberapay; cannot call none_found without discussing donation scope. Do not treat recurring monthly total as discrete round.'},
 'payloadcms':{'issue':'Do not confuse Payload CMS (YC; acquired by Figma) with Payload payments fintech.'},
 'farmbot':{'issue':'Do not confuse FarmBot Inc open hardware robot with Australian Farmbot Monitoring Solutions.'},
 'dbeaver':{'issue':'Company calls April 2023 first funding round; database suggests 2021/March 2023 + accelerator. Need investigate not blindly duplicate.'},
 'radixark':{'source':recent[0]['url'],'issue':'May 2026 $100m seed latest primary source, not just February 2026 reported fundraising.'},
 'va-linux':{'issue':'Intel March 1999 investment may close November 1998 Series A, check filing before double count; June 1999 $25m private round.'}
},indent=2))
print('Reconciliation flags persisted.')

Reconciliation flags persisted.


In [26]:
gap_queries=[
 '"Odoo" investment funding 2025 2026 General Atlantic secondary',
 '"Anaconda" "funding" "2025" investors',
 '"MindsDB" "Series C" "47"',
 '"EvolutionaryScale" funding "2025" "2026"',
 '"OpenBB" funding "2025" "2026"',
 '"ResearchHub" "token sale" funding',
 '"ClearML" funding "2025" "2026"',
 '"Efabless" "grant" funding government'
]
gap_results=await asyncio.gather(*(web_search(q,num_results=5) for q in gap_queries))
Path('audit_work/gap_searches.json').write_text(json.dumps(list(zip(gap_queries,gap_results)),indent=2))
for query,res in zip(gap_queries,gap_results):
 print('\nQUERY',query)
 for x in res: print(x['title'],x['url'],(x.get('snippet') or '')[:400])


QUERY "Odoo" investment funding 2025 2026 General Atlantic secondary
General Atlantic announces close of further investment into Odoo, acquiring additional stake from Wallonie Entreprendre | General Atlantic https://www.generalatlantic.com/media-article/general-atlantic-announces-close-of-further-investment-into-odoo-acquiring-additional-stake-from-wallonie-entreprendre/ # General Atlantic announces close of further investment into Odoo, acquiring additional stake from Wallonie Entreprendre

 January 16, 2026 

Belgium, 16 January 2026 – General Atlantic, a leading growth investor, today announced the close of its follow-on investment in Odoo SA (or “the Company”), a leading provider of open-source integrated business software applications for small and medium-siz
Odoo tops €7 billion valuation as General Atlantic increases stake https://thenextweb.com/news/odoo-tops-7-billion-valuation This article was published on January 20, 2026 GMT

...

Belgian business software company Odoo, al

In [27]:
for query,res in list(zip(gap_queries,gap_results))[2:6]:
 print('\nQUERY',query)
 for x in res: print(x['title'],x['url'],'\n',(x.get('snippet') or '')[:1300])


QUERY "MindsDB" "Series C" "47"
Startups Funded in the Week of May 29 to June 4, 2023 - Fundraise Insider https://fundraiseinsider.com/blog/startups-funded-in-the-week-of-may-29-to-june-4-2023/ 
 Startups Funded in the Week of May 29 to June 4, 2023 - Fundraise Insider

# Startups Funded in the Week of May 29 to June 4, 2023

Chris Walker

May 6, 2024

...

## MindsDB Raises $2 Million in Seed Funding

MindsDB, a San Francisco-based startup that provides a platform for businesses to build and deploy AI models, announced on June 1, 2023 that it has raised $2 million in Seed funding. The round was led by Neo, with participation from Initialized Capital, and others.

MindsDB’s platform allows businesses to build and deploy AI models without having to write any code. The platform also provides businesses with a way to manage and monitor their AI models.

The company plans to use the funding to expand its team, develop new features, and grow its customer base.
57 Growing AI Companies & Sta

In [28]:
odoo2026=gap_results[0][0]
avp2026=gap_results[0][-1]
for z in [odoo2026,avp2026]:
 text=await fetch_parsed_page(z['url'])
 z['fetched_text']=text
 retain(z)
 print(z['url'],'\n',text[:16000])

https://www.generalatlantic.com/media-article/general-atlantic-announces-close-of-further-investment-into-odoo-acquiring-additional-stake-from-wallonie-entreprendre/ 
 General Atlantic announces close of further investment into Odoo, acquiring additional stake from Wallonie Entreprendre | General Atlantic

[![General Atlantic](https://www.generalatlantic.com/wp-content/uploads/2024/07/general-atlantic-logo.svg "width=171 | height=32")](https://www.generalatlantic.com/) General Atlantic

* [About](https://www.generalatlantic.com/media-article/general-atlantic-announces-close-of-further-investment-into-odoo-acquiring-additional-stake-from-wallonie-entreprendre/#)
    + [About Us](https://www.generalatlantic.com/media-article/general-atlantic-announces-close-of-further-investment-into-odoo-acquiring-additional-stake-from-wallonie-entreprendre/#)
        - [Our Story](https://www.generalatlantic.com/our-story/)
        - [Our Culture](https://www.generalatlantic.com/our-culture/)
        -

https://avpcap.com/avp-and-alkeon-announce-e60m-secondary-round-in-odoo-s-a-at-a-valuation-exceeding-e7-billion/ 
 AVP and Alkeon announce €60 Million secondary round in Odoo S.A. at a valuation exceeding €7 billion — AVP

[Close Search](https://avpcap.com/avp-and-alkeon-announce-e60m-secondary-round-in-odoo-s-a-at-a-valuation-exceeding-e7-billion/#)

[![AVP](https://avpcap.com/wp-content/uploads/2025/12/AVP-red.svg "width=1080 | height=521")![AVP](https://avpcap.com/wp-content/uploads/2025/12/AVP-red.svg "width=1080 | height=521")![AVP](https://avpcap.com/wp-content/uploads/2025/12/AVP-red.svg "width=1080 | height=521")](https://avpcap.com)

[Menu](https://avpcap.com/avp-and-alkeon-announce-e60m-secondary-round-in-odoo-s-a-at-a-valuation-exceeding-e7-billion/#slide-out-widget-area)

* [Who we are](https://avpcap.com/who-we-are/)
* [Companies](https://avpcap.com/companies/)
* [News](https://avpcap.com/news/)

[Portfolio](https://avpcap.com/category/portfolio/)[Enterprise software](http

In [29]:
for z in [odoo2026,avp2026]:
 text=z['fetched_text']
 idx=text.find('January 16, 2026', 200)
 print(text[idx:idx+9000])
print([(p.name,p.stat().st_size) for p in Path('audit_work').iterdir()])

January 16, 2026

[Share](https://www.generalatlantic.com/media-article/general-atlantic-announces-close-of-further-investment-into-odoo-acquiring-additional-stake-from-wallonie-entreprendre/)

![](https://www.generalatlantic.com/wp-content/uploads/2026/01/General-Atlantic-News-Insights-Posts-4.png "width=818 | height=652")

**Belgium, 16 January 2026** – General Atlantic, a leading growth investor, today announced the close of its follow-on investment in Odoo SA (or “the Company”), a leading provider of open-source integrated business software applications for small and medium-sized enterprises (SMEs). As part of this transaction, which valued Odoo at €7bn, General Atlantic acquired an additional minority stake in Odoo from Wallonie Entreprendre, the Walloon region’s investment arm.

Founded in 2002 by CEO Fabien Pinckaers, Odoo SA is a leading provider of open-source integrated business software applications for micro and small and medium-sized enterprises (SMEs). Odoo’s “open core” 

In [30]:
print(avp2026['fetched_text'][:7000])
print('elapsed notebook work sources',len(own_evidence))

AVP and Alkeon announce €60 Million secondary round in Odoo S.A. at a valuation exceeding €7 billion — AVP

[Close Search](https://avpcap.com/avp-and-alkeon-announce-e60m-secondary-round-in-odoo-s-a-at-a-valuation-exceeding-e7-billion/#)

[![AVP](https://avpcap.com/wp-content/uploads/2025/12/AVP-red.svg "width=1080 | height=521")![AVP](https://avpcap.com/wp-content/uploads/2025/12/AVP-red.svg "width=1080 | height=521")![AVP](https://avpcap.com/wp-content/uploads/2025/12/AVP-red.svg "width=1080 | height=521")](https://avpcap.com)

[Menu](https://avpcap.com/avp-and-alkeon-announce-e60m-secondary-round-in-odoo-s-a-at-a-valuation-exceeding-e7-billion/#slide-out-widget-area)

* [Who we are](https://avpcap.com/who-we-are/)
* [Companies](https://avpcap.com/companies/)
* [News](https://avpcap.com/news/)

[Portfolio](https://avpcap.com/category/portfolio/)[Enterprise software](https://avpcap.com/category/sector/enterprise-software/)

# AVP and Alkeon announce €60 Million secondary round in Odoo

In [31]:
# Extra early-stage checks for small ventures where database names can hide missed awards.
qs=['"Fossorial" "Pangolin" funding investment','"Cheshire Labs" funding grant accelerator','"Fincept" funding grant investment','"neuromore" funding Republic','"Terrastruct" seed Haystack Bloomberg Fathom']
smallchecks=await asyncio.gather(*(web_search(q,num_results=5) for q in qs))
Path('audit_work/small_checks.json').write_text(json.dumps(list(zip(qs,smallchecks)),indent=2))
for q1,res in zip(qs,smallchecks):
 print('\n',q1)
 for x in res: print(x['title'],x['url'],'\n',x.get('snippet','')[:1100])


 "Fossorial" "Pangolin" funding investment
Pangolin company profile: firmographics, business model, customers, competitors | akta.pro https://akta.pro/company-profiles/0002dna/pangolin 
 ### Pangolin

Full company profile

Name string Pangolin

Legal name string Fossorial Inc.

Website url

digpangolin.com

Company type enum Private

Founded year int 2025

...

Pangolin (legally Fossorial Inc.)

...

Pangolin employs 1-10 people and is based in San Francisco. The company was co-founded by Milo Schwartz (CEO) and Owen Schwartz (CTO) and has raised approximately $5.2 million across a 2025 Y Combinator round ($500K) and a larger October 2025 seed round ($4.7M) led by Chemistry with participation from 468 Capital, Blast. Club, Pioneer Fund, Transpose Platform Management, and Y Combinator. Distribution is product-led and community-led, relying on GitHub visibility, developer documentation, the FLOSS Weekly podcast feature, and direct self-serve downloads rather than a traditional field sal

In [32]:
for q1,res in zip(qs[1:4],smallchecks[1:4]):
 print('\n',q1)
 for x in res: print(x['title'],x['url'],'\n',x.get('full_text','')[:7000])


 "Cheshire Labs" funding grant accelerator
Cheshire Labs Revenue, Funding & Valuation https://prospeo.io/c/cheshire-labs-revenue 
 Cheshire Labs Revenue, Funding & Valuation

# Cheshire Labs Revenue

Software Development • San Francisco, California, United States • 1-10 Employees

$

Cheshire Labs revenue & valuation

| Annual revenue | $85,555 |
| --- | --- |
| Revenue per employee | $86,000 |
| Estimated valuation? This valuation is estimated based on industry average for the Software Development industry and current estimated revenues | $273,776 |
| Total funding | No funding |

Key Contact at Cheshire Labs

Michael Salmi

Founder & CEO / Lab Automation Software Consultant & Contractor

Email

Company overview

| Headquarters | San Francisco, California, United States |
| --- | --- |
| Keywords | Biotechnology, Consulting, Software Development, Contracting, Laboratory Automation, Automation Scheduler |
| Employees | 1-10 |

## About Cheshire Labs

Cheshire Labs is the developer of 

In [33]:
print('Fincept results')
for x in smallchecks[2]:print(x['title'],x['url'],x.get('full_text','')[:7000])
print('neuromore titles',[(x['title'],x['url']) for x in smallchecks[3]])

Fincept results
Fincept Corporation 2026 Company Profile: Valuation, Funding & Investors | PitchBook https://pitchbook.com/profiles/company/649229-41 Fincept Corporation 2026 Company Profile: Valuation, Funding & Investors | PitchBook

# Fincept Corporation

 

This is a profile preview from the PitchBook Platform.

## Fincept Corporation Overview 

- Year Founded
- 2022

 

- Status
- Private

- Employees
- 2

 

- Latest Deal Type
- Accelerator/​Inc

- Investors
- 3

### Fincept Corporation General Information

#### Description

Developer of a financial intelligence terminal designed for professional market analysis and equity research. The company's platform offers an artificial intelligence research assistant, multi-agent research, natural language processing for investment insights, sentiment analysis, real-time data processing, and institutional security controls, enabling portfolio managers and analysts to access real-time market data and perform automated equity research.

####

In [34]:
print(datetime.datetime.now())
print([(p.name,p.stat().st_size) for p in Path('audit_work').iterdir()])
# Fetch unusual small-venture signals to use only when money is established.
fincept_text=await fetch_parsed_page('https://pitchbook.com/profiles/company/649229-41')
retain({'title':'Fincept Corporation 2026 Company Profile: Valuation, Funding & Investors','url':'https://pitchbook.com/profiles/company/649229-41','full_text':fincept_text},'PitchBook')
republic_text=await fetch_parsed_page('https://republic.com/neuromore')
retain({'title':'neuromore','url':'https://republic.com/neuromore','full_text':republic_text},'Republic')
print(republic_text[:20000])

2026-11-26 06:02:12.477721
[('reconciliation_flags.json', 1070), ('latest_checks.json', 760240), ('supplement_task.txt', 36), ('independent_evidence.json', 74229), ('small_checks.json', 114956), ('tasks.json', 378), ('gap_searches.json', 237928)]


neuromore — Republic

[![Republic](https://assets.republic.com/assets/logo/full/black/logo-2c113fd1a194abf7fa3c5809c1470ca753c7a8dda7c2a3e86544990495e3056e.svg "Republic") ![Republic](https://assets.republic.com/assets/logo/full/black/logo-2c113fd1a194abf7fa3c5809c1470ca753c7a8dda7c2a3e86544990495e3056e.svg "Republic") ![Republic](https://assets.republic.com/assets/logo/icon/brand/36x40/logo-584528d3c0290325573833b80233780cd36a20e325d5b0d173bcf210463b0a3a.svg)](https://republic.com/)

* [Log in](https://republic.com/login)

* 
*

[Open account](https://republic.com/open-account)

![](https://assets.republic.com/assets/ui/search/loader-62242cf1ddeed701e13538628d675bd36bfd2548f882dca8ed127fb90043bbd1.svg)

![](https://assets.republic.com/assets/ui/search/sad-face-4e0a6424ec40ef8fcdfeafe700612e45fed331a733b7e1eb91db5eb95ed0ab2c.svg) Oops! We couldn’t find any results...

Can’t find a deal? Try [advanced search](https://republic.com/investment-opportunities).   
Is something missing? Add y

In [35]:
import re
for pat in ['raised','2023','2026','investor','funded','Minimum','Investment']:
 matches=list(re.finditer(pat,republic_text,re.I))
 print('\n',pat,len(matches))
 for m in matches[:12]: print(republic_text[max(0,m.start()-130):m.start()+250].replace('\n',' '))


 raised 6
Deal Portal LLC, CRD #283874) is considering hosting this Reg CF securities offering by neuromore co.  Aug 31  2023  Successfully raised $30,000 🍾  Nov 22  2022  Opened for reservations ⏳  ### Hear from some of the 5 reserved investors in neuromore  ---  Show more  ## Highlights  ---  * ⏰ Timing: 1 billion people on the planet suffer from mental health issues * 📈TAM: $1 trillio
ps://www.akiliinteractive.com/), who launched EndeavorRx®, a prescription treatment for ADHD delivered through a video game. They raised $300 million and went public (NASDAQ:AKLI) in August 2022.  neuromore's major differentiation is a truly dynamic and personalized treatment experience run on real-time bio-data and biofeedback. It's a game changer!  ![](https://republic.com/cd
resilience.*  Source: [Positive Psychology](https://positivepsychology.com/benefits-of-mental-health/)  ## Funding  ---  ## $750k raised from accredited angel investors  In addition to our grant funding, neuromore has raised $7

In [36]:
statuses=await asyncio.gather(*(check_task(t) for t in task_ids+[extra_task]))
print([(i+1,s.status,s.runtime_minutes,s.steps) for i,s in enumerate(statuses)])
print([(p.name,p.stat().st_size) for p in Path('audit_work').glob('*')])

Task b167ec30-2689-496e-b683-3d6e527b2778 still running (status: in progress). Call check_task(task_id) again later.
Task d837b2d6-cfbe-4d5f-96db-7a65294f01ba still running (status: in progress). Call check_task(task_id) again later.
Task 84dea63a-6319-43c0-a0ff-0a9d45ebc0ba still running (status: in progress). Call check_task(task_id) again later.
Task f1d49db0-fc7f-4bb4-885e-8c31cc9553f8 still running (status: in progress). Call check_task(task_id) again later.
Task cb65c732-d0ce-4369-860f-29e5528ac62b still running (status: in progress). Call check_task(task_id) again later.
Task 5b9368af-2a81-4e85-98cb-9fca8dbf03fc still running (status: in progress). Call check_task(task_id) again later.
Task 78387c12-5338-4c51-b039-365853670889 still running (status: in progress). Call check_task(task_id) again later.
Task 65955bd0-9605-41d7-80dd-9e036c8f6d8f still running (status: in progress). Call check_task(task_id) again later.
Task 09c9f518-f5d8-4501-9671-a3f2ada07c89 still running (status:

In [37]:
neuromore_grants=await web_search('"neuromore" "grant" funding foundation',num_results=7)
for x in neuromore_grants: print(x['title'],x['url'],'\n',x.get('snippet','')[:1600])
Path('audit_work/neuromore_grant_checks.json').write_text(json.dumps(neuromore_grants,indent=2))

NEUROMORE INC. — $40K raised | Noair Lab https://www.noairlab.ai/companies/neuromore-inc 
 NEUROMORE INC. — $40K raised | Noair Lab

# NEUROMORE INC.

Updated Nov 24, 2026 · neuromore.com

Real-time neurofeedback platform that processes brain signals and delivers closed-loop feedback, device-agnostic and cloud-deployed

Sector

Medtech

Headquarters

Miami Beach, FL

Founded

2016

Status

Private

Latest Form D

$40K sold · Apr 2016

Total funding (grants + private)

$40K

Federal grants

None identified No awards found in SBIR/STTR, NIH, NSF or DoD data.

Private capital raised

$40K

Sold · 1 Form D · last Apr 2016

127 months since last filing

Federal contracts

None identified No federal contract awards found.

Granted patents

None identified No USPTO grants. Applications publish after 18 months.

## Financing and Funding History

Grant Contract Equity Cumulative

### Latest events

- Apr 2016 Private capital raised Form D · SEC · 1st filing

$40K

Node area is scaled to amount.

43596

In [38]:
for x in neuromore_grants:
 if 'workingwoman' in x['url']:
  print(x.get('full_text',''))
  retain(x)
print('Available work files:',[p.name for p in Path('audit_work').iterdir()])

Startups Out To Build A Mindful Miami – Working Woman Report

# Startups Out To Build A Mindful Miami

 By Working Woman Report Posted on September 26, 2016 

By Nancy Dahlberg The Miami Herald

WWR Article Summary (tl;dr) Startups are clearly hopping onto the mindful bandwagon to service a growing market of consumers, searching for well-being. From yoga, to meditation to special clothing and food…many entrepreneurs (especially in Miami) are tapping into the quest for inner peace.

The Miami Herald

Miami a magnet for mindfulness? Really?

If you are reading this stuck in Interstate 95 gridlock, stressed out at work, or distressed by news of yet another senseless shooting death or an only-in-Miami fraud, you may think not. But a growing number of local startups are betting the time has come for mindfulness and well-being in the Magic City.

Consider this: While yoga classes, spas and life coaches have long been a part of the scene, the past year has brought businesses specializing in s

In [39]:
fxq=await web_search('ECB euro foreign exchange reference rates 16 January 2026 USD 1.16',num_results=4)
for x in fxq: print(x['title'],x['url'],x.get('snippet','')[:2500])

Euro exchange rates – 16 January 2026 https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ%3AC_202600055 Official Journal of the European Union

EN C series C/2026/55 19.1.2026

Euro exchange rates ( 1) 
16 January 2026 
(C/2026/55) 

## 1 euro =

| | Currency | Exchange rate | | Currency | Exchange rate |
| --- | --- | --- | --- | --- | --- |
| USD | US dollar | 1,1617 | CAD | Canadian dollar | 1,6135 |
| JPY | Japanese yen | 183,67 | HKD | Hong Kong dollar | 9,0613 |
| DKK | Danish krone | 7,4712 | NZD | New Zealand dollar | 2,0179 |
| GBP | Pound sterling | 0,86700 | SGD | Singapore dollar | 1,4962 |
| SEK | Swedish krona | 10,7190 | KRW | South Korean won | 1 711,10 |
| CHF | Swiss franc | 0,9313 | ZAR | South African rand | 19,0382 |
| ISK | Iceland króna | 146,40 | CNY | Chinese yuan renminbi | 8,0967 |
| NOK | Norwegian krone | 11,7203 | IDR | Indonesian rupiah | 19 653,76 |
| BGN | Bulgarian lev | | MYR | Malaysian ringgit | 4,7136 |
| CZK | Czech koruna | 24,286 | PHP | 

In [40]:
fxpage=await fetch_parsed_page(fxq[1]['url'])
retain({'title':fxq[1]['title'],'url':fxq[1]['url'],'publishedDate':'2026-01-16','full_text':fxpage},"Banca d'Italia")
Path('audit_work/odoo2026_supplement.json').write_text(json.dumps({'events':[{'type':'secondary','date':'2026-01','lead':'General Atlantic','amount_usd':'','source_url':odoo2026['url'],'note':'Additional stake acquired from Wallonie Entreprendre; price undisclosed, €7bn is valuation not funding.'},{'type':'secondary','date':'2026-01','lead':'','investors':'AVP | Alkeon Capital','amount_usd':str(round(60000000*1.1617)),'source_url':avp2026['url'],'note':'€60m secondary management-share sale; USD conversion at ECB 16 Jan 2026: 1 EUR = USD 1.1617. Separate from GA buyout of regional investor.'}],'fx_url':fxq[1]['url']},indent=2))
print('EUR60m -> USD',round(60000000*1.1617))

EUR60m -> USD 69702000


In [41]:
print([(p.name,p.stat().st_size) for p in Path('audit_work').iterdir()])
print(datetime.datetime.now())

[('reconciliation_flags.json', 1070), ('latest_checks.json', 760240), ('odoo2026_supplement.json', 1006), ('supplement_task.txt', 36), ('independent_evidence.json', 203682), ('small_checks.json', 114956), ('tasks.json', 378), ('gap_searches.json', 237928), ('neuromore_grant_checks.json', 43596)]
2026-11-26 06:04:28.470181


In [42]:
# Search investment-history edges omitted by conventional priced-round summaries.
edge_queries=[
 '"Airbyte" "Y Combinator" funding 2020 seed',
 '"Meltano" "2025" "funding" acquired',
 '"Coqui" "grant" "Mozilla" funding',
 '"Open Bionics" "2025" funding investment',
 '"Bela" "Trill" "Gliss" Kickstarter raised',
 '"FarmBot" "Kickstarter" funding raised grants'
]
edge_results=await asyncio.gather(*(web_search(q,num_results=5) for q in edge_queries))
Path('audit_work/edge_searches.json').write_text(json.dumps(list(zip(edge_queries,edge_results)),indent=2))
for q1,res in zip(edge_queries,edge_results):
 print('\n',q1)
 for x in res: print(x['title'],x['url'],x.get('snippet','')[:700])


 "Airbyte" "Y Combinator" funding 2020 seed
Airbyte company information, funding & investors | Dealroom.co https://app.dealroom.co/companies/airbyte_1 # Airbyte

Open-source data integration for analytics and AI.

HQ location

San Francisco, United States

Website airbyte.com

Launch date

2020

...

| Date | Investors | Amount | Round |
| --- | --- | --- | --- |
| Mar 2020 | Rebel Fund | N/A | Seed |
| Mar 2020 | Liquid 2 Ventures | N/A | Seed |
| Aug 2020 | Y Combinator | $125k | Seed |
| Mar 2021* | Accel Y Combinator Calvin French-Owen Auren Hoffman Alain Rossmann Travis May 8VC Charles Zedlewski | $5.2m | Seed |
| May 2021 | Accel Benchmark SV Angel Y Combinator Dev Ittycheria Auren Hoffman 8VC Shay Banon | $26.0m | Series A |
| Dec 2021* | Accel Benchmark SV Angel Thrive Capital Coatue Management Salesforce Ventures Altimeter Capital
Airbyte: The open standard in data connectivity for AI and analytics. | Y Combinator https://www.ycombinator.com/companies/airbyte Airbyte

The ope

In [43]:
# Record corroborated gaps for integration after the parallel passes finish.
for x in edge_results[-1]:
 if x['url'].endswith('the-farmbot-launch-campaign-wrapup'):
  txt=await fetch_parsed_page(x['url']); x['fetched_text']=txt; retain(x,'FarmBot')
Path('audit_work/edge_searches.json').write_text(json.dumps(list(zip(edge_queries,edge_results)),indent=2))
print('Primary FarmBot launch campaign retained.')
# Validate input identity once; these mappings are authoritative for output.
assert len(sample)==40 and sample['key'].nunique()==40
identities=sample.set_index('key')['venture'].to_dict()
columns=['key','venture','type','label','date','amount_usd','lead','investors','source_url','confidence']
allowed_types=set(re.findall(r'`([a-z_]+)`',Path('deal_rules.md').read_text().split('| `type`')[1].split('\n')[0]))|{'none_found'}
print('Input:',len(sample),'ventures; allowed types:',sorted(allowed_types))

Primary FarmBot launch campaign retained.
Input: 40 ventures; allowed types: ['accelerator', 'angel', 'convertible_note', 'corporate_strategic', 'crowdfunding_equity', 'crowdfunding_reward', 'debt', 'grant_government', 'grant_philanthropic', 'growth', 'none_found', 'other', 'pre_seed', 'prize', 'secondary', 'seed', 'series_a', 'series_b', 'series_c', 'series_d', 'series_e', 'series_f_plus', 'token_sale', 'venture_unspecified']


In [44]:
print(s2.model_dump_json(indent=2)[:18000])

{
  "task_id": "d837b2d6-cfbe-4d5f-96db-7a65294f01ba",
  "status": "success",
  "query": "Independently research disclosed outside funding events for ONLY the ventures below. Read /workspace/deal_rules.md and /workspace/funders_audit_sample.csv. User needs a funding audit, not licensing or company metadata. Search web freshly (no facts from memory); about 3-6 searches/fetches each, more for many rounds. Include ALL disclosed events: seed, angel, accelerator, series, grants, crowdfunding, token, debt, secondary etc. One row per event with columns exactly key,venture,type,label,date,amount_usd,lead,investors,source_url,confidence. Types from rules, or none_found only after reasonable searches. Date YYYY-MM; blank when month unknown, explain in notes. Amount digits USD, blank undisclosed; avoid unsubstantiated exchange conversion. People 'Full Name (person)', firms usual names, lead excluded from investors, separators ' | '. Each event source must actually name investors or amount. Prefer

In [45]:
from e14c.subagents import get_task_messages
msgs2=await get_task_messages(task_ids[1])
print(len(msgs2))
Path('audit_work/group2_messages.json').write_text(json.dumps(msgs2,indent=2))
for m in msgs2[-8:]: print(str(m)[:14000])

67
{'role': 'tool', 'content': '[Appended Cell #24] [stdout]\nSaved 8 rows across 5 ventures; 1 incomplete.\n', 'name': 'run_cell', 'tool_call_id': 'call_278bb5792f1244729d17ab1848362f01', 'info': {'start_ts': 10888.157148752, 'service_usage': {'source': 'envoy', 'correlation': 'sandbox_execution_window', 'truncated': False, 'providers': []}, 'end_ts': 10888.260101222}}
{'role': 'assistant', 'content': None, 'function_call': None, 'tool_calls': [{'id': 'call_57c0239a0afc4f11bdb599422325b7dc', 'type': 'function', 'function': {'arguments': '{"code": "# Publication metadata transcribed from the fetched pages; blank dates mean not established.\\nmeta={\\n0:(\'GrapesJS creator founds Grapes Studio Inc. to bring the open-source, multi-project, no-code website editor mainstream\',\'Erica Lindberg\',\'Open Core Ventures\',\'2024-06-27\'),\\n1:(\'GrapesJS and Beyond: Introducing Grapes Studio and Our Plans for the Future\',\'Artur Arseniev\',\'GrapesJS\',\'2024-06-13\'),\\n2:(\'Fincept Corporat

In [46]:
def extract_cells(messages):
 cells=[]
 for m in messages:
  for c in m.get('tool_calls',[]) or []:
   f=c.get('function',{})
   if f.get('name')=='run_cell':
    args=json.loads(f['arguments']); cells.append(args.get('code',''))
 return cells
cells2=extract_cells(msgs2)
for i,c in enumerate(cells2): print(i,c[:250].replace('\n',' '))
print('\nROW BUILDING\n')
for i,c in enumerate(cells2):
 if 'rows=' in c or 'rows =' in c: print(i,c[:26000])

0 from pathlib import Path print(Path('/workspace/deal_rules.md').read_text()) print(Path('/workspace/funders_audit_sample.csv').read_text()) print('\nSkills:', [str(p) for p in Path('/workspace/.skills').glob('*')])
1 from pathlib import Path import os print('cwd',os.getcwd()) print('workspace',list(Path('/workspace').iterdir())) for name in ['deal_rules.md','funders_audit_sample.csv']:  matches=list(Path('/workspace').rglob(name)); print(name,matches)  for p in m
2 import asyncio,json,time,datetime from e14c.web import web_search,fetch_parsed_page,save_citation out=Path('/workspace/audit_work');out.mkdir(exist_ok=True) evidence={'retrieved_at_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),'limita
3 for s in evidence['searches']:  print('\n',s['key'])  for x in s['results']:   print(x['title'],x['url']);print((x.get('snippet') or '')[:1100])
4 for k in ['Fincept-Corporation','fosrl','letta-ai']:  print('\n',k)  for s in evidence['searches']:   if s['key']==k:    for x i

In [47]:
print(cells2[24][:18000])

for i,x in enumerate(evidence['fetches']):
 print(i,x['key'],x['url'],str(x['text']).splitlines()[0][:140])
print('\nSearch date retrieved',evidence['retrieved_at_utc'])
# Last check sourced seed investors.
t=next(x['text'] for x in evidence['fetches'] if 'prnewswire' in x['url'] and 'letta' in x['url']);
i=t.find('SAN FRANCISCO,');print(t[i:i+1800])
t=next(x['text'] for x in evidence['fetches'] if 'tekedia' in x['url']);i=t.find('For this reason');print(t[i:i+700])


In [48]:
print([(i,len(c)) for i,c in enumerate(cells2)])
for i,c in enumerate(cells2):
 if 'rows=[' in c or 'rows = [' in c: print('CELL',i,'\n',c[:16000])

[(0, 214), (1, 350), (2, 1273), (3, 144), (4, 206), (5, 1047), (6, 1018), (7, 442), (8, 821), (9, 820), (10, 268), (11, 919), (12, 676), (13, 685), (14, 672), (15, 874), (16, 574), (17, 899), (18, 589), (19, 1162), (20, 761), (21, 968), (22, 485), (23, 807), (24, 470), (25, 10212), (26, 5372), (27, 5992), (28, 1433)]
CELL 25 
 import csv,hashlib
F='Fincept-Corporation'; P='fosrl'; G='GrapesJS'; K='Kilo-Org'; L='letta-ai'
u=lambda i:evidence['fetches'][i]['url']
columns=['key','venture','type','label','date','amount_usd','lead','investors','source_url','confidence']
rows=[]
def row(key,venture,typ,label,date='',amount='',lead='',investors='',sources='',confidence='high'):
 rows.append(dict(zip(columns,[key,venture,typ,label,date,amount,lead,investors,sources,confidence])))
for n in [1,2,3]:
 row(F,'Fincept Corporation','accelerator',f'Reported accelerator/incubator deal {n}; capital transfer and round-specific investors unverified',sources=u(5),confidence='low')
row(P,'Fossorial (Pangol

In [49]:
# Recover the remote agent's numbered source index and its curated row definitions.
for m in msgs2:
 c=m.get('content')
 if m.get('role')=='tool' and isinstance(c,str) and 'Search date retrieved' in c: print(c[:18000])

[Appended Cell #23] [stdout]
0 GrapesJS https://opencoreventures.com/insights/grapesjs-creator-founds-grapes-studio-inc-to-bring-the-open-source-multi-project-no-code-website-editor-mainstream/ Open Core Ventures — We build commercial open source
1 GrapesJS https://grapesjs.com/blog/grapesjs-and-beyond GrapesJS and Beyond: Introducing Grapes Studio and Our Plans for the Future | GrapesJS Blog
2 Fincept-Corporation https://fossunited.org/grants/directory/fincept_corporation Fincept Corporation – Funding Profile [Skip to Main Content](https://fossunited.org/grants/directory/fincept_corporation#main-content)
3 Kilo-Org https://blog.kilo.ai/p/kilo-raised-8-million-seed-round We raised $8 Million to bring Kilo Speed to Agentic Engineering 
4 Kilo-Org https://www.cnbc.com/2025/12/10/former-gitlab-ceo-raises-8-million-for-kilo-to-compete-in-vibe-coding.html Former GitLab CEO raises $8 million for Kilo to compete in vibe coding
5 Fincept-Corporation https://pitchbook.com/profiles/company/64922

In [50]:
# Recover group 2 from its auditable execution transcript (remote retrieval agents have isolated filesystems).
indextext=next(m['content'] for m in msgs2 if m.get('role')=='tool' and isinstance(m.get('content'),str) and 'Search date retrieved' in m['content'])
fetched2=[]
for line in indextext.splitlines():
 mm=re.match(r'^(\d+) (\S+) (https?://\S+) (.*)$',line)
 if mm:
  n,key,url,title=mm.groups(); assert int(n)==len(fetched2)
  fetched2.append({'key':key,'url':url,'text':title})
ns2={'evidence':{'fetches':fetched2,'searches':[]}}
exec(cells2[25].split('with (out/')[0],ns2)
# Final code changes from original author's QA.
print(cells2[27][:6500])
pd.DataFrame(ns2['rows']).to_csv('audit_work/group2.csv',index=False)
Path('audit_work/group2_notes.json').write_text(json.dumps(ns2['notes'],indent=2))
print(pd.DataFrame(ns2['rows']).to_string(index=False))
# Save full source metadata using the mandatory citation API, not guessed metadata.
meta_code=cells2[26].split('def bib_escape')[0]
exec(meta_code,ns2)
for i,f in enumerate(fetched2):
 title,author,publisher,date=ns2['meta'][i]
 if 'Unavailable purported' not in title:
  retain({'url':f['url'],'title':title,'publishedDate':date},author.strip('{}') or publisher)
print('Recovered group2 data and bibliography.')

evidence['bibtex_entries']=bibtex_entries
script='''#!/usr/bin/env python3
"""Reproduce group 2 audit files from the persisted, manually reviewed evidence.

Usage: python /workspace/audit_work/group2_reproduce.py
Optional: python .../group2_reproduce.py --refresh
--refresh also repeats the saved searches and page fetches into
  group2_refreshed_evidence.json. Newly retrieved facts require manual review;
  refresh does not automatically change the audited event interpretations.
Dependencies for refresh only: e14c.
"""
import argparse, asyncio, csv, datetime, json
from pathlib import Path

BASE = Path(__file__).resolve().parent

def reproduce(evidence):
    with (BASE / 'group2.csv').open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=evidence['method']['csv_columns'])
        writer.writeheader()
        writer.writerows(evidence['curated_rows'])
    (BASE / 'group2_notes.json').write_text(
        json.dumps(evidence['curated_notes'], i

In [51]:
print(cells2[28])
statuses=await asyncio.gather(*(check_task(t) for t in task_ids+[extra_task]))
print([(i+1,s.status,s.runtime_minutes,s.steps) for i,s in enumerate(statuses)])

# Do not infer a lead designation merely because an accelerator is the only named investor.
rows[3]['lead']='';rows[3]['investors']='Y Combinator'
notes[1]['note']=notes[1]['note'].replace('so confidence is medium.','so confidence is medium. YC is listed as investor, not inferred as round lead.')
evidence['curated_rows']=rows;evidence['curated_notes']=notes
evidence['validation']={'status':'passed','row_count':8,'venture_count':5,'incomplete_count':1,'incomplete_keys':['Fincept-Corporation'],'search_count':len(evidence['searches']),'fetch_attempt_count':len(evidence['fetches']),'bibtex_entry_count':32,'checks':['Exact ten-column CSV schema','UTF-8 CSV and JSON readable','Amounts digits or blank','Dates YYYY-MM or blank','No lead duplicated in investors','All five requested venture keys present','No cumulative funding totals entered as new rounds','All fetched/attempted sources have full BibTeX entries','Missing rules, missing sample and unavailable skill disclosed']}
persist()
complete

Task 84dea63a-6319-43c0-a0ff-0a9d45ebc0ba still running (status: in progress). Call check_task(task_id) again later.
Task b167ec30-2689-496e-b683-3d6e527b2778 still running (status: in progress). Call check_task(task_id) again later.
Task 78387c12-5338-4c51-b039-365853670889 still running (status: in progress). Call check_task(task_id) again later.
Task cb65c732-d0ce-4369-860f-29e5528ac62b still running (status: in progress). Call check_task(task_id) again later.
Task 09c9f518-f5d8-4501-9671-a3f2ada07c89 still running (status: in progress). Call check_task(task_id) again later.


Task 65955bd0-9605-41d7-80dd-9e036c8f6d8f still running (status: in progress). Call check_task(task_id) again later.
Task f1d49db0-fc7f-4bb4-885e-8c31cc9553f8 still running (status: in progress). Call check_task(task_id) again later.
[(1, 'in progress', 9.22154625, 43), (2, 'success', 8.751370866666667, 33), (3, 'in progress', 9.516139916666667, 39), (4, 'in progress', 9.570055116666667, 49), (5, 'in progress', 9.337573099999998, 55), (6, 'in progress', 9.355831349999999, 55), (7, 'in progress', 9.550803666666667, 46), (8, 'in progress', 9.642550983333333, 58), (9, 'success', 8.2200736, 41)]


In [52]:
se=await check_task(extra_task)
print(se.answer)
msgse=await get_task_messages(extra_task)
Path('audit_work/supplement_messages.json').write_text(json.dumps(msgse,indent=2))
cellse=extract_cells(msgse)
for i,c in enumerate(cellse):print(i,len(c),c[:180].replace('\n',' '))

## Files Ready for Analysis

| File | Description | Format | Size |
|---|---|---|---:|
| `/workspace/audit_work/supplement_nonvc.csv` | 22 non-VC funding candidate records, with the requested columns | CSV | 4,706 bytes |
| `/workspace/audit_work/supplement_nonvc_notes.md` | Original currencies, event-level evidence, exclusions, and recipient caveats | Markdown | 26,533 bytes |
| `/workspace/audit_work/supplement_nonvc_evidence.zip` | Supporting evidence: fetched source snapshots and search log | ZIP | 589,836 bytes |
| `/workspace/audit_work/supplement_nonvc_provenance.json` | Supporting metadata: source URLs, timestamps, archive members, and checksums | JSON | 19,075 bytes |
| `/workspace/audit_work/supplement_nonvc_sources.bib` | Supporting bibliography covering 38 fetched sources | BibTeX | 10,427 bytes |
| `/workspace/audit_work/retrieve_nonvc_sources.py` | Source-retrieval reproduction script | Python | 1,161 bytes |

**Important limitations:** Neither `/workspace/funders_audit_s

0 147 from pathlib import Path print(Path('/workspace/deal_rules.md').read_text()) print(Path('/workspace/funders_audit_sample.csv').read_text()[:22000])
1 248 from pathlib import Path import os print('cwd',os.getcwd()); print(list(Path('/workspace').iterdir())) for name in ('deal_rules.md','funders_audit_sample.csv'):  p=Path('/workspace
2 1034 import asyncio, json, time from e14c.web import web_search, fetch_parsed_page, save_citation out=Path('/workspace/audit_work'); out.mkdir(exist_ok=True) queries=[ 'Antmicro company
3 143 for s in searches[2:9]:  print('\nQUERY',s['query'])  for r in s.get('results',[]): print(r['title'],r['url'], '\n',r.get('snippet','')[:1400])
4 162 for i in (3,4,5,6,7,8):  print('\nQUERY',searches[i]['query'])  for r in searches[i].get('results',[]): print(r['title'],r['url'], '\n',r.get('snippet','')[:850])
5 159 for i in (4,5,6,7):  print('\nQUERY',searches[i]['query'])  for r in searches[i].get('results',[]): print(r['title'],r['url'], '\n',r.get('snippet

In [53]:
print(cellse[36])

import csv, datetime
rows=[]; eventnotes=[]
cols=['key','venture','type','label','date','amount_usd','lead','investors','source_url','confidence']
def add(key,venture,typ,label,date,usd,lead,investors,source,confidence,note):
 rows.append(dict(zip(cols,[key,venture,typ,label,date,usd,lead,investors,sources[source]['url'],confidence])))
 eventnotes.append(f'### {key}: {label}\n\n{note}\n\nEvidence: {sources[source]["url"]}. Snapshot: `sources/{Path(sources[source]["path"]).name}` in `supplement_nonvc_evidence.zip`.\n')
for key,label,date,amt,period in [
 ('antmicro_axiom_camera','AXIOM camera grant 645560','2014-12-15','EUR 136,850.00','2015-03-15 to 2016-06-14'),
 ('antmicro_xmine','X-MINE grant 730270','2017-05-02','EUR 470,750.00','2017-06-01 to 2021-08-31'),
 ('antmicro_vedliot_cordis','VEDLIoT grant 957197','2020-07-29','EUR 645,250.00','2020-11-01 to 2024-01-31'),
 ('antmicro_tristan','TRISTAN grant 101095947','2022-11-27','EUR 788,593.75','2022-12-01 to 2026-06-30')]:
 add('antmi

In [54]:
statuses=await asyncio.gather(*(check_task(t) for t in task_ids))
print([(i+1,s.status,s.runtime_minutes) for i,s in enumerate(statuses)])
for i,s in enumerate(statuses):
 if s.status=='success' and i!=1:
  print('GROUP',i+1,s.answer[:4000])

Task b167ec30-2689-496e-b683-3d6e527b2778 still running (status: in progress). Call check_task(task_id) again later.
Task 09c9f518-f5d8-4501-9671-a3f2ada07c89 still running (status: in progress). Call check_task(task_id) again later.


Task 84dea63a-6319-43c0-a0ff-0a9d45ebc0ba still running (status: in progress). Call check_task(task_id) again later.
Task 65955bd0-9605-41d7-80dd-9e036c8f6d8f still running (status: in progress). Call check_task(task_id) again later.
Task cb65c732-d0ce-4369-860f-29e5528ac62b still running (status: in progress). Call check_task(task_id) again later.
Task f1d49db0-fc7f-4bb4-885e-8c31cc9553f8 still running (status: in progress). Call check_task(task_id) again later.
Task 78387c12-5338-4c51-b039-365853670889 still running (status: in progress). Call check_task(task_id) again later.
[(1, 'in progress', 9.729401933333332), (2, 'success', 8.751370866666667), (3, 'in progress', 9.516139916666667), (4, 'in progress', 9.570055116666667), (5, 'in progress', 9.337573099999998), (6, 'in progress', 9.355831349999999), (7, 'in progress', 9.934820116666666), (8, 'in progress', 9.993023533333334)]


In [55]:
# Source map literals from the supplemental agent's executed cells.
for i in [8,10,19,23,27,32,33]: print('\nCELL',i,'\n',cellse[i][:8000])


CELL 8 
 for j in (2,3,4,5,6,7):
 s=more[j]; print('\nQUERY',s['query'])
 for r in s.get('results',[]): print(r['title'],r['url'], '\n',r.get('snippet','')[:700])

CELL 10 
 sources={}
async def get_source(key,url):
 try:
  text=await fetch_parsed_page(url)
  (out/'sources').mkdir(exist_ok=True)
  path=out/'sources'/f'{key}.md'; path.write_text(text)
  sources[key]={'url':url,'text':text,'path':str(path),'retrieved_utc':time.strftime('%Y-%m-%dT%H:%M:%SZ',time.gmtime())}
  return key,len(text),text[:80]
 except Exception as e: return key,str(e)
urls={
'neuromore_republic':'https://republic.com/neuromore',
'bitcraze_vinnova':'https://www.vinnova.se/en/p/efficiency-optimisation-of-mass-produced-micro-drone-propellers',
'farmbot_campaign':'https://farm.bot/blogs/news/the-farmbot-launch-campaign-wrapup',
'bela_kickstarter':'https://www.kickstarter.com/projects/423153472/bela-an-embedded-platform-for-low-latency-interact',
'bela_trill':'https://www.kickstarter.com/projects/423153472/trill-t

In [56]:
import ast
supp_source_map={}
for c in cellse:
 tree=ast.parse(c)
 for node in ast.walk(tree):
  if isinstance(node,ast.Dict):
   try: val=ast.literal_eval(node)
   except Exception:continue
   if val and all(isinstance(k,str) and isinstance(v,str) and v.startswith('http') for k,v in val.items()):
    supp_source_map.update(val)
print(supp_source_map)
# Re-fetch only supplement award pages relevant for adding direct funding events.
supp_keys=['antmicro_axiom_camera','antmicro_xmine','antmicro_vedliot_cordis','antmicro_tristan','bitcraze_vinnova','bela_kickstarter','bela_trill','bela_gliss','bcn3d_about','openbionics_sbri_gov','openbionics_crowdcube2024']
supp_sources={}
async def suppfetch(k):
 if k not in supp_source_map:return
 url=supp_source_map[k]
 async with sem:
  try:
   text=await fetch_parsed_page(url)
   supp_sources[k]={'url':url,'text':text,'path':k+'.md'}
  except Exception as e: supp_sources[k]={'url':url,'text':'ERROR '+str(e),'path':k+'.md'}
await asyncio.gather(*(suppfetch(k) for k in supp_keys))
Path('audit_work/supplement_sources_local.json').write_text(json.dumps(supp_sources,indent=2))
print({k:len(v['text']) for k,v in supp_sources.items()})

<unknown>:10: SyntaxWarning: invalid escape sequence '\d'


{'neuromore_republic': 'https://republic.com/neuromore', 'bitcraze_vinnova': 'https://www.vinnova.se/en/p/efficiency-optimisation-of-mass-produced-micro-drone-propellers', 'farmbot_campaign': 'https://farm.bot/blogs/news/the-farmbot-launch-campaign-wrapup', 'bela_kickstarter': 'https://www.kickstarter.com/projects/423153472/bela-an-embedded-platform-for-low-latency-interact', 'bela_trill': 'https://www.kickstarter.com/projects/423153472/trill-touch-sensing-for-makers', 'bela_gliss': 'https://www.crowdsupply.com/bela/gliss', 'openbionics_ukri': 'https://gtr.ukri.org/organisation/7925EC0D-C6CE-4AC6-B4CD-5ECCD5B342B0', 'efabless_google': 'https://opensource.googleblog.com/2023/11/open-source-pdks-joining-linux-foundation-chips-alliance.html', 'bcn3d_about': 'https://bcn3d.com/es/sobre-nosotros/', 'antmicro_vedliot': 'https://antmicro.com/blog/2021/01/vedliot-launch', 'antmicro_axiom': 'https://cordis.europa.eu/project/id/645496', 'antmicro_vedliot_cordis': 'https://cordis.europa.eu/projec

{'antmicro_axiom_camera': 21238, 'bitcraze_vinnova': 20919, 'antmicro_xmine': 38347, 'antmicro_tristan': 86841, 'bela_kickstarter': 57812, 'bela_gliss': 18462, 'bela_trill': 60440, 'openbionics_sbri_gov': 7470, 'antmicro_vedliot_cordis': 35008, 'bcn3d_about': 24644, 'openbionics_crowdcube2024': 10963}


In [57]:
# Inspect exact company shares, signature dates and campaign totals in fetched primary sources.
for k,v in supp_sources.items():
 txt=v['text']
 print('\n',k,txt.splitlines()[0])
 terms=['ANTMICRO','Signature date','317','Funded on','raised','£54','£25','54,902','25,375','ICEX','VIVALDI','HIBRI','697,464']
 for term in terms:
  m=re.search(re.escape(term),txt,re.I)
  if m: print(txt[max(0,m.start()-100):m.start()+700].replace('\n',' '))


 antmicro_axiom_camera Apertus° eXtendable Integrated Open Modular Cinema Camera | AXIOM | Project | Fact Sheet | H2020 | CORDIS | European Commission 
ng direct and indirect costs. This amount is a subset of the overall project budget.  € 322 125,00  ANTMICRO SP ZOO   Poland  Net EU contribution   Net EU financial contribution. The sum of money that the participant receives, deducted by the EU contribution to its linked third party. It considers the distribution of the EU financial contribution between direct beneficiaries of the project and other types of participants, like third-party participants.  € 136 850,00  Address  UL ZWIERZYNIECKA 3    60 813 POZNAN   See on map  SME   The organization defined itself as SME (small and medium-sized enterprise) at the time the Grant Agreement was signed.  Yes  Region  Makroregion województwo mazowieckie   Warszawski stołeczny   Miasto Warszawa  Activity type  Private for-profit entities (excludin
gital Object Identifier (DOI)  [10.3030/645560

In [58]:
from e14c.web import cite_doi
for k,v in supp_sources.items():
 if k.startswith('antmicro'):
  doi=re.search(r'10\.3030/\d+',v['text']).group(0)
  try:
   bib=await cite_doi(doi)
   print(doi,bib[:120])
  except Exception as exc:
   print('CORDIS DOI resolution unavailable; saving full web record:',doi,type(exc).__name__)
   retain({'url':v['url'],'title':v['text'].splitlines()[0],'full_text':v['text']},'European Commission')
 else:
  retain({'url':v['url'],'title':v['text'].splitlines()[0],'full_text':v['text']})
print('Supplement source citations saved.')

CORDIS DOI resolution unavailable; saving full web record: 10.3030/645560 ValueError


CORDIS DOI resolution unavailable; saving full web record: 10.3030/730270 ValueError


CORDIS DOI resolution unavailable; saving full web record: 10.3030/101095947 ValueError


CORDIS DOI resolution unavailable; saving full web record: 10.3030/957197 ValueError
Supplement source citations saved.


In [59]:
statuses=await asyncio.gather(*(check_task(t) for t in task_ids))
print([(i+1,s.status,s.runtime_minutes,s.steps) for i,s in enumerate(statuses)])
for i,s in enumerate(statuses):
 if s.status=='success':
  Path(f'audit_work/group{i+1}_summary.json').write_text(s.model_dump_json(indent=2))
  if i!=1:print('GROUP',i+1,s.answer[:4500])

Task 09c9f518-f5d8-4501-9671-a3f2ada07c89 still running (status: in progress). Call check_task(task_id) again later.


Task b167ec30-2689-496e-b683-3d6e527b2778 still running (status: in progress). Call check_task(task_id) again later.
Task f1d49db0-fc7f-4bb4-885e-8c31cc9553f8 still running (status: in progress). Call check_task(task_id) again later.
Task 65955bd0-9605-41d7-80dd-9e036c8f6d8f still running (status: in progress). Call check_task(task_id) again later.
Task 78387c12-5338-4c51-b039-365853670889 still running (status: in progress). Call check_task(task_id) again later.
[(1, 'success', 10.075688116666667, 44), (2, 'success', 8.751370866666667, 33), (3, 'in progress', 11.016556733333335, 41), (4, 'success', 10.524003850000001, 52), (5, 'in progress', 11.005176266666666, 58), (6, 'in progress', 11.013668433333333, 60), (7, 'in progress', 10.647739883333333, 49), (8, 'in progress', 10.941743133333334, 61)]
GROUP 1 ## Files Ready for Analysis

| File | Description | Format | Size |
|---|---|---|---|
| `/workspace/audit_work/group1.csv` | Funding audit: 14 rows, including explicitly flagged candid

In [60]:
allmsgs={2:msgs2}; allcells={2:cells2}
for n in [1,4]:
 messages=await get_task_messages(task_ids[n-1]); allmsgs[n]=messages; allcells[n]=extract_cells(messages)
 Path(f'audit_work/group{n}_messages.json').write_text(json.dumps(messages,indent=2))
 print('\nGROUP',n)
 for i,c in enumerate(allcells[n]):
  if 'rows' in c and ('append' in c or 'rows=' in c or 'rows =' in c): print(i,len(c),c[:240])


GROUP 1
36 3804 import csv, hashlib
cols=['key','venture','type','label','date','amount_usd','lead','investors','source_url','confidence']
rows=[]
def add(key,venture,type,label,date='',amount='',lead='',investors='',source='',confidence='high'):
 rows.app
37 7177 keys=['BruceDevices','Cheshire-Labs','coqui-ai','dbeaver','e2b-dev']
common='The requested deal_rules.md and funders_audit_sample.csv were absent after a recursive /workspace search; type values are provisional conventional funding-stage la
39 4513 # Save a small reproducibility/verification script; network refreshes go to a separate evidence file.
script='''"""Recheck the saved funding audit, optionally refresh the documented searches/pages.
Usage: python /workspace/audit_work/group1



GROUP 4
42 3061 import csv, hashlib
cols=['key','venture','type','label','date','amount_usd','lead','investors','source_url','confidence']
rows=[
 dict(zip(cols,['redisson','Redisson','none_found','No disclosed outside funding found','','','','','https://r
45 5306 from urllib.parse import urlparse
# Create complete BibTeX records for every searched/fetched source in the retained evidence,
# including rejected candidates; inclusion in bibliography is not endorsement.
registry={}
for s in evidence['sea
46 4494 # Pair grantor's recipient list with the company's explicit identification of the Buildspace award.
rows[1]['source_url']='https://buildspace.so/s5/ty-s5 | https://supermemory.ai/blog/supermemory-raises-3-million-and-building-the-best-memor


In [61]:
print(allcells[1][36]);print('\nGROUP4\n',allcells[4][42]);print('\nGROUP4 CHANGES\n',allcells[4][46][:5000])

import csv, hashlib
cols=['key','venture','type','label','date','amount_usd','lead','investors','source_url','confidence']
rows=[]
def add(key,venture,type,label,date='',amount='',lead='',investors='',source='',confidence='high'):
 rows.append(dict(zip(cols,[key,venture,type,label,date,str(amount),lead,investors,source,confidence])))
add('BruceDevices','Bruce Devices','crowdfunding','Ongoing Patreon community crowdfunding (campaign launched November 2024)','2024-11',source='https://graphtreon.com/creator/BruceFirmware',confidence='medium')
add('Cheshire-Labs','Cheshire Labs','none_found','No disclosed outside funding found after targeted searches',source='https://cheshirelabs.io/',confidence='medium')
add('coqui-ai','Coqui','convertible_note','Initial financing (Form D; convertible-note classification from CB Insights)','2021-10',1550000,investors='Mango Capital',source=securl,confidence='medium')
add('coqui-ai','Coqui','convertible_note','Convertible Note II (database-reported; indepe

In [62]:
print(allcells[4][42])
# Group1 actually prints CSV, recover verbatim.
for m in allmsgs[1]:
 txt=m.get('content')
 if m.get('role')=='tool' and isinstance(txt,str) and 'key,venture,type,label,date,amount_usd,lead,investors,source_url,confidence' in txt:
  csvtxt=txt[txt.index('key,venture,type,label,date,amount_usd,lead,investors,source_url,confidence'):].strip()
  print('CSV bytes',len(csvtxt));Path('audit_work/group1.csv').write_text(csvtxt+'\n')
print(pd.read_csv('audit_work/group1.csv').shape)

import csv, hashlib
cols=['key','venture','type','label','date','amount_usd','lead','investors','source_url','confidence']
rows=[
 dict(zip(cols,['redisson','Redisson','none_found','No disclosed outside funding found','','','','','https://redisson.pro/about.html','medium'])),
 dict(zip(cols,['supermemoryai','Supermemory','grant','Buildspace nights and weekends season 5 grant','2024-08','5000','Buildspace','','https://buildspace.so/s5/ty-s5','high'])),
 dict(zip(cols,['supermemoryai','Supermemory','grant','Reported Y Combinator Summer Fellows grant award; receipt unverified','','','Y Combinator','','https://analyticsindiamag.com/ai-features/the-20-year-old-who-said-no-to-yc','low'])),
 dict(zip(cols,['supermemoryai','Supermemory','seed','Pre-seed; company announced $3M, lead investor reported $2.6M','2025-10','3000000','Susa Ventures | Browder Capital | SF1.vc','Dane Knecht (person) | Theo Browne (person) | David Cramer (person) | Julian Weisser (person) | Jeff Dean (person) | Logan Kil

In [63]:
# Recover directly executable data-only segments and verbose per-venture notes.
ns4={};exec(allcells[4][42].split('# Save explicitly')[0],ns4)
ns4['rows'][1]['source_url']='https://buildspace.so/s5/ty-s5 | https://supermemory.ai/blog/supermemory-raises-3-million-and-building-the-best-memory-engine-for-llms'
pd.DataFrame(ns4['rows']).to_csv('audit_work/group4.csv',index=False)
print('GROUP1 NOTES CODE\n',allcells[1][37])
for i,c in enumerate(allcells[4]):
 if 'notes=' in c or 'notes =' in c:print('GROUP4 NOTES CELL',i,c[:18000])

GROUP1 NOTES CODE
 keys=['BruceDevices','Cheshire-Labs','coqui-ai','dbeaver','e2b-dev']
common='The requested deal_rules.md and funders_audit_sample.csv were absent after a recursive /workspace search; type values are provisional conventional funding-stage labels, not verified against those files. '
notes_text={
'BruceDevices': 'Searched exact venture/domain, firmware identity, investment/grants/accelerators, crowdfunding, Patreon and hardware sponsors; fetched official site, repository, donate page, Patreon (also rendered), Ko-fi, Open Collective and Graphtreon. One ongoing Patreon crowdfunding campaign is confirmed; Graphtreon reports launch 2024-11-02 and two paying members with about $13/month at fetch (search-index snapshots differed). The date records campaign launch, not a round close. amount_usd is blank because a monthly revenue rate is not a campaign total. Patreon, Ko-fi, Open Collective and cryptocurrency payment links are not separate confirmed funding events; no identifia

In [64]:
def literals_from_code(code):
 vals={}
 for node in ast.parse(code).body:
  if isinstance(node,ast.Assign):
   try:val=ast.literal_eval(node.value)
   except Exception:continue
   for t in node.targets:
    if isinstance(t,ast.Name): vals[t.id]=val
 return vals
lit1=literals_from_code(allcells[1][37]);print(lit1.keys())
Path('audit_work/group1_notes.json').write_text(json.dumps([{'key':k,'note':v,'incomplete':k in ['coqui-ai','dbeaver','e2b-dev'],'sources':[]} for k,v in lit1['notes_text'].items()],indent=2))
for i,c in enumerate(allcells[4]):
 lit=literals_from_code(c)
 if 'notes' in lit:
  print('notes recovered from',i)
  Path('audit_work/group4_notes.json').write_text(json.dumps(lit['notes'],indent=2))
print(Path('audit_work/group4_notes.json').read_text()[:11000])

dict_keys(['keys', 'common', 'notes_text', 'notes'])
notes recovered from 44
[
  {
    "key": "redisson",
    "note": "Fresh searches covered Redisson/redisson.pro company and founder history, outside investment, seed/angel/accelerator, grants, debt and sponsorship; fetched first-party About, Alibaba founder interview and Crunchbase. No disclosed outside-capital event found; none_found is not proof of no financing and no sourced bootstrapping claim was found. Unrelated Redison music-device crowdfunding was excluded. All five ventures were researched, but this venture has no unresolved material funding signal. Required deal_rules.md and funders_audit_sample.csv were absent; explicit prompt schema and categories were followed.",
    "incomplete": false,
    "sources": [
      "https://redisson.pro/about.html",
      "https://www.alibabacloud.com/blog/interview-with-the-creator-of-redisson-building-an-open-source-enterprise-redis-client_593854",
      "https://www.crunchbase.com/organizat

In [65]:
n=3;s=await check_task(task_ids[n-1]);print(s.answer)
messages=await get_task_messages(task_ids[n-1]);allmsgs[n]=messages;allcells[n]=extract_cells(messages)
Path(f'audit_work/group{n}_messages.json').write_text(json.dumps(messages,indent=2))
for i,c in enumerate(allcells[n]):
 if ('rows' in c and ('append' in c or 'rows=' in c or 'rows =' in c)) or 'notes=' in c:print(i,len(c),c[:220])

## Files Ready for Analysis

| File | Description | Format | Size |
|---|---|---|---|
| `/workspace/audit_work/group3.csv` | Funding audit: 27 event rows across five ventures; read with caveats | CSV | 6.4 KB |
| `/workspace/audit_work/group3_notes.json` | Per-venture search scope, uncertainties, and completeness flags | JSON | 16.3 KB |
| `/workspace/audit_work/group3_evidence.json` | Fresh search/fetch evidence and file-level provenance | JSON | 2.34 MB |
| `/workspace/audit_work/group3_sources.bib` | 191 source citations, saved using `e14c.web.save_citation` | BibTeX | 91.9 KB |
| `/workspace/audit_work/group3_retrieve.py` | Snapshot reproduction and optional fresh retrieval | Python | 2.9 KB |

Source URLs, retrieval parameters, the November 26, 2026 cutoff, and transformations for these files are recorded in `group3_evidence.json`. No independent currency conversions were made. Reproduction instructions are in the saved script.

**Incomplete histories: 3 — MindsDB, neuromore, and 

34 7715 import csv
rows=[]
cols=['key','venture','type','label','date','amount_usd','lead','investors','source_url','confidence']
def event(key,venture,typ,label,date,amount,lead,investors,url,confidence='high'):
 rows.append(di
36 11996 shared = (' The requested deal_rules.md and funders_audit_sample.csv were not present, so type codes are provisional: accelerator, pre_seed, seed, series_a, series_b, grant, debt, secondary, crowdfunding. Confidence is h
39 3977 script = '''#!/usr/bin/env python3
"""Reproduce group 3 audited tables and optionally repeat their fresh web evidence retrieval.
Usage: python group3_retrieve.py [--refresh] [--output /workspace/audit_work/group3_reprodu
40 3176 file_provenance=[]
for name,description,role,transformation in [
 ('group3.csv','Event-level outside-funding audit for the five requested ventures; qualified and low-confidence events must be interpreted with notes.','an


In [66]:
print(allcells[3][34])
print('\nNOTES prefix\n',allcells[3][36][:3000])

import csv
rows=[]
cols=['key','venture','type','label','date','amount_usd','lead','investors','source_url','confidence']
def event(key,venture,typ,label,date,amount,lead,investors,url,confidence='high'):
 rows.append(dict(zip(cols,[key,venture,typ,label,date,str(amount) if amount!='' else '',lead,investors,url,confidence])))
# MindsDB: round-specific participants only; no cumulative total entered as a new round.
event('mindsdb','MindsDB','accelerator','Berkeley SkyDeck Spring 2018 first investment (month and amount unconfirmed)','','','Berkeley SkyDeck Fund','','https://www.unite.ai/jorge-torres-co-founder-ceo-of-mindsdb-interview-series/','medium')
event('mindsdb','MindsDB','pre_seed','2018 pre-seed (reported amounts conflict)','2018-10','','','Speedinvest | Tuck Lye Koh (person) | Berkeley SkyDeck Fund | Angel Invest','https://www.crunchbase.com/funding_round/mindsdb-pre-seed--c7f4d965','medium')
event('mindsdb','MindsDB','accelerator','Y Combinator Winter 2020 investment (amount un

In [67]:
# Recover group3 event definitions and notes from executed data-building code.
ns3={'evidence':{'fetches':[]}}
exec(allcells[3][34].split('with (out/')[0],ns3)
exec(allcells[3][36].split('with (out/')[0].split("(out/'group3_notes.json')")[0],ns3) if False else None
print(allcells[3][36][-1900:])
pd.DataFrame(ns3['rows']).to_csv('audit_work/group3.csv',index=False)
print(pd.DataFrame(ns3['rows'])[['key','type','label','date','amount_usd']].to_string(index=False))

reference files were absent.') ,'incomplete':False,'sources':checked(['payload','2018/07/10/200000','whitmer-announces-148','launch-week-day-2-fundraise','mlive.com','crainsgrandrapids'])})
(out/'group3_notes.json').write_text(json.dumps(notes,indent=2,ensure_ascii=False))
evidence['audit_cutoff']='2026-11-26'
evidence['scope']=[{'key':k,'venture':v} for k,v in [('mindsdb','MindsDB'),('neuromore','neuromore'),('odoo','Odoo SA'),('openbq-org','OpenBB Inc.'),('payloadcms','Payload')]]
evidence['method']={'sources':'Fresh e14c.web.web_search results followed by e14c.web.fetch_parsed_page; raw outputs preserved, including failures.','taxonomy':'Provisional because /workspace/deal_rules.md and /workspace/funders_audit_sample.csv were absent.','citation_skill':'Requested citation-integrity using load_skill; tool reported skill not found. BibTeX saved with documented e14c.web.save_citation anyway.','currency':'Only directly reported USD amounts are numeric; euro-only amounts are stated in lab

In [68]:
exec(allcells[3][36].split("(out/'group3_notes.json')")[0],ns3)
Path('audit_work/group3_notes.json').write_text(json.dumps(ns3['notes'],indent=2))
print('Recovered',len(ns3['notes']),'group3 notes')
# Preserve final group2 lead correction.
ns2['rows'][3]['lead']='';ns2['rows'][3]['investors']='Y Combinator'
pd.DataFrame(ns2['rows']).to_csv('audit_work/group2.csv',index=False)
# Spot-check the questionable Payload award, avoiding amounts never paid.
print(next(n['note'] for n in ns3['notes'] if n['key']=='payloadcms'))

Recovered 5 group3 notes
Fresh searches covered Start Garden 2018 awards, YC S22, the November 2022 seed, Michigan grants, later follow-ons and the 2025 Figma acquisition. Five disclosed events/awards are tabulated, not licensing or acquisition proceeds. Contemporaneous Fox17 July 10, 2018 coverage says all 100 April finalists received $1K and explicitly names Payload/James Mikrut among the later $20K winners; the distinct April $1K and July $20K awards are retained, with medium confidence on the finalist allocation. A later Crain’s caption calls the $20K check the first outside funding, so that shorthand discrepancy is noted rather than silently dropping the earlier contemporaneous award. June 2022 $500K YC investment is explicit in Payload’s own post and dated by June 21 Crain’s coverage. November 2022 $4.7M seed and lead Gradient Ventures come from Payload’s release; Payload’s contemporaneous LinkedIn post adds Rogue Capital and Twenty Two Ventures as participants. YC appears explic

In [69]:
statuses=await asyncio.gather(*(check_task(task_ids[i-1]) for i in [5,6,7,8]))
print([(n,s.status,s.runtime_minutes,s.steps) for n,s in zip([5,6,7,8],statuses)])
for n,s in zip([5,6,7,8],statuses):
 if s.status=='success':
  print('GROUP',n,s.answer[:2000])
  messages=await get_task_messages(task_ids[n-1]);allmsgs[n]=messages;allcells[n]=extract_cells(messages)
  Path(f'audit_work/group{n}_messages.json').write_text(json.dumps(messages,indent=2))
  for i,c in enumerate(allcells[n]):
   if ('rows' in c and ('append' in c or 'rows=' in c or 'rows =' in c)) or 'notes=' in c:print(i,len(c),c[:160])

Task 65955bd0-9605-41d7-80dd-9e036c8f6d8f still running (status: in progress). Call check_task(task_id) again later.


Task 78387c12-5338-4c51-b039-365853670889 still running (status: in progress). Call check_task(task_id) again later.
[(5, 'success', 12.5139271, 62), (6, 'in progress', 12.821996766666667, 65), (7, 'in progress', 12.456054466666666, 52), (8, 'success', 12.807476483333334, 65)]
GROUP 5 ## Files Ready for Analysis

| File | Description | Format | Size |
|---|---|---|---|
| `/workspace/audit_work/group5.csv` | Funding audit: 29 events across five ventures | CSV | 6.2 KB |
| `/workspace/audit_work/group5_notes.json` | Search scope, uncertainties and completeness by venture | JSON | 12.7 KB |
| `/workspace/audit_work/group5_evidence.json` | Search/fetch evidence and full provenance manifest | JSON | 4.1 MB |
| `/workspace/audit_work/group5_sources.bib` | Supporting bibliography: 65 fetched sources | BibTeX | 27.5 KB |
| `/workspace/audit_work/group5_retrieve.py` | Regeneration and optional source-refresh script | Python | 35.7 KB |
| `/workspace/audit_work/group5_EXCEED_official.pdf` | Offi

48 4491 import csv
columns=['key','venture','type','label','date','amount_usd','lead','investors','source_url','confidence']
ventures={'anaconda':'Anaconda, Inc. (Conti
50 9579 anaconda_sources=[secidx[-1],urls[2],urls[1],urls[3],secdoc[2],secdoc[1],urls[4],newurls[2],secdoc[0],extra[0],urls[5],newurls[0],urls[0],extra[1],u6[4],newurls
52 3648 from urllib.parse import urlparse
import hashlib
search_meta={}
for s in evidence['searches']:
 for x in s['results']:
  if x.get('url'):search_meta.setdefault(
53 5041 # Save a self-contained regeneration script, with optional fresh web verification.
script = '''#!/usr/bin/env python3
"""Regenerate the manually curated funding
55 4205 import subprocess, sys
run=subprocess.run([sys.executable,str(out/'group5_retrieve.py')],text=True,capture_output=True,timeout=30)
assert run.returncode==0,run.
GROUP 8 ## Files Ready for Analysis

| File | Description | Format | Size |
|---|---|---|---:|
| `/workspace/audit_work/group8.csv` | Analysis-ready audit:

54 7527 # Persist visual readings of image-only SEC filings as audit evidence.
evidence['visual_transcriptions']=[{'url':pdf_sources[0],'pages':'1, 2, 4, 6','facts':{'i
56 9022 add('va-linux','VA Linux Systems','debt','Geeknet renewal of existing USD 5 million revolver; Q4 2012; no new draw','','','Wells Fargo','','https://www.sec.gov/
57 3713 import hashlib,urllib.parse
# Complete source ledger: fetched pages, visually inspected scans, plus every discovered
# source that was actually used for a row o
58 5198 script='''#!/usr/bin/env python3
"""Reproduce audited tables from the immutable retrieved evidence snapshot.
Usage: python group8_reproduce.py
The snapshot cont


In [70]:
print('GROUP5 DATA\n',allcells[5][48]);print('GROUP8 DATA\n',allcells[8][54]);

GROUP5 DATA
 import csv
columns=['key','venture','type','label','date','amount_usd','lead','investors','source_url','confidence']
ventures={'anaconda':'Anaconda, Inc. (Continuum Analytics)','antmicro':'Antmicro','bela':'Augmented Instruments (Bela)','bcn3d':'BCN3D Technologies','bitcraze':'Bitcraze'}
rows=[]
def add(key,typ,label,date='',amount='',lead='',investors='',url='',confidence='high'):
 rows.append(dict(zip(columns,[key,ventures[key],typ,label,date,str(amount),lead,investors,url,confidence])))
add('anaconda','seed','Seed equity private placement; final amended amount sold','2011-12',2250000,url=secidx[-1])
add('anaconda','grant','DARPA XDATA research award','2013-02',3000000,investors='DARPA',url=urls[2])
add('anaconda','series_a','Series A','2015-07',24000000,lead='General Catalyst | BuildGroup',url=urls[1])
add('anaconda','debt','Debt financing','2015-12',10000000,lead='Silicon Valley Bank',url=urls[3])
add('anaconda','venture','Equity private placement; stage not specified 

In [71]:
for n in [5,8]:
 print('\nGROUP',n)
 needed=['secidx','urls','secdoc','newurls','pdfurl','u_more','u6','extra','lasturls','vurl'] if n==5 else ['pdf_sources','other_sources','sys_urls','va_more','sec_va','va_tail','later_va']
 for name in needed:
  print('\nVAR',name)
  for i,c in enumerate(allcells[n]):
   if re.search(r'^'+name+r'\s*=',c,re.M):
    tree=ast.parse(c)
    for node in tree.body:
     if isinstance(node,ast.Assign) and any(isinstance(t,ast.Name) and t.id==name for t in node.targets):print(i,ast.get_source_segment(c,node)[:5000])


GROUP 5

VAR secidx
24 secidx=[re.search(r'\((https://www.sec.gov[^)]+)\)',l).group(1) for l in t_more[10].splitlines() if 'sec.gov' in l]

VAR urls
9 urls=[rr[0][0]['url'],r2[0][0]['url'],r2[0][2]['url'],r2[0][3]['url'],r2[1][0]['url'],r2[1][6]['url'],r2[3][0]['url'],rr[2][0]['url'],r2[5][3]['url'],r2[5][4]['url'],'https://cordis.europa.eu/project/id/645560','https://cordis.europa.eu/project/id/957197']

VAR secdoc
25 secdoc=[re.search(r'\((https://www.sec.gov[^)]+/xslFormDX01/primary_doc.xml)\)',t).group(1) for t in sect[:4]]

VAR newurls
35 newurls=[r9[1][0]['url'],r9[3][0]['url'],'https://projects.propublica.org/coronavirus/bailouts/loans/anaconda-inc-5365807309','https://www.exceed-padr.eu/exceed-consortium/','https://antmicro.com/blog/2021/01/vedliot-launch','https://antmicro.com/blog/2017/06/x-mine-press-release',r9[4][2]['url']]

VAR pdfurl
39 pdfurl='https://eda.europa.eu/docs/default-source/padr/padr-exceed-projectweb.pdf'

VAR u_more
17 u_more=['https://cordis.europa.eu/pro

In [72]:
for n in [5,8]:
 for m in allmsgs[n]:
  txt=m.get('content')
  if m.get('role')=='tool' and isinstance(txt,str) and 'key,venture,type,label,date,amount_usd,lead,investors,source_url,confidence' in txt:
   csvtxt=txt[txt.index('key,venture,type,label,date,amount_usd,lead,investors,source_url,confidence'):].strip()
   print('GROUP',n,'CSV printed',len(csvtxt));print(csvtxt[:800])
 print('Last cells:',[(i,c[:140].replace('\n',' ')) for i,c in enumerate(allcells[n])][-10:])

GROUP 5 CSV printed 1314
key,venture,type,label,date,amount_usd,lead,investors,source_url,confidence`.
- Use the types defined by the missing rules; `none_found` is allowed only after reasonable searches.
- Dates must be `YYYY-MM`; leave month blank when unknown and explain that in notes.
- Amounts must be digits in USD; leave undisclosed amounts blank and do not make unsupported currency conversions.
- Format people as `Full Name (person)`, firms by their usual names, separate multiple investors with ` | `, and exclude the lead from `investors`.
- A source must actually name an investor or state an amount; prefer company/investor filings and reputable news, and do not infer investors between rounds.
- Do not treat acquisitions or ordinary IPOs as funding; include an IPO only when outside capital was issued and th
Last cells: [(46, 'q13=[\'"Bitcraze" site:vinnova.se/en/p/\',\'"Bitcraze" grant funding propeller 2025 2026\',\'"Antmicro" grants "projects" "EU" -github -grokipedi'), (47, '

In [73]:
for n in [5,8]:
 print('\nGROUP',n)
 for m in allmsgs[n]:
  txt=m.get('content')
  if m.get('role')=='tool' and isinstance(txt,str):
   if ('FETCH' in txt or 'https://' in txt) and len(txt)<7000:print(txt[:7000])


GROUP 5
[Appended Cell #7] [stdout]

QUERY Continuum Analytics funding 2012 2013 2014 2015 2016 2017 2018 series $ DARPA
Python distribution vendor Continuum Analytics raises $24M | VentureBeat https://venturebeat.com/business/python-distribution-vendor-continuum-analytics-raises-24m
DARPA Backs Python Big Data Projects With $3M Cash - SiliconANGLE https://siliconangle.com/2013/02/07/darpa-backs-python-big-data-projects-with-3m-cash/
Python gets a big data boost from DARPA – Computerworld https://www.computerworld.com/article/1405122/python-gets-a-big-data-boost-from-darpa.html
Continuum Analytics snags $10 mln from SVB https://www.privateequityinternational.com/continuum-analytics-snags-10-mln-from-svb/
Continuum Analytics Gets $24 Million in Venture Capital - SiliconHills https://www.siliconhillsnews.com/2015/07/23/continuum-analytics-gets-24-million-in-venture-capital/
Continuum Analytics funding: rounds and investors https://millionrounds.com/company/continuum-analytics
Austin tec

In [74]:
tooltexts={}
for n in [5,8]:
 text='\n'.join(m.get('content','') for m in allmsgs[n] if m.get('role')=='tool' and isinstance(m.get('content'),str))
 tooltexts[n]=text
 Path(f'audit_work/group{n}_tool_evidence.txt').write_text(text)
 print('GROUP',n,'retained tool evidence chars',len(text))

GROUP 5 retained tool evidence chars 453747
GROUP 8 retained tool evidence chars 442373


In [75]:
# Literal + proxy reconstruction avoids any re-execution of remote networking or arbitrary code.
class SourceRefs:
 def __init__(self,name):self.name=name
 def __getitem__(self,i):return f'REF:{self.name}:{i}'
ns5={name:SourceRefs(name) for name in ['secidx','urls','secdoc','newurls','u_more','u6','extra','lasturls']}
ns5.update(pdfurl='https://eda.europa.eu/docs/default-source/padr/padr-exceed-projectweb.pdf',vurl=supp_source_map['bitcraze_vinnova'])
exec(allcells[5][48].split('with (out/')[0],ns5)
print(pd.DataFrame(ns5['rows'])[['key','label','source_url']].to_string(index=False))
# Available source URLs containing SEC primary document or specific BCN3D funding keywords.
for u in dict.fromkeys(re.findall(r'https?://[^\s\]\)\}\>\"\'|,]+',tooltexts[5])):
 if ('sec.gov' in u and ('1538860' in u)) or any(s in u for s in ['seed-funding','3-million','bcn3d-raises','bcn3d-secures','innvierte-destina']):print(u)

     key                                                                                     label                                                                                  source_url
anaconda                                  Seed equity private placement; final amended amount sold                                                                               REF:secidx:-1
anaconda                                                                DARPA XDATA research award                                                                                  REF:urls:2
anaconda                                                                                  Series A                                                                                  REF:urls:1
anaconda                                                                            Debt financing                                                                                  REF:urls:3
anaconda                                   Eq

In [76]:
resolved5={
 'REF:secidx:-1':'https://www.sec.gov/Archives/edgar/data/1538860/000153886012000001/0001538860-12-000001-index.htm',
 'REF:urls:0':'https://www.anaconda.com/press/anaconda-raises-150m-series-c-funding-ai-enterprise',
 'REF:urls:1':'https://venturebeat.com/business/python-distribution-vendor-continuum-analytics-raises-24m',
 'REF:urls:2':'https://www.computerworld.com/article/1405122/python-gets-a-big-data-boost-from-darpa.html',
 'REF:urls:3':'https://www.privateequityinternational.com/continuum-analytics-snags-10-mln-from-svb/',
 'REF:urls:4':'https://www.anaconda.com/blog/anaconda-funded-by-citi-ventures',
 'REF:urls:5':'https://www.businessinsider.com/snowflake-announces-partnership-with-anaconda-python-data-science-2021-9',
 'REF:secdoc:0':'https://www.sec.gov/Archives/edgar/data/1538860/000153886021000002/xslFormDX01/primary_doc.xml',
 'REF:secdoc:1':'https://www.sec.gov/Archives/edgar/data/1538860/000153886017000003/xslFormDX01/primary_doc.xml',
 'REF:secdoc:2':'https://www.sec.gov/Archives/edgar/data/1538860/000153886016000004/xslFormDX01/primary_doc.xml',
 'REF:newurls:0':'https://www.anaconda.com/blog/accelerating-innovation-anaconda-announces-financing-for-the-future',
 'REF:newurls:2':'https://projects.propublica.org/coronavirus/bailouts/loans/anaconda-inc-5365807309',
 'REF:urls:6':supp_source_map['bela_trill'],'REF:urls:7':supp_source_map['bela_kickstarter'],
 'REF:u_more:2':'https://www.crowdsupply.com/bela/bela-gem-stereo-and-multi',
 'REF:u_more:3':supp_source_map['bela_gliss'],
 'REF:u_more:5':'https://bcn3d.com/bcn3d-spin-offs-announces-3-million-funding-idex-3d-printing-technology/',
 'REF:u_more:6':'https://www.segre.com/es/economia/241117/la-empresa-bcn3d-traslada-su-produccion-a-lleida_635204.html',
 'REF:u6:6':'https://bcn3d.com/bcn3d-secures-e2-8-million-funding-round-led-by-cdti-and-spanish-industrial-business-group-mondragon-during-pandemic/',
 'REF:u6:7':'https://bcn3d.com/bcn3d-secures-a-new-investment-round-to-fuel-light-industrial-3d-printing-and-unveils-new-factory-on-omega-i60-first-anniversary/',
 'REF:extra:3':'https://www.europapress.es/ciencia/noticia-cdti-aprueba-nueva-coinversion-millon-euros-traves-innvierte-20211103123208.html',
 'REF:lasturls:3':'https://capital-riesgo.es/es/articles/innvierte-destina-en-febrero-cerca-de-8-millones-de-euros-a-seis-nuevas-coinversiones-en-capital-riesgo/',
 'REF:extra:2':'https://www.bitcraze.io/2014/03/joining-the-business-incubator-minc/'
}
for row in ns5['rows']:
 row['source_url']=resolved5.get(row['source_url'],row['source_url'])
 assert row['source_url'].startswith('http')
pd.DataFrame(ns5['rows']).to_csv('audit_work/group5.csv',index=False)
print('Group5 rows recovered:',len(ns5['rows']))
# Notes are append calls with lookup-list sources; preserve literal note text and completeness safely via AST.
def recover_note_objects(code):
 notes=[]
 for node in ast.walk(ast.parse(code)):
  if isinstance(node,ast.Dict):
   keys=[]
   for k in node.keys:
    try:keys.append(ast.literal_eval(k))
    except Exception:keys.append(None)
   if 'key' in keys and 'note' in keys and 'incomplete' in keys:
    obj={}
    for k,v in zip(keys,node.values):
     try:obj[k]=ast.literal_eval(v)
     except Exception:
      if k=='note':
       # Concatenations involving a shared caution can be evaluated with shared=''.
       try:obj[k]=eval(compile(ast.Expression(v),'<notes>','eval'),{'shared':'','common':''})
       except Exception:obj[k]=ast.unparse(v)
      elif k=='sources':obj[k]=[]
    notes.append(obj)
 return notes
notes5=recover_note_objects(allcells[5][50]);print([(x['key'],x.get('incomplete')) for x in notes5])
Path('audit_work/group5_notes.json').write_text(json.dumps(notes5,indent=2))

Group5 rows recovered: 29
[('anaconda', True), ('antmicro', False), ('bela', False), ('bcn3d', False), ('bitcraze', False)]


8373

In [77]:
# Group8 uses literal URLs for most source arrays; resolve the few dynamic ones from its original source list.
ns8={}
for c in allcells[8][:54]:
 ns8.update(literals_from_code(c))
print({k:ns8.get(k) for k in ['urls','other_sources','sys_urls','va_more','sec_va','va_tail','later_va']})
print(allcells[8][19]);print(allcells[8][54][1900:4600]);print(allcells[8][56][:1700])

{'urls': ['https://pydantic.dev/articles/company-announcement', 'https://pydantic.dev/articles/logfire-announcement', 'https://www.radixark.com/blog/radixark-launches-100m-seed', 'https://www.businesswire.com/news/home/20260505077157/en/RadixArk-Launches-with-%24100-Million-in-Seed-Funding-Led-by-Accel-to-Grow-SGLang-and-Democratize-Frontier-AI-Infrastructure'], 'other_sources': ['https://www.networkworld.com/article/940972/container-visibility-startup-sysdig-lands-series-a-funding-aims-to-resolve-container-visibility-iss.html', 'https://www.washingtontechnology.com/2019/05/ic-venture-arm-backs-automation-cloud-security-tech-firms/326818/', 'https://techcrunch.com/2023/02/16/sequoia-backs-open-source-data-validation-framework-pydantic-to-commercialize-with-cloud-services/', 'https://techcrunch.com/2026/01/21/sources-project-sglang-spins-out-as-radixark-with-400m-valuation-as-inference-market-explodes/'], 'sys_urls': [], 'va_more': ['https://www.linuxjournal.com/article/2576', 'https://

In [78]:
# Recover Sysdig source-index order printed immediately after its construction.
for m in allmsgs[8]:
 txt=m.get('content')
 if m.get('role')=='tool' and isinstance(txt,str) and 'sysdig.com/press-releases/' in txt:
  lines=[l.strip() for l in txt.splitlines() if l.startswith('https://sysdig.com/press-releases/') or l.startswith('https://www.sysdig.com/press-releases/')]
  if len(lines)>=5: print('\n'.join(lines[:20]));ns8['sys_urls']=lines;break
print('Nicira new',ns8.get('nic_new'))
print(allcells[8][54][:2700])

https://www.sysdig.com/press-releases/350m-series-g-financing-2-5b-valuation
https://www.sysdig.com/press-releases/sysdig-series-b-container-monitoring
https://www.sysdig.com/press-releases/sysdig-partners-with-in-q-tel
https://www.sysdig.com/press-releases/sysdig-announces-10-7-million-in-series-a-funding-and-launches-native-monitoring-for-container-revolution-with-sysdig-cloud
https://www.sysdig.com/press-releases/sysdig-raises-25m-for-the-container-intelligence-platform
https://www.sysdig.com/press-releases/sysdig-closes-series-d-funding
https://www.sysdig.com/press-releases/sysdig-closes-70m-series-e-funding
https://www.sysdig.com/press-releases/sysdig-raises-188m-at-a-valuation-of-1-19b-to-secure-modern-cloud-applications
Nicira new []
# Persist visual readings of image-only SEC filings as audit evidence.
evidence['visual_transcriptions']=[{'url':pdf_sources[0],'pages':'1, 2, 4, 6','facts':{'issuer':'Nicira Networks, Inc.','filing_received':'2008-07-03','signature_date':'2008-06-2

In [79]:
# Resolve Nicira's electronic-filings list as printed in its fetch ledger.
for i,c in enumerate(allcells[8]):
 if 'nic_new' in c and i<54: print('CELL',i,c[:4000])

CELL 23 for s in evidence['searches']:
 if s['query'] in ['VA Research Intel investment February 19 1999 1.5 million','VA Linux September 1999 4.8 million Series B funding','VA Linux 1999 Silicon Valley Bank 4 million equipment loan']:
  print('\nQUERY',s['query'])
  for r in s['results']:print(r['title'],r['url'],'\n',(r.get('snippet')or r.get('full_text') or '')[:2300])
# More filing links for Nicira retrieved from discovered filing history (not guessed).
nic_new=[]
for url in re.findall(r'\]\((https?://[^)]+)\)',gettxt(sec_nicira[2])):
 if 'sec.gov/Archives/edgar/data/' in url:
  u=url.replace('http:','https:')
  if u not in nic_new:nic_new.append(u)
print('\nNICIRA SOURCES',nic_new)
await asyncio.gather(*[fetch(u) for u in nic_new]);

CELL 24 for u in nic_new:
 print('\nURL',u)
 t=gettxt(u)
 for term in ['Date of First Sale','Total Offering Amount','Total Amount Sold','Clarification of Response','Series A','Series B','Series C']:
  for m in list(re.finditer(re.escape(term),t,re.I))

In [80]:
mm=re.search(r'NICIRA SOURCES (\[.*?\])',tooltexts[8]); ns8['nic_new']=ast.literal_eval(mm.group(1));print(ns8['nic_new'])
ns8['evidence']={}
exec(allcells[8][54],ns8)
# Only first statement of follow-up adds a 2012 revolver renewal.
exec(allcells[8][56].split('# Keep chronological')[0],ns8)
pd.DataFrame(ns8['rows']).to_csv('audit_work/group8.csv',index=False)
notes8=recover_note_objects(allcells[8][56]);Path('audit_work/group8_notes.json').write_text(json.dumps(notes8,indent=2))
print('Recovered',len(ns8['rows']),'rows and',len(notes8),'notes.')

['https://sec.gov/Archives/edgar/data/1439476/000143947611000001/xslFormDX01/primary_doc.xml', 'https://sec.gov/Archives/edgar/data/1439476/000143947610000007/xslFormDX01/primary_doc.xml', 'https://sec.gov/Archives/edgar/data/1439476/000143947610000005/xslFormDX01/primary_doc.xml', 'https://sec.gov/Archives/edgar/data/1439476/000143947610000004/xslFormDX01/primary_doc.xml', 'https://sec.gov/Archives/edgar/data/1439476/000143947610000001/xslFormDX01/primary_doc.xml', 'https://sec.gov/Archives/edgar/data/1439476/000143947609000004/xslFormDX01/primary_doc.xml', 'https://sec.gov/Archives/edgar/data/1439476/000143947609000003/xslFormDX01/primary_doc.xml']
Rows 31 by venture {'nicira': 6, 'pydantic': 2, 'radixark': 2, 'sysdig': 9, 'va-linux': 12}
Recovered 32 rows and 5 notes.


In [81]:
statuses=await asyncio.gather(*(check_task(task_ids[i-1]) for i in [6,7]))
for n,s in zip([6,7],statuses):
 print('GROUP',n,s.status,s.answer[:2300])
 if s.status=='success':
  messages=await get_task_messages(task_ids[n-1]);allmsgs[n]=messages;allcells[n]=extract_cells(messages)
  Path(f'audit_work/group{n}_messages.json').write_text(json.dumps(messages,indent=2))
  for i,c in enumerate(allcells[n]):
   if ('rows' in c and ('append' in c or 'rows=' in c or 'rows =' in c)) or 'notes=' in c:print(i,len(c),c[:200])

Task 78387c12-5338-4c51-b039-365853670889 still running (status: in progress). Call check_task(task_id) again later.
GROUP 6 in progress 
GROUP 7 success ## Files Ready for Analysis

| File | Description | Format | Size |
|---|---|---|---|
| `/workspace/audit_work/group7.csv` | 28 funding-event rows | CSV | 6.9 KB |
| `/workspace/audit_work/group7_notes.json` | Search scope, uncertainties and completeness for all five ventures | JSON | 15.7 KB |
| `/workspace/audit_work/group7_evidence.json` | Supporting search/fetch evidence and file provenance | JSON | 5.5 MB |
| `/workspace/audit_work/group7_sources.bib` | 69 source citations, also saved through `save_citation` | BibTeX | 28.8 KB |
| `/workspace/audit_work/group7_reproduce.py` | Rebuild and validation script | Python | 2.4 KB |

**Incomplete: 3 ventures — Airbyte, ClearML and JFrog.** Material unresolved details and excluded unsupported events are documented in their notes. Imply and Meltano received reasonable research coverage.

O

44 5611 import csv,hashlib
cols=['key','venture','type','label','date','amount_usd','lead','investors','source_url','confidence']
rows=[]
def add(k,v,typ,label,date,amount,lead,investors,url,confidence='high'
45 1490 # Final source retrieval for the corrected personal name and the unresolved predecessor extension.
_=await fetch('https://www.elastic.co/about/leadership')
u=rr3[3][3]['url']
async with httpx.AsyncCli
46 10185 # Each venture has one narrative note; sources are URLs actually fetched, not merely search-result links.
allfetch=[x['url'] for x in evidence['fetches'] if x['url'].startswith('https://')]
def releva
47 4317 from urllib.parse import urlparse
# Cite every fetched source (including checked but blocked pages), plus every CSV/notes source.
searchmeta={}
for se in evidence['searches']:
 for item in se['results
48 5250 # Save a concise, executable reproduction/validation script. The evidence snapshot is the immutable input.
script='''#!/usr/bin/env python3
"""Rebuild the 

In [82]:
print(allcells[7][44]);print(allcells[7][45][:1600])

import csv,hashlib
cols=['key','venture','type','label','date','amount_usd','lead','investors','source_url','confidence']
rows=[]
def add(k,v,typ,label,date,amount,lead,investors,url,confidence='high'):
 rows.append(dict(zip(cols,[k,v,typ,label,date,str(amount) if amount is not None else '',lead,investors,url,confidence])))
A='Airbyte'; C='ClearML (Allegro AI)'; I='Imply'; J='JFrog'; M='Meltano'
aearly=more2[1][3]['url']
add('airbyte',A,'seed','Pre-seed','2020-01',600000,'','',aearly)
add('airbyte',A,'accelerator','Y Combinator Winter 2020; individual investment terms undisclosed','',None,'','Y Combinator',more2[1][0]['url'],'medium')
add('airbyte',A,'seed','Seed','2020-05',1200000,'','',aearly)
add('airbyte',A,'seed','Seed; announced March 2021, closed February','2021-03',5200000,'Accel','Y Combinator | 8VC | Calvin French-Owen (person) | Charles Zedlewski (person) | Auren Hoffman (person) | Travis May (person) | Alain Rossmann (person)','https://www.globenewswire.com/news-release/202

In [83]:
class NestedRefs:
 def __init__(self,s):self.s=s
 def __getitem__(self,i):return NestedRefs(self.s+':'+str(i))
 def __str__(self):return self.s
ns7={x:NestedRefs('REF:'+x) for x in ['more2','more','rr3','rr4','rr5','rr6','results']}
# For any source expression the final ['url'] must resolve to a string.
class URLRefs:
 def __init__(self,s):self.s=s
 def __getitem__(self,i):
  return self.s if i=='url' else URLRefs(self.s+':'+str(i))
ns7={x:URLRefs('REF:'+x) for x in ['more2','more','rr3','rr4','rr5','rr6','results']}
exec(allcells[7][44].split('with (out/')[0],ns7)
print(pd.DataFrame(ns7['rows'])[['key','label','source_url']].to_string(index=False))
text7='\n'.join(m.get('content','') for m in allmsgs[7] if m.get('role')=='tool' and isinstance(m.get('content'),str))
Path('audit_work/group7_tool_evidence.txt').write_text(text7)
# Print only funding-source URL candidates, rather than navigation links.
for u in dict.fromkeys(re.findall(r'https?://[^\s\]\)\}\>\"\'|,]+',text7)):
 if any(t in u.lower() for t in ['airbyte','allegro','clearml','seematics','imply','jfrog','meltano']) and any(t in u.lower() for t in ['raises','funding','secures','series','million','news-release','press','ipo','seed','finder.startup','10-q','s-1']):print(u)

    key                                                                     label                                                                                                                                             source_url
airbyte                                                                  Pre-seed                                                                                                                                          REF:more2:1:3
airbyte         Y Combinator Winter 2020; individual investment terms undisclosed                                                                                                                                          REF:more2:1:0
airbyte                                                                      Seed                                                                                                                                          REF:more2:1:3
airbyte                               Seed; announced March 2021, cl

In [84]:
urls7=list(dict.fromkeys(re.findall(r'https?://[^\s\]\)\}\>\"\'|,]+',text7)))
for u in urls7:
 if '#' not in u and any(t in u.lower() for t in ['airbyte','allegro','clearml','seematics','imply','jfrog','meltano']) and any(t in u.lower() for t in ['raises','funding','secures','series','million','finder.startup','ipo','seed']) and not any(t in u for t in ['c212','cts.businesswire','?','/de/','/fr/','gitlab.com/-']):print(u)

https://imply.io/blog/series-a/
https://imply.io/blog/imply-raises-series-b-funding/
https://tracxn.com/d/companies/imply/__RCt8MOu3ddtioSlVeq2aXIVtsEFol9RnNyZOg973mfQ/funding-and-investors
https://about.gitlab.com/press/releases/2021-06-30-meltano-spins-out-of-gitlab-raises-seed-funding-led-by-gv/
https://techcrunch.com/2022/06/08/gitlab-spinoff-meltano-raises-another-8-2m-for-its-open-source-dataops-platform/
https://www.prweb.com/releases/meltano-releases-version-2-0-of-its-open-source-dataops-platform-infrastructure-raises-additional-capital-extending-seed-round-to-12-4-million-896671998.html
https://tracxn.com/d/companies/meltano/__LIso2VhvbDH3jfg3Ky8vLNEAb-c5jZJ0kRcG01zSpP8/funding-and-investors
https://airbyte.com/blog/how-airbyte-raised-its-series-a-round-2-months-after-its-seed
https://www.businesswire.com/news/home/20211217005648/en/Airbyte-Closes-%24150-Million-Series-B-Funding-Round-Led-by-Altimeter-Capital-and-Coatue-Management
https://sapphireventures.com/blog/congratulat

In [85]:
for u in urls7:
 if any(t in u.lower() for t in ['allegro','clearml','j-frog','jfrog/raises','jfrog-raises','jfrog-announces','jfrog-3','seed','airbyte']) and '#' not in u and not any(t in u for t in ['?','c212','cts.businesswire']):
  if any(x in u for x in ['allegro','clearml','jfrog','pre-seed','ycombinator','cbinsights']):print(u)
print('group7 notes objects:',[(x['key'],x['incomplete']) for x in recover_note_objects(allcells[7][46])])

https://globalventuring.com/blog/2018/04/27/allegro-ai-accepts-11m/
https://techcrunch.com/2018/04/25/allegro-ai-nabs-11m-for-a-platform-that-helps-businesses-build-computer-vision-based-services/
https://app.dealroom.co/companies/clearml_1
https://www.cbinsights.com/company/allegroai/financials
https://startupintros.com/orgs/allegroai
https://www.startuphub.ai/startups/allegro-ai
https://venturebeat.com/business/devops-platform-jfrog-raises-165-million-with-valuation-way-north-of-1-billion
https://www.finsmes.com/2018/10/jfrog-raises-165m-in-series-d-funding.html
https://www.cbinsights.com/company/airbyte/financials
https://en.globes.co.il/en/article-israeli-deep-learning-co-allegroai-raises-7m-1001233156
https://www.hyundai.com/content/hyundai/ww/data/news/data/2018/0000016062/attach/Hyundai%20Invests%20in%20DL%20computer%20vision%20allegro%20ai.pdf
https://twitter.com/clearmlapp
https://www.prnewswire.com/news-releases/leading-open-source-machine-learning-ops-tool-suite-allegro-trai

In [86]:
# Resolve remaining indexed references to source URLs actually present in the agent transcript.
resolved7={
 'REF:more2:1:3':'https://airbyte.com/blog/how-we-closed-our-pre-seed-round-in-2-weeks',
 'REF:more2:1:0':'https://www.ycombinator.com/companies/airbyte',
 'REF:more2:0:1':'https://airbyte.com/blog/airbyte-raised-a-26m-series-a-to-change-how-data-is-moved',
 'REF:results:0:1':'https://www.businesswire.com/news/home/20211217005648/en/Airbyte-Closes-%24150-Million-Series-B-Funding-Round-Led-by-Altimeter-Capital-and-Coatue-Management',
 'REF:more2:2:0':'https://www.cbinsights.com/company/airbyte/financials',
 'REF:rr3:3:4':'https://app.dealroom.co/companies/allegro_1',
 'REF:more:2:0':'https://en.globes.co.il/en/article-israeli-deep-learning-co-allegroai-raises-7m-1001233156',
 'REF:rr5:0:0':'https://en.globes.co.il/en/article-hyundai-invests-in-israeli-deep-learning-co-allegroai-1001259284',
 'REF:more2:3:3':'https://finder.startupnationcentral.org/company_page/seematics-systems',
 'REF:results:1:3':'https://www.cbinsights.com/company/allegroai/financials',
 'REF:more:4:0':'https://www.finsmes.com/2015/10/imply-launches-with-2m-in-seed-funding.html',
 'REF:results:2:0':'https://imply.io/blog/series-a/',
 'REF:results:2:1':'https://imply.io/blog/imply-raises-series-b-funding/',
 'REF:results:2:6':'https://imply.io/in-the-news/imply-closes-70-million-series-c-at-700m-valuation-to-extend-leadership-as-foundational-platform-for-analytics-in-motion/',
 'REF:results:2:3':'https://imply.io/blog/imply-raises-100mm-in-series-d-funding/',
 'REF:rr4:5:1':'https://www.startupranking.com/startup/jfrog/funding-rounds',
 'REF:more:5:0':'https://www.geektime.co.il/jfrog-series-a-3-5-million/',
 'REF:rr6:2:0':'https://techcrunch.com/2016/01/20/jfrog-takes-big-leap-forward-with-50-million-round/',
 'REF:results:3:5':'https://www.prnewswire.com/news-releases/jfrog-secures-165-million-investment-to-lead-universal-devops-in-the-enterprise-300724416.html',
 'REF:rr6:1:3':'https://www.globenewswire.com/news-release/2020/09/18/2096064/0/en/JFrog-Announces-Closing-of-Its-Initial-Public-Offering-and-Exercise-in-Full-of-the-Underwriters-Option-to-Purchase-Additional-Shares.html',
 'REF:results:4:0':'https://about.gitlab.com/press/releases/2021-06-30-meltano-spins-out-of-gitlab-raises-seed-funding-led-by-gv/',
 'REF:results:4:3':'https://www.prweb.com/releases/meltano-releases-version-2-0-of-its-open-source-dataops-platform-infrastructure-raises-additional-capital-extending-seed-round-to-12-4-million-896671998.html'
}
for row in ns7['rows']:
 row['source_url']=resolved7.get(row['source_url'],row['source_url'])
 assert row['source_url'].startswith('http'),row
pd.DataFrame(ns7['rows']).to_csv('audit_work/group7.csv',index=False)
notes7=recover_note_objects(allcells[7][46]);Path('audit_work/group7_notes.json').write_text(json.dumps(notes7,indent=2))
print('Group7 recovered',len(ns7['rows']),len(notes7))

Group7 recovered 28 5


In [87]:
s6=await check_task(task_ids[5]);print(s6.status,s6.runtime_minutes,s6.answer[:4000])
if s6.status=='success':
 messages=await get_task_messages(task_ids[5]);allmsgs[6]=messages;allcells[6]=extract_cells(messages)
 Path('audit_work/group6_messages.json').write_text(json.dumps(messages,indent=2))
 for i,c in enumerate(allcells[6]):
  if ('rows' in c and ('append' in c or 'rows=' in c or 'rows =' in c)) or 'notes=' in c: print(i,len(c),c[:240])
print('Local rows recovered',sum(len(pd.read_csv(p)) for p in Path('audit_work').glob('group[1-8].csv')))

Task 78387c12-5338-4c51-b039-365853670889 still running (status: in progress). Call check_task(task_id) again later.
in progress 15.021763616666668 
Local rows recovered 147


In [88]:
partial=pd.concat([pd.read_csv(p,dtype=str).fillna('') for p in sorted(Path('audit_work').glob('group[1-8].csv'))],ignore_index=True)
urlset=list(dict.fromkeys(u for v in partial['source_url'] for u in v.split(' | ')))
source_cache={v['url']:v for v in own_evidence}
source_cache.update({v['url']:v for v in supp_sources.values()})
async def get_audit_source(url):
 if url in source_cache and (source_cache[url].get('full_text') or source_cache[url].get('text') or source_cache[url].get('fetched_text')):return
 async with sem:
  try:
   text=await fetch_parsed_page(url)
   source_cache[url]={'url':url,'text':text,'title':text.splitlines()[0] if text else url}
  except Exception as e:source_cache[url]={'url':url,'error':str(e)}
await asyncio.gather(*(get_audit_source(u) for u in urlset))
Path('audit_work/source_cache.json').write_text(json.dumps(source_cache,indent=2))
print('Checked source URLs',len(urlset),'error count',sum('error' in source_cache[u] for u in urlset))
for u in urlset:
 if 'error' in source_cache[u]:print('ERROR',u,source_cache[u]['error'][:80])

Checked source URLs 146 error count 0


In [89]:
s6=await check_task(task_ids[5]);print(s6.status,s6.runtime_minutes,s6.answer[:4000])
if s6.status=='success':
 messages=await get_task_messages(task_ids[5]);allmsgs[6]=messages;allcells[6]=extract_cells(messages)
 Path('audit_work/group6_messages.json').write_text(json.dumps(messages,indent=2))
 for i,c in enumerate(allcells[6]):
  if ('rows' in c and ('append' in c or 'rows=' in c or 'rows =' in c)) or 'notes=' in c: print(i,len(c),c[:220])
# Check page support for ambiguous lead designations.
def src(u):
 d=source_cache[u];return d.get('text') or d.get('fetched_text') or d.get('full_text','')
u='https://techcrunch.com/2016/01/20/jfrog-takes-big-leap-forward-with-50-million-round/'
t=src(u)
m=t.lower().find('led by');print('\nJFROG series C:\n',t[max(0,m-450):m+1000])

success 15.378129566666667 ## Files Ready for Analysis

| File | Description | Format | Size |
|---|---|---|---:|
| `/workspace/audit_work/group6.csv` | Funding audit: 65 event rows across five ventures | CSV | 13.6 KB |
| `/workspace/audit_work/group6_notes.json` | Per-venture searches, uncertainties, and completeness flags | JSON | 45.7 KB |
| `/workspace/audit_work/group6_evidence.json` | Persisted search, fetch, filing, and API evidence | JSON | 4.72 MB |
| `/workspace/audit_work/group6_sources.bib` | Full BibTeX records saved using `save_citation` | BibTeX | 157.5 KB |
| `/workspace/audit_work/group6_manifest.json` | Complete artifact inventory with source URLs, parameters, transformations, and checksums | JSON | 42.6 KB |
| `/workspace/audit_work/group6_retrieve.py` | Supporting-data retrieval replay script | Python | 1.5 KB |

**Incomplete histories: 3 — Efabless, FarmBot, and Open Bionics.** All five ventures were researched; unresolved issues are explicit in the notes. Support

60 8296 # Construct the audit. Provisional type vocabulary is explicit because both requested rule files were absent.
cols=['key','venture','type','label','date','amount_usd','lead','investors','source_url','confidence']
rows=[]
62 12250 # Record the validation findings and interpretive decisions explicitly.
# Use the regulator/government record for co-lead attribution; TechCrunch provides USD amount independently.
# Keep the event source plus the indepe
63 2434 # Preserve a full BibTeX record for every checked web source, including sources used to reject a lead.
# The saved citations distinguish unknown publication dates from access dates.
from urllib.parse import urlparse
impo
65 5935 # Save a concise replay script for the authoritative downloadable financial evidence.
script='''"""Replay public evidence downloads for group6.
Run from a notebook with: await replay()
Or from a terminal: python /workspa

JFROG series C:
 JFrog Takes Big Leap Forward With $50 Million Round | TechCrunch


In [90]:
print(allcells[6][60])
print('CHANGES\n',allcells[6][62][:3200])

# Construct the audit. Provisional type vocabulary is explicit because both requested rule files were absent.
cols=['key','venture','type','label','date','amount_usd','lead','investors','source_url','confidence']
rows=[]
names={'efabless':'Efabless','evolutionaryscale':'EvolutionaryScale','farmbot':'FarmBot','open-bionics':'Open Bionics','researchhub':'ResearchHub (ResearchHub Technologies)'}
def add(k,typ,label,date='',amt='',lead='',inv='',src='',conf='high'):
 rows.append(dict(zip(cols,[k,names[k],typ,label,date,str(amt),lead,inv,urls0.get(src,src),conf])))
for dt,amount in [('2013-01',100000),('2014-03',300000),('2015-08',500000)]:
 add('efabless','convertible_note','Convertible note (database-reported)',dt,amount,src='efabless_dealroom',conf='low')
add('efabless','equity','Series A (conflicting database amounts)','2020-07',5000000,src='efabless_dealroom',conf='low')
add('efabless','equity','Series A-1 initial funding announcement','2022-01',lead='New North Ventures',src='efabless_

In [91]:
print(allcells[6][60][4500:7600])
for i,c in enumerate(allcells[6][:60]):
 if any(re.search(r'^'+name+r'\s*=',c,re.M) for name in ['urls0','loan_records','manifest_url','filing_docs','extra_urls','awards']): print('\nCELL',i,'\n',c[:11000])

up cash prize','2014-11',200000,inv='Intel',src='ob_intel_final')
add('open-bionics','convertible_note','Intel Capital convertible note (database-reported)','2015-01',100000,inv='Intel Capital',src='ob_dealroom',conf='low')
add('open-bionics','accelerator','Disney Accelerator powered by Techstars; combined package','2015-07',120000,inv='Disney Accelerator | Techstars',src='ob_seedtable',conf='medium')
add('open-bionics','grant','UK James Dyson Award cash prize','2015-08',inv='James Dyson Foundation',src='ob_dyson')
add('open-bionics','grant','Nominet Trust Social Tech Seed grant','2016-09',inv='Nominet Trust',src='ob_nominet')
add('open-bionics','grant','SBRI Healthcare feasibility award','2017-01',inv='SBRI Healthcare',src='ob_sbri_1')
add('open-bionics','grant','UAE AI and Robotics Award for Good cash prize','2017-02',1000000,inv='UAE AI and Robotics Award for Good',src='ob_uae_company')
add('open-bionics','equity','Seed investment by Downing Ventures','2017-03',inv='Downing Ventures

In [92]:
text6='\n'.join(m.get('content','') for m in allmsgs[6] if m.get('role')=='tool' and isinstance(m.get('content'),str))
Path('audit_work/group6_tool_evidence.txt').write_text(text6)
print('Chars',len(text6),'headers',text6.count('key,venture,type'))
for i,c in enumerate(allcells[6]):
 if any(x in c for x in ['urls0.update','ukri_awards=','ukri_awards =','manifest_url','filing_docs=']):print(i,c[:1900])
print('ALL SOURCE LEDGER printed?')
for m in allmsgs[6]:
 txt=m.get('content')
 if m.get('role')=='tool' and isinstance(txt,str) and 'efabless_ext' in txt and 'https' in txt:print(txt[:3500])

Chars 529679 headers 0
9 more_urls={
'efabless_2022':r2[0][1]['url'],
'es_tc':results[1][3]['url'],
'farm_2014':r2[2][0]['url'],'farm_shuttle':r2[2][3]['url'],'farm_shuttle_review':r2[2][4]['url'],
'ob_dealroom':results[3][0]['url'],'ob_rounds':r2[3][0]['url'],'ob_loan':r2[6][-2]['url'],
'rh_seed':r2[4][0]['url'],'rh_block':r2[4][2]['url'],'rh_2025':r2[5][1]['url'],'rh_root':'https://www.rootdata.com/projects/detail/ResearchHub?k=ODIzNg%3D%3D'}
texts=await asyncio.gather(*(fetch(u) for u in more_urls.values()))
pages.update(dict(zip(more_urls,texts))); urls0.update(more_urls)
for k in ['ob_dealroom','ob_rounds','rh_seed','rh_block','rh_2025']:
 print('\n###',k,'\n',pages[k][:12500])
15 new_urls={
'ob_intel_final':r3[0][1]['url'],'ob_intel_initial':r3[0][4]['url'],
'ob_disney_tc':r3[1][2]['url'],'ob_disney_variety':r3[1][1]['url'],
'ob_crowd_company':r3[3][1]['url'],'ob_crowd':r3[3][2]['url'],
'rh_sora':r3[6][0]['url'],
'rh_company':'https://www.researchhub.com/post/979/researchhub-rais

In [93]:
print(allcells[6][61])
for i in [43,44,45,48,49,50,51,52,53,54,55,56,57,58,59]:
 c=allcells[6][i]
 if any(x in c for x in ['ukri_awards','urls0[','manifest_url','loan','filing_docs']): print('\nCELL',i,c[:6000])

for k in ['ob_tc','es_reuters','rh_sec','rh_boost_company']:
 print('\n',k)
 for line in pages[k].splitlines():
  if any(w in line.lower() for w in ['led','million','first sale','equity','boost','january 14','january 10']):print(line[:1400])
print('ukri api awards',[(a['projectTitle'],a['startDate'],a['totalGrantValue']) for a in ukri_awards])
print('check supporting source label',urls0['ob_loan'])
print('rules absence',Path('/workspace/deal_rules.md').exists(),Path('/workspace/funders_audit_sample.csv').exists())

CELL 43 for i in [2,3,4,5,6,7]:
 print('\n',i)
 for r in r9[i][:2]:print(r['url'],r.get('full_text','')[:2300])
print('farm capitalization complete already; timeline fetched next')
more_urls2={'farm_timeline':re.findall(r'\]\(([^)]*finances/timeline[^)]*)',pages['farm_cap'])[0],
'ob_nominet':r9[1][0]['url'],
'ob_nominet_uk':r9[1][1]['url'],
'ob_grant2018':r8[4][0]['url']}
texts=await asyncio.gather(*(fetch(u) for u in more_urls2.values()));pages.update(dict(zip(more_urls2,te

In [94]:
import httpx,csv,io
from urllib.parse import urljoin
farm_manifest_url='https://meta.farm.bot/assets/data/timeline/accounts/manifest.csv'
# Use the exact retrieved URL from the agent execution transcript rather than assuming path prefix.
ma=re.search(r'https://meta\.farm\.bot/[^\s\]\)\"\']*manifest\.csv',text6)
print('Discovered manifest',ma.group(0) if ma else None)
if ma:farm_manifest_url=ma.group(0)
async with httpx.AsyncClient(timeout=60,follow_redirects=True) as client:
 resp=await client.get(farm_manifest_url)
print(resp.status_code,resp.text[:5000])
local_loan_records=list(csv.DictReader(io.StringIO(resp.text))) if resp.status_code==200 else []

Discovered manifest https://meta.farm.bot/assets/data/timeline/accounts/manifest.csv
200 account,file
Cash,/assets/data/timeline/accounts/cash.csv
Credit Cards,/assets/data/timeline/accounts/credit_cards/amex_credit_card_2001.csv
Credit Cards,/assets/data/timeline/accounts/credit_cards/amex_credit_card_2006.csv
Credit Cards,/assets/data/timeline/accounts/credit_cards/wells_fargo_credit_card.csv
Line of Credit,/assets/data/timeline/accounts/line_of_credit.csv
Shopify Loan,/assets/data/timeline/accounts/shopify_capital_loans/shopify_capital_loan_2018-11-21.csv
Shopify Loan,/assets/data/timeline/accounts/shopify_capital_loans/shopify_capital_loan_2020-01-03.csv
Shopify Loan,/assets/data/timeline/accounts/shopify_capital_loans/shopify_capital_loan_2020-07-30.csv
Shopify Loan,/assets/data/timeline/accounts/shopify_capital_loans/shopify_capital_loan_2022-08-26.csv
Shopify Loan,/assets/data/timeline/accounts/shopify_capital_loans/shopify_capital_loan_2023-03-15.csv
Shopify Loan,/assets/data/t

In [95]:
loan_records=[x for x in local_loan_records if 'Loan' in x['account'] or x['account']=='Line of Credit']
async def rawget(u):
 async with sem:
  async with httpx.AsyncClient(timeout=60,follow_redirects=True) as cl:
   r=await cl.get(u);r.raise_for_status();return r.text
loan_texts=await asyncio.gather(*(rawget(urljoin(farm_manifest_url,x['file'])) for x in loan_records))
raw_farm_sources=[{'url':urljoin(farm_manifest_url,x['file']),'text':t,'title':f"FarmBot financial ledger: {x['account']} ({Path(x['file']).stem})"} for x,t in zip(loan_records,loan_texts)]
for d in raw_farm_sources:source_cache[d['url']]=d
Path('audit_work/farmbot_raw_ledgers.json').write_text(json.dumps(raw_farm_sources,indent=2))
# Get direct Open Bionics grants through current public API (company-specific contribution, not consortium total).
apiurl='https://gtr.ukri.org/api/v3/awards'
async with httpx.AsyncClient(timeout=60,follow_redirects=True) as cl:
 results_api=await asyncio.gather(*(cl.get(apiurl,params={'query':q}) for q in ['Hero 2 Product Range','Novel Partial Hand Prosthesis','Digital clinical pathway for upper limb prosthetics','End-to-end digital service for upper limb amputees']))
 ukri_awards=[]
 for rr0 in results_api:
  data=rr0.json()
  for z in data.get('awardList',[]):
   a=z['awardListItem']
   if a.get('leadOrgName')=='OPEN BIONICS LTD' and a not in ukri_awards:ukri_awards.append(a)
 details=await asyncio.gather(*(cl.get(a['href']) for a in ukri_awards))
ukri_details={a['href']:r.json() for a,r in zip(ukri_awards,details)}
Path('audit_work/openbionics_awards.json').write_text(json.dumps(ukri_details,indent=2))
print([(a['projectTitle'],a['startDate'],a['href']) for a in ukri_awards])
for a in ukri_awards:
 d=ukri_details[a['href']]
 print(a['projectTitle'],[(p['organisationName'],p.get('grantOffer')) for p in d.get('participantValues',[])])

[('Hero 2 Product Range', '2024-04-01T00:00:00Z', 'https://gtr.ukri.org/api/v3/awards/ae48f672-a774-43e7-a958-4c8c1e727fb7'), ('Novel Partial Hand Prosthesis for people with disability', '2023-04-01T00:00:00Z', 'https://gtr.ukri.org/api/v3/awards/404463bf-b6e2-4cd1-a1fa-1b9b396b7ff5'), ('Digital clinical pathway for upper limb prosthetics and rehabilitation', '2020-10-01T00:00:00Z', 'https://gtr.ukri.org/api/v3/awards/9c2540e5-e001-4808-be81-1a927e2e06de'), ('End-to-end digital service for upper limb amputees and clinicians', '2020-06-01T00:00:00Z', 'https://gtr.ukri.org/api/v3/awards/81eb9a62-1ce2-4e73-8ad4-1b506f21eec9')]
Hero 2 Product Range [('OPEN BIONICS LTD', 229460.27)]
Novel Partial Hand Prosthesis for people with disability [('OPEN BIONICS LTD', 50000)]
Digital clinical pathway for upper limb prosthetics and rehabilitation [('OPEN BIONICS LTD', 174717.6)]
End-to-end digital service for upper limb amputees and clinicians [('OPEN BIONICS LTD', 74992)]


In [96]:
# Parse the Companies House filing links to reproduce the reported cash-allotment source URLs.
filings_url='https://find-and-update.company-information.service.gov.uk/company/09001817/filing-history'
filings_page=await fetch_parsed_page(filings_url)
filing_docs=[(m.group(1),m.group(2)) for m in re.finditer(r'\[([^\]]*?)\]\((https://[^)]*document[^)]*)\)',filings_page) if 'allotment' in m.group(1)]
print('filing docs',len(filing_docs),filing_docs[:12])
urls6=list(dict.fromkeys(re.findall(r'https?://[^\s\]\)\}\>\"\'|,]+',text6)))
for u in urls6:
 if '#' not in u and any(t in u.lower() for t in ['efabless','evolutionaryscale','open-bionics','openbionics','researchhub']) and not any(t in u for t in ['c212','cts.businesswire','/cdn','/image','/wp-content/uploads/','/js/','/static/','/help','/careers','/cookie','/privacy','/terms','/press-kit','linkedin.com/in/','/catalog','/events/','/category','/tags/','/about','/menu','/products']):
  if len(u)<230:print(u)

filing docs 12 [('View PDF **Statement of capital following an allotment of shares** on 12 June 2026 * GBP 3,464.246 \\- link opens in a new window - 3 pages', 'https://find-and-update.company-information.service.gov.uk/company/09001817/filing-history/MzU1NjcwMTgyNWFkaXF6a2N4/document?format=pdf&download=0'), ('View PDF **Statement of capital following an allotment of shares** on 20 December 2023 * GBP 3,461.053 \\- link opens in a new window - 3 pages', 'https://find-and-update.company-information.service.gov.uk/company/09001817/filing-history/MzUyNjQ2NDI2NmFkaXF6a2N4/document?format=pdf&download=0'), ('View PDF **Statement of capital following an allotment of shares** on 20 December 2023 * GBP 3,461.053 \\- link opens in a new window - 3 pages', 'https://find-and-update.company-information.service.gov.uk/company/09001817/filing-history/MzUyNjQ2NDQ4MGFkaXF6a2N4/document?format=pdf&download=0'), ('View PDF **Statement of capital following an allotment of shares** on 23 July 2025 * GBP 

In [97]:
for term in ['efabless_sec','ob_crowd_2023','ob_aventure','ob_dyson','ob_nominet','ob_openhand','ob_intel_final','ob_seedtable','ob_bif_pdf']:
 print('\n',term)
 for i,c in enumerate(allcells[6]):
  if term in c and i<60:
   print(i,c[:400] if term not in ['efabless_sec','ob_crowd_2023','ob_aventure'] else c[:2000])
print('\nURL filtered')
for u in urls6:
 if any(t in u for t in ['crowdcube.com/companies/open','aventure','nominet','dyson','indiegogo','rewired','nhs-funding','intel-make','seedtable.com/companies/open','business-innovation','2039822','boost','activity:729802','activity-729802']):
  if '#' not in u:print(u)


 efabless_sec
34 next_urls={
'efabless_sec_index':r7[5][0]['url'],
'efabless_ms':r7[4][3]['url'],
'efabless_pitchbook':r7[4][1]['url'],
'ob_seedtable':r7[6][2]['url'],
'ob_women_result':r7[7][2]['url'],
'ob_dyson':r7[8][0]['url'],
'ob_openhand':r7[9][0]['url'],
'ob_sbri_2016':r6[3][1]['url']}
texts=await asyncio.gather(*(fetch(u) for u in next_urls.values()));pages.update(dict(zip(next_urls,texts)));urls0.update(next_urls)
for k in ['efabless_sec_index','efabless_ms','efabless_pitchbook','ob_dyson','ob_openhand','ob_sbri_2016']:
 print('\n###',k)
 t=pages[k]
 if k=='efabless_sec_index':print(t[:8000])
 else:
  for line in t.splitlines():
   if any(w in line.lower() for w in ['million','fund','invest','£','$','2013','2016','2017','2020','2025','2021','2022','2023']):print(line[:1500])
38 for i in [2,3,4,5,6,7,8]:
 print('\n',i,moreq[i])
 for r in r8[i][:3]:print(r['title'],r['url'],'\n',r.get('full_text','')[:1450])
# SEC document URL discovered in fetched filing index
sec_url=re.searc

In [98]:
# Reconstruct group6's static events with symbolic URL keys; generated debt/grant records use live primary data.
from collections import defaultdict
urlkeys6=defaultdict(str)
for c in allcells[6]:
 for key in re.findall(r"(?:src=|urls0\[)['\"]([^'\"]+)['\"]",c):urlkeys6[key]='REF:'+key
class KeyDict(dict):
 def get(self,k,default=None):return super().get(k,k if default is None else default)
urlkeys6=KeyDict(urlkeys6)
ns6={'urls0':urlkeys6,'manifest_url':farm_manifest_url,'loan_records':loan_records,'loan_texts':loan_texts,'ukri_awards':ukri_awards,'filing_docs':filing_docs,'extra_urls':{'ob_bif_pdf':'REF:ob_bif_pdf'},'datetime':datetime,'urljoin':urljoin,'csv':csv,'io':io,'re':re}
exec(allcells[6][60].split("print('Row count'")[0],ns6)
print('Rows',len(ns6['rows']))
for row in ns6['rows']:
 if row['source_url'].startswith('REF:'): print(row['key'],row['label'][:65],row['source_url'])

Efabless later close publication 2022-05-10 04:08:06.651000+00:00
Rows 65
efabless Convertible note (database-reported) REF:efabless_dealroom
efabless Convertible note (database-reported) REF:efabless_dealroom
efabless Convertible note (database-reported) REF:efabless_dealroom
efabless Series A (conflicting database amounts) REF:efabless_dealroom
efabless Series A-1 initial funding announcement REF:efabless_2022
efabless Series A-1 closing; possible continuation of January financing REF:efabless_synopsys
efabless Series A-1 extension REF:efabless_ext
efabless Private debt offering; first sale September 2024 REF:efabless_sec
evolutionaryscale Seed financing reported by Forbes (at least $40 million) REF:es_forbes_2023
evolutionaryscale Seed financing (more than $142 million announced) REF:es_aws
farmbot Shuttleworth fellowship supporting FarmBot project REF:farm_cap
farmbot FarmBot Genesis rewards/preorder crowdfunding REF:farm_launch
open-bionics Open Hand Project precursor rewards crow

In [99]:
# Narrow source candidates needed to resolve the remaining group6 symbolic references.
for u in urls6:
 if '#' not in u and len(u)<260 and (any(t in u for t in ['intel','dyson','open-hand-project','openhand','bif','innovation-fund','seedtable.com/companies/open-bionics','pitchbook.com/profiles/company/','techcrunch.com/2019','globenewswire.com/news-release/2022/01','2039822']) or ('indiegogo' in u and '/projects/' in u)):
  print(u)
print('\nBIF context')
for mm in list(re.finditer('ob_bif_pdf',text6))[:2]:print(text6[max(0,mm.start()-400):mm.start()+500])

https://pitchbook.com/profiles/company/83804-86
https://techcrunch.com/2019/01/14/open-bionics-closes-5-9m-series-a-for-its-affordable-and-cool-bionic-limbs/
https://pitchbook.com/profiles/company/457169-05
https://www.aboutamazon.com/artificial-intelligence-ai-news
https://seedtable.com/companies/open-bionics/funding-rounds
https://app.dealroom.co/companies/intel
https://app.dealroom.co/investors/intel_capital
https://techspark.co/blog/2014/11/05/joel-gibbard-of-robotics-firm-open-bionics-wins-125k-intel-prize/
https://3dprintingindustry.com/news/open-bionics-intel-make-wearable-3d-printed-prosthetic-34165/
https://dataintelo.com/report/global-prosthetic-robot-arm-market
https://synbiointel.com/companies/evolutionaryscale/
https://synbiointel.com/funding/
https://synbiointel.com/synbio-venture-capital/
https://pitchbook.com/profiles/company/119534-95
https://pitchbook.com/profiles/company/534690-28
https://motivationalspeakersagency.co.uk/artificial-intelligence-speakers/samantha-payn

In [100]:
qsresolve=[
 'Efabless Series A-1 January 12 2022 New North Ventures globenewswire',
 'Open Bionics West of England Business Innovation Fund 20000 March 2022 pdf',
 'Open Bionics Disney Techstars 120000 2015 funding',
 'Open Hand Project crowdfunding 2013 43000 Indiegogo',
]
resresolve=await asyncio.gather(*(web_search(q,num_results=4) for q in qsresolve))
for q1,rs in zip(qsresolve,resresolve):
 print('\n',q1)
 for x in rs:print(x['title'],x['url'],x.get('snippet','')[:650])


 Efabless Series A-1 January 12 2022 New North Ventures globenewswire
Efabless Series A-1 Round Led by New North Ventures Drives https://www.globenewswire.com/fr/news-release/2022/01/12/2365804/0/en/efabless-series-a-1-round-led-by-new-north-ventures-drives-open-source-enabled-chip-design.html # Efabless Series A-1 Round Led by New North Ventures Drives Open-Source-Enabled Chip Design

 12 janv. 2022 10h00 HE | Source: Efabless Corporation 

SAN JOSE, Calif., Jan. 12, 2022 (GLOBE NEWSWIRE) -- Efabless, the creator community platform for the design, development and commercialization of integrated circuits (“ICs” or “chips”), announced it has received series A-1 venture funding led by New North Ventures to capitalize on its successful chipIgnite and Google-sponsored Open-MPW programs. Efabless will use proceeds to support business development and marketing, and to further product development. 

Jeremy Hitchcock, partner at New North 
Efabless Series A-1 funding round drives open-source-

In [101]:
resolved6={
'efabless_dealroom':'https://app.dealroom.co/companies/efabless_corporation',
'efabless_2022':resresolve[0][0]['url'],
'efabless_synopsys':'https://www.linkedin.com/posts/efabless_chips-simplify-chip-activity-6929643205143465984-Dd1N',
'efabless_ext':'https://www.globenewswire.com/news-release/2023/10/10/2757684/0/en/Efabless-Raises-6-3-Million-in-Series-A-1-Extension-Featuring-Investments-from-GlobalFoundries-Synopsys-and-New-North-Ventures.html',
'efabless_sec':'https://www.sec.gov/Archives/edgar/data/2039822/000203982224000001/xslFormDX01/primary_doc.xml',
'es_forbes_2023':'https://www.forbes.com/sites/kenrickcai/2023/08/25/evolutionaryscale-ai-biotech-startup-meta-researchers-funding/',
'es_aws':'https://press.aboutamazon.com/aws/2024/6/evolutionaryscale-launches-with-esm3-a-milestone-ai-model-for-biology',
'farm_cap':'https://meta.farm.bot/v2026/finances/capitalization.html',
'farm_launch':supp_source_map['farmbot_campaign'],
'ob_openhand':resresolve[3][-1]['url'],
'ob_dealroom':'https://app.dealroom.co/companies/open_bionics',
'ob_intel_final':'https://techspark.co/blog/2014/11/05/joel-gibbard-of-robotics-firm-open-bionics-wins-125k-intel-prize/',
'ob_seedtable':'https://seedtable.com/companies/open-bionics/funding-rounds',
'ob_dyson':'https://www.siliconrepublic.com/machines/james-dyson-award-open-bionics',
'ob_nominet':'https://startupbeat.com/nominet-trust-unlocks-400000/23676/',
'ob_sbri_1':'https://www.bristol247.com/business/news-business/100k-nhs-funding-for-open-bionics/',
'ob_uae_company':supp_source_map['openbionics_prize_company'],
'ob_rewired_launch':'https://rewiredstudio.medium.com/launching-rewired-a-robotics-focused-venture-studio-and-fund-49b73725161d',
'ob_pitchbook':'https://pitchbook.com/profiles/company/119534-95',
'ob_uk':supp_source_map['openbionics_sbri_gov'],
'ob_aventure':'https://aventure.vc/companies/open-bionics-college-green-bristol-gb/fundraising',
'ob_tc':'https://techcrunch.com/2019/01/14/open-bionics-closes-5-9m-series-a-for-its-affordable-and-cool-bionic-limbs/',
'ob_bif_pdf':resresolve[1][0]['url'],
'ob_crowd_2023':'https://www.crowdcube.com/companies/open-bionics-ltd/pitches/Z7pr4b?country=GB',
'ob_crowd':supp_source_map['openbionics_crowdcube2024'],
'ob_loan':'https://inngot.com/case-study/open-bionics-uk-prosthetics-firm-gets-funding-from-natwest-supported-by-inngot',
'rh_company':'https://www.researchhub.com/post/979/researchhub-raises-5m-to-help-scientists-monetize-their-research',
'rh_sora':supp_source_map['researchhub_sora'],
'rh_boost_company':'https://www.linkedin.com/feed/update/urn:li:activity:7298027491040907264'
}
for row in ns6['rows']:
 if row['source_url'].startswith('REF:'):
  row['source_url']=resolved6[row['source_url'][4:]]
pd.DataFrame(ns6['rows']).to_csv('audit_work/group6.csv',index=False)
notes6=recover_note_objects(allcells[6][62]);Path('audit_work/group6_notes.json').write_text(json.dumps(notes6,indent=2))
print('Recovered all',len(ns6['rows']),'group6 rows and',len(notes6),'notes.')
for n in range(1,9):print(n,pd.read_csv(f'audit_work/group{n}.csv').shape)

Recovered all 65 group6 rows and 5 notes.
1 (14, 10)
2 (8, 10)
3 (27, 10)
4 (9, 10)
5 (29, 10)
6 (65, 10)
7 (28, 10)
8 (32, 10)


In [102]:
raw=pd.concat([pd.read_csv(f'audit_work/group{n}.csv',dtype=str).fillna('') for n in range(1,9)],ignore_index=True)
allnotes={}
for n in range(1,9):
 for note in json.loads(Path(f'audit_work/group{n}_notes.json').read_text()):allnotes[note['key']]=note
print('RAW',len(raw),'events;',raw['key'].nunique(),'ventures; incomplete',sum(x['incomplete'] for x in allnotes.values()))
print('Unexpected types',raw.loc[~raw['type'].isin(allowed_types),'type'].value_counts())
print('Noninteger amounts',raw.loc[~raw['amount_usd'].str.fullmatch(r'\d*'),['key','label','amount_usd']].to_string(index=False))
print('Mismatched venture names',[(k,v,identities.get(k)) for k,v in raw[['key','venture']].itertuples(index=False,name=None) if identities.get(k)!=v])
# Fetch remaining group6 sources now, including raw APIs and regulatory documents.
newurls=list(dict.fromkeys(u for v in raw['source_url'] for u in v.split(' | ')))
await asyncio.gather(*(get_audit_source(u) for u in newurls))
Path('audit_work/source_cache.json').write_text(json.dumps(source_cache,indent=2))
print('Total URLs',len(newurls),'cache entries',len(source_cache))

RAW 212 events; 40 ventures; incomplete 20
Unexpected types type
series          31
grant           28
equity          19
venture         13
crowdfunding    11
ipo              2
Name: count, dtype: int64
Noninteger amounts     key                                                label amount_usd
farmbot Shopify Capital loan (published funding transaction)   63000.00
farmbot Shopify Capital loan (published funding transaction)   99000.00
farmbot Shopify Capital loan (published funding transaction)   80000.00
farmbot Shopify Capital loan (published funding transaction)   75000.00
farmbot Shopify Capital loan (published funding transaction)   40000.00
farmbot Shopify Capital loan (published funding transaction)   60000.00
farmbot Shopify Capital loan (published funding transaction)   50000.00
farmbot Shopify Capital loan (published funding transaction)   69000.00
farmbot Shopify Capital loan (published funding transaction)   39000.00
farmbot Shopify Capital loan (published funding transact

Total URLs 205 cache entries 234


In [103]:
checks=[('anaconda','Seed equity'),('airbyte','Pre-seed'),('jfrog','Series C'),('open-bionics','Series A'),('efabless','Series A-1 closing'),('evolutionaryscale','Seed financing reported'),('farmbot','Shuttleworth'),('getzep','DV seed'),('supermemoryai','Reported Y Combinator')]
for key,label in checks:
 matches=raw[(raw.key==key)&raw.label.str.startswith(label)]
 if matches.empty:continue
 rr0=matches.iloc[0];text=src(rr0.source_url.split(' | ')[0]);print('\nCHECK',key,label)
 if key=='anaconda':print(text[-4500:]);continue
 terms=['50 million','led by','40 million','125,000','got the','Zep','600k','600K','1.2m','1.2M','Series A-1','Seed']
 found=[]
 for term in terms:
  m=re.search(re.escape(term),text,re.I)
  if m:
   snippet=text[max(0,m.start()-130):m.start()+550].replace('\n',' ')
   if snippet not in found:found.append(snippet)
 print('\n'.join(found[:3]) or text[:500])


CHECK anaconda Seed equity
EDGAR Filing Documents for 0001538860-12-000001

* [![](https://www.sec.gov/edgar/search/images/edgar-logo-2x.png)](https://www.sec.gov)
* [SEC.gov](https://www.sec.gov)
* [EDGAR](https://www.sec.gov/submit-filings/about-edgar)
+ [Latest Filings](https://www.sec.gov/cgi-bin/browse-edgar?action=getcurrent)
+ [Filings search tools](https://www.sec.gov/edgar/search-and-access)

# Filing Detail

* [SEC Home](https://www.sec.gov/index.htm) »
* [Company Search](https://www.sec.gov/edgar/searchedgar/companysearch.html) »
* Current Page

**Form D** \- Notice of Exempt Offering of Securities

**SEC Accession No.** 0001538860-12-000001

Filing Date

2012-01-09

Accepted

2012-01-06 19:41:47

Documents

1

Effectiveness Date

2012-01-09

Items

Item 05: Rule 505

Document Format Files

| Seq | Description          | Document             | Type | Size |
| --- | -------------------- | -------------------- | ---- | ---- |
| 1   |                      | [primary\_doc.html]

In [104]:
for key,label in [('airbyte','Pre-seed'),('jfrog','Series C'),('anaconda','Seed equity')]:
 rr0=raw[(raw.key==key)&raw.label.str.startswith(label)].iloc[0];text=src(rr0.source_url)
 print(key,'len',len(text))
 for term in ['600','1.2','$50','Sapphire','Scale','January','May 2020']:
  for m in list(re.finditer(re.escape(term),text,re.I))[:3]: print(term,text[max(0,m.start()-100):m.start()+380].replace('\n',' '))
# SEC first seed actual amended source.
print('\nGroup5 transcript SEC seed context:')
for m in re.finditer('2,250,000|2250000|Total Amount Sold',tooltexts[5]):
 if '2250000' in m.group() or '2,250,000' in m.group():print(tooltexts[5][max(0,m.start()-500):m.start()+350])

airbyte len 11480
January  of December 2019. Originally, we wanted to raise a pre-seed round before starting YC on the 6th of January. But at the same time, we didn’t want to incorporate the company in 2019 so we wouldn’t have to deal with the taxes for 2019. Call us lazy! One would say lazy is not bad, as long as it’s smart lazy!  So we decided to prepare for a pre-seed round at the beginning of January and incorporate on 01/01/2020 at 12.01am.  ## Levers you can use for a fund-raising round  There a
January bad, as long as it’s smart lazy!  So we decided to prepare for a pre-seed round at the beginning of January and incorporate on 01/01/2020 at 12.01am.  ## Levers you can use for a fund-raising round  There are a few levers that you can activate to motivate investors to participate in your round, and fast. Here are a few of them (please feel free to add more in the comments, and we’ll edit the article!).  Think of fundraising as a sales funnel. Some levers will help you get mor
Janu

In [105]:
for c in allcells[7]:
 if re.search(r'^more2\s*=',c,re.M) or 'q2=' in c or 'queries2=' in c:print(c[:4500])
for term in ['600','1.2','aearly','How We Closed','closed our pre-seed']:
 print('\nTERM',term)
 for m in list(re.finditer(re.escape(term),text7,re.I))[:5]:print(text7[max(0,m.start()-260):m.start()+500].replace('\n',' '))
# Locate Anaconda amended filing context preceding $2.25m.
m=tooltexts[5].find('$2,250,000');print('\nAnaconda context source',tooltexts[5][max(0,m-14000):max(0,m-14000)+1700])

queries=[
'Airbyte raises 26 million May 2021 Benchmark investors seed',
'Airbyte Y Combinator 2020 125000 150000 funding accelerator',
'Airbyte 2024 accelerator October 4 funding investors',
'ClearML July 24 2024 investment Gandyr VentureIsrael funding',
'ClearML 2021 accelerator July 28 Allegro 2018 August 30',
'JFrog initial seed funding 2008 2009 chief scientist founders',
'JFrog 2018 165 million funding press release investors',
'JFrog IPO closing September 2020 405.3 million primary shares',
'JFrog 2025 2026 convertible senior notes offering 600 million debt'
]
more2=await asyncio.gather(*[search(q,6) for q in queries])

TERM 600
 to create data pipelines from sources such as PostgreSQL, MySQL, Facebook Ads, Salesforce, Stripe, and connect to destinations that include Redshift, Snowflake, and BigQuery.   The number of Airbyte users has grown from 250 companies at the end of January to 600 today. They include: Safegraph, Dribbble, Mercato, GraniteRock, Agridigital, Cart.com.   "Co

In [106]:
for k in ['antmicro','bela','bcn3d','bitcraze','open-bionics']:
 print('\n',k,allnotes[k]['note'][:14000])
print('\nAnaconda notes',allnotes['anaconda']['note'][:7000])


 antmicro Fresh searches covered equity/investors, EU/Horizon research grants, CORDIS, Polish/NCBR and 2024–2026 updates; five distinct funded projects were verified with official beneficiary-level allocations, including EXCEED and TRISTAN omitted from the three-grant database summary. USD amounts are blank because disclosed allocations are EUR136850 (AXIOM), EUR470750 (X-MINE), EUR645250 (VEDLIoT), EUR300000 (EXCEED; grant-agreement maximum subject to final confirmation) and EUR788593.75 (TRISTAN), not the much larger consortium totals. Dates are grant signature months: Dec 2014, May 2017, Jul 2020, Oct 2020 and Nov 2022; corresponding project starts were Mar 2015, Jun 2017, Nov 2020, Nov 2020 and Dec 2022. The European Commission is the funding body, not an equity investor; consortium coordinators/partners are not investors. No independently substantiated equity, debt, crowdfunding, token, IPO or secondary funding event was found in this scope. No currency conversion was performed; 

In [107]:
# Start a reversible reconciliation log; original parallel-pass tables remain untouched.
audit=raw.copy()
changes=[];note_additions=defaultdict(list)
def change(mask,reason,**updates):
 global audit
 ix=audit.index[mask]
 for i in ix:
  old=audit.loc[i].to_dict()
  for k,v in updates.items():audit.at[i,k]=str(v)
  changes.append({'action':'update','reason':reason,'before':old,'after':audit.loc[i].to_dict()})
def drop(mask,reason):
 global audit
 for i in audit.index[mask]: changes.append({'action':'exclude','reason':reason,'row':audit.loc[i].to_dict()})
 audit=audit.loc[~mask].copy()
def append_event(row,reason):
 global audit
 complete=dict.fromkeys(columns,'');complete.update(row);complete['venture']=identities[complete['key']]
 audit=pd.concat([audit,pd.DataFrame([complete])],ignore_index=True)
 changes.append({'action':'add','reason':reason,'row':complete})
# Normalize only the taxonomy; stage-specific labels are retained.
seriesmap={'A':'series_a','B':'series_b','C':'series_c','D':'series_d','E':'series_e','F':'series_f_plus','G':'series_f_plus'}
for i,r0 in audit.iterrows():
 typ=r0['type'];label=r0['label']
 if typ in ['series','equity','venture']:
  stage=re.search(r'Series\s+([A-G])(?:[- ]?1)?\b',label,re.I)
  if stage:typ=seriesmap[stage.group(1).upper()]
  elif re.search(r'pre[- ]seed',label,re.I):typ='pre_seed'
  elif re.search(r'\bseed\b',label,re.I):typ='seed'
  elif 'Strategic' in label or 'strategic' in label:typ='corporate_strategic'
  else:typ='venture_unspecified'
 elif typ=='grant':
  if r0['key']=='payloadcms' and 'Start Garden' in label:typ='prize'
  elif r0['key']=='open-bionics' and any(t in label for t in ['Intel Make','James Dyson','UAE AI']):typ='prize'
  elif r0['key'] in ['supermemoryai','farmbot'] or 'Nominet' in label:typ='grant_philanthropic'
  else:typ='grant_government'
 elif typ=='crowdfunding':
  if r0['key']=='open-bionics' and 'Crowdcube' in label:typ='crowdfunding_equity'
  elif r0['key']=='neuromore':typ='crowdfunding_equity'
  elif r0['key'] in ['BruceDevices','TecharoHQ']:typ='other'
  else:typ='crowdfunding_reward'
 elif typ=='ipo':typ='other'
 audit.at[i,'type']=typ
# Respect explicit pre-seed labeling even when an agent's provisional type was seed.
change(audit['label'].str.startswith(('Pre-seed','Pre-seed financing')),'Use rule-defined pre_seed for explicitly named pre-seed rounds.',type='pre_seed')
# Remove noncash participation, a cancelled unpaid grant, and explicitly uncertain donations/pledges.
drop((audit.key=='dbeaver')&audit.label.str.startswith('MassChallenge'),'Non-equity accelerator assistance: no disclosed cash cheque or prize.')
drop((audit.key=='bitcraze')&audit.label.str.startswith('Minc'),'Incubator membership is not evidence that outside money changed hands.')
drop((audit.key=='airbyte')&audit.label.str.startswith('Plug and Play'),'Database participation record not corroborated as capital funding.')
drop((audit.key=='payloadcms')&audit.label.str.startswith('Michigan Business'),'Award cancelled with no disbursement; not funding received. Retained in notes as excluded award.')
drop((audit.key=='supermemoryai')&audit.label.str.startswith('Reported Y Combinator'),'YC Summer Fellows offer/receipt is ambiguous; no completed grant verified.')
drop((audit.key=='neuromore')&audit.label.str.startswith('Republic'),'Primary page labels $30000 as reservations despite contradictory success language; no confirmed settlement.')
# Merge the likely duplicate DBeaver 2023 filing/closing into its confirmed public seed announcement.
drop((audit.key=='dbeaver')&audit.label.str.startswith('Reported Seed II'),'March 2023 record plausibly same April announced seed; not treated as a second independent event.')
# Preserve disclosed investor associations without manufacturing three cash events.
drop(audit.key=='Fincept-Corporation','Three date-less database incubation records do not substantiate three discrete cash transfers.')
append_event({'key':'Fincept-Corporation','type':'venture_unspecified','label':'Reported accelerator backing; cash transfer and event structure unverified','investors':'CHARUSAT Innovative Ventures Foundation | i-Hub Gujarat | Startup Gujarat','source_url':'https://pitchbook.com/profiles/company/649229-41','confidence':'low'},'Rules allow unallocated disclosed backers; one provisional association row instead of three invented event assignments.')
# Source recovery corrections and unequivocal lead attributions.
change((audit.key=='airbyte')&audit.date.isin(['2020-01','2020-05']),'Company retrospective states both amounts; earlier pre-seed tactics post does not.',source_url='https://airbyte.com/blog/what-the-fundraising-roller-coaster-looks-like-under-covid-19-lockdown')
change((audit.key=='jfrog')&audit.label.eq('Series C'),'Source lists five new investors but does not designate them leads.',lead='',investors='Scale Venture Partners | Sapphire Ventures | Battery Ventures | Vintage Investment Partners | Qumra Capital | Gemini Israel Ventures | VMware')
change((audit.key=='open-bionics')&audit.label.eq('Series A'),'Normalize fund vehicle and nominee names to firms; co-leads as UK government announcement.',lead='Foresight Group | Ananda Impact Ventures | Downing Ventures',investors='Rathbones',source_url=supp_source_map['openbionics_sbri_gov'])
change((audit.key=='open-bionics')&audit.label.str.startswith('Disney Accelerator'),'Primary contemporary reporting explicitly confirms the venture received accelerator capital.',source_url='https://www.wired.com/story/open-bionics-hands-joel-gibbard/')
change((audit.key=='anaconda')&audit.label.str.startswith('Seed equity'),'Use final amended filing that actually states amount sold.',source_url='https://www.sec.gov/Archives/edgar/data/1538860/000153886012000004/xslFormDX01/primary_doc.xml',date='2012-01',label='Seed equity private placement; January 2012 notice, final August 2012 amended amount')
# Convert fractional-dollar ledger amounts to nearest whole USD; preserve rounding fact in notes.
from decimal import Decimal, ROUND_HALF_UP
for i,r0 in audit.iterrows():
 if r0['amount_usd'] and '.' in r0['amount_usd']:
  audit.at[i,'amount_usd']=str(Decimal(r0['amount_usd']).quantize(Decimal('1'),rounding=ROUND_HALF_UP))
# Crowdfunding funders are platform backers, not platforms investing themselves.
for i,r0 in audit.iterrows():
 if r0['type'].startswith('crowdfunding'):
  if 'Kickstarter' in r0['label']:funder='Kickstarter backers'
  elif 'Crowd Supply' in r0['label']:funder='Crowd Supply backers'
  elif 'Crowdcube' in r0['label']:funder='Crowdcube backers'
  elif 'Open Hand' in r0['label']:funder='Indiegogo backers'
  elif r0['key']=='farmbot':funder='Self-hosted campaign backers'
  else:funder=''
  audit.at[i,'lead']='';audit.at[i,'investors']=funder
# Programs and single lenders are not labeled leads absent an explicit lead designation.
for i,r0 in audit.iterrows():
 if r0['type'] in ['grant_philanthropic','grant_government','prize','accelerator','debt'] and r0['lead'] and not r0['investors']:
  audit.at[i,'investors']=r0['lead'];audit.at[i,'lead']=''
# Use date of public announcement where a closing month was substituted.
change((audit.key=='fosrl')&audit.type.eq('seed'),'Founder post and company public announcement are October; August was closing date.',date='2025-10')
change((audit.key=='va-linux')&audit.label.str.startswith('Series A first'),'November 12 1998 is announcement; October was first closing.',date='1998-11')
change((audit.key=='va-linux')&audit.label.str.startswith('Series A additional'),'Intel public announcement was March 1999; February was closing.',date='1999-03')
# No invented January month for a winter batch / no award-announcement date from project starts.
change((audit.key=='getzep')&audit.type.eq('accelerator'),'January 1 in databases is a placeholder; YC states Winter 2024 only.',date='')
print('Working table',len(audit),'rows; canonical types',audit.type.value_counts().to_dict())
Path('audit_work/reconciliation_log.json').write_text(json.dumps(changes,indent=2))

Working table 203 rows; canonical types {'debt': 35, 'seed': 33, 'series_a': 24, 'venture_unspecified': 20, 'grant_government': 17, 'accelerator': 8, 'secondary': 8, 'series_b': 8, 'convertible_note': 6, 'prize': 6, 'crowdfunding_reward': 6, 'pre_seed': 5, 'series_c': 5, 'other': 4, 'corporate_strategic': 3, 'grant_philanthropic': 3, 'series_d': 3, 'none_found': 2, 'crowdfunding_equity': 2, 'angel': 2, 'series_f_plus': 2, 'series_e': 1}


21448

In [108]:
print(audit[audit.amount_usd.eq('')][['key','type','label','date','lead','investors']].to_string(index=False))

                key                type                                                                                     label    date                                lead                                                                                                                                                                                                              investors
       BruceDevices               other                  Ongoing Patreon community crowdfunding (campaign launched November 2024) 2024-11                                                                                                                                                                                                                                                           
      Cheshire-Labs          none_found                                No disclosed outside funding found after targeted searches                                                                                             

In [109]:
fx_jobs=[]
def schedule_fx(mask,currency,amount,month=None,detail=''):
 for i in audit.index[mask]:
  r0=audit.loc[i]
  fx_jobs.append({'index':int(i),'key':r0['key'],'label':r0['label'],'currency':currency,'amount':str(amount),'month':month or r0['date'],'detail':detail})
schedule_fx((audit.key=='odoo')&audit.date.eq('2010-01'),'EUR',3000000)
schedule_fx((audit.key=='odoo')&audit.date.eq('2022-06'),'EUR',112000000)
schedule_fx((audit.key=='odoo')&audit.date.eq('2023-06'),'EUR',150000000)
schedule_fx((audit.key=='odoo')&audit.label.str.startswith('AVP and Alkeon'),'EUR',60000000)
for word,amt in [('AXIOM',136850),('X-MINE',470750),('VEDLIoT',645250),('EXCEED',300000),('TRISTAN',788593.75)]:
 schedule_fx((audit.key=='antmicro')&audit.label.str.startswith(word),'EUR',amt,detail='Company-specific EU contribution; not consortium total.')
schedule_fx((audit.key=='bela')&audit.label.str.startswith('Bela Kickstarter'),'GBP',54902)
schedule_fx((audit.key=='bela')&audit.label.str.startswith('Trill Kickstarter'),'GBP',25375)
schedule_fx((audit.key=='bcn3d')&audit.date.eq('2020-07'),'EUR',2800000)
schedule_fx((audit.key=='bcn3d')&audit.date.eq('2024-11'),'EUR',3000000)
schedule_fx((audit.key=='bitcraze')&audit.type.eq('grant_government'),'SEK',317289,detail='Exchange month is disclosed project start, award-announcement month unknown.')
schedule_fx((audit.key=='open-bionics')&audit.label.str.startswith('Open Hand'),'GBP',43593)
schedule_fx((audit.key=='open-bionics')&audit.label.str.startswith('UK James Dyson'),'GBP',2000)
schedule_fx((audit.key=='open-bionics')&audit.label.str.startswith('SBRI Healthcare feasibility'),'GBP',100000)
schedule_fx((audit.key=='open-bionics')&audit.label.str.startswith('Seed financing; GBP 435000'),'GBP',435000)
schedule_fx((audit.key=='open-bionics')&audit.label.str.startswith('West of England'),'GBP',20000)
schedule_fx((audit.key=='open-bionics')&audit.label.str.startswith('Venture financing; GBP 2.8'),'GBP',2800000)
schedule_fx((audit.key=='open-bionics')&audit.label.str.startswith('Crowdcube equity campaign; GBP 1514842'),'GBP',1514842)
schedule_fx((audit.key=='open-bionics')&audit.label.str.startswith('Crowdcube equity campaign; GBP 1118096'),'GBP',1118096)
schedule_fx((audit.key=='open-bionics')&audit.label.str.startswith('NatWest'),'GBP',600000)
for a in ukri_awards:
 d=ukri_details[a['href']];p=next(p for p in d['participantValues'] if p['organisationName']=='OPEN BIONICS LTD')
 schedule_fx((audit.key=='open-bionics')&audit.source_url.eq(a['href']),'GBP',p['grantOffer'],detail='Company-specific grant offer; exchange month is project start, not inferred announcement.')
# Cash-allotment amounts from original filings; issue prices exclude nominal-value accounting figures.
for token,shares in [('28 March–2 April',340251),('24 April',86118),('21 May',21825),('23 July',28580)]:
 schedule_fx((audit.key=='open-bionics')&audit.label.str.contains(token,regex=False),'GBP',str(Decimal(shares)*Decimal('6.37')),detail=f'{shares} cash shares x GBP6.37 issue price; unnamed subscribers.')
print('Currency conversions to calculate',len(fx_jobs))
fx_rates={}
async def get_fx(currency,month):
 key=(currency,month)
 if key in fx_rates:return
 date=month+'-15'
 url=f'https://api.frankfurter.dev/v1/{date}?base={currency}&symbols=USD'
 async with sem:
  async with httpx.AsyncClient(timeout=45,follow_redirects=True) as cl:
   r=await cl.get(url)
   if r.status_code==200:fx_rates[key]={'url':str(r.url),**r.json()}
   else:fx_rates[key]={'url':str(r.url),'error':r.status_code,'text':r.text[:300]}
await asyncio.gather(*(get_fx(c,m) for c,m in set((x['currency'],x['month']) for x in fx_jobs)))
print('FX error records',[v for v in fx_rates.values() if 'error' in v])
Path('audit_work/fx_rates.json').write_text(json.dumps(list(fx_rates.values()),indent=2))

Currency conversions to calculate 31


FX error records []


5541

In [110]:
fxdocs=await fetch_parsed_page('https://frankfurter.dev/')
retain({'url':'https://frankfurter.dev/','title':fxdocs.splitlines()[0],'full_text':fxdocs},'Frankfurter')
print(fxdocs[:6500])
for job in fx_jobs:
 rate=fx_rates[(job['currency'],job['month'])]
 val=Decimal(job['amount'])*Decimal(str(rate['rates']['USD']))
 usd=str(val.quantize(Decimal('1'),rounding=ROUND_HALF_UP))
 audit.at[job['index'],'amount_usd']=usd
 label=job['label']
 if job['detail'] and 'Company-specific' in job['detail'] and job['key']=='antmicro':
  audit.at[job['index'],'label']=label+f"; company share {job['currency']} {job['amount']}"
 if job['key']=='open-bionics' and label.startswith('Innovate UK:'):
  audit.at[job['index'],'label']=label+f"; company grant GBP {job['amount']}"
 note_additions[job['key']].append(f"{label}: {job['currency']} {job['amount']} converted to USD {usd} at {rate['rates']['USD']} USD/{job['currency']} ({rate['date']}; {rate['url']}). {job['detail']}")
for rate in fx_rates.values():
 retain({'url':rate['url'],'title':f"Historical {rate['base']} to USD exchange rate, {rate['date']}",'publishedDate':rate['date'],'full_text':json.dumps(rate)},'Frankfurter')
print('Applied',len(fx_jobs),'conversions, rounded to whole dollars.')
# Add company-disclosed consortium grants missed by VC-centric event summaries, without inventing allocations or dates.
for label in ['VIVALDI (MIP-20211033)','HIBRI4D (MIP-20211041)']:
 append_event({'key':'bcn3d','type':'grant_government','label':label+' R&D project grant; 2021 call, award month undisclosed','investors':'CDTI | European Union NextGenerationEU','source_url':supp_source_map['bcn3d_about'],'confidence':'medium'},'Company explicitly identifies itself as consortium beneficiary and names the public funders; no company allocation disclosed.')
change((audit.key=='BruceDevices')&audit.type.eq('other'),'Ongoing donation programme; identify funder class rather than implying platform equity.',investors='Patreon backers')
change((audit.key=='TecharoHQ')&audit.type.eq('other'),'Use ongoing-support disclosure date; not an invented round-close date.',label='Ongoing donations and corporate sponsorship; funding update August 2025',date='2025-08')
print('After supplements:',len(audit),'rows')

Frankfurter | API kurs gratis

[v2](https://frankfurter.dev/id/) [v1](https://frankfurter.dev/id/v1/) [Mata Uang](https://frankfurter.dev/id/currencies/) [Penyedia](https://frankfurter.dev/id/providers/) [Integrasi](https://frankfurter.dev/id/integrations/) [Deploy](https://frankfurter.dev/id/deploy/) [OpenAPI](https://api.frankfurter.dev/v2/openapi.json)

![Frankfurter](https://frankfurter.dev/images/logo.png?v=1795589442 "width=240 | height=128")

* [Kurs](https://frankfurter.dev/id/#rates)
* [Kurs Tunggal](https://frankfurter.dev/id/#rate)
* [Mata Uang](https://frankfurter.dev/id/#currencies)
* [Detail Mata Uang](https://frankfurter.dev/id/#currency)
* [Penyedia](https://frankfurter.dev/id/#providers)
* [Konversi](https://frankfurter.dev/id/#conversion)
* [Error](https://frankfurter.dev/id/#errors)
* [Hosting Mandiri](https://frankfurter.dev/id/#self-hosting)
* [FAQ](https://frankfurter.dev/id/#faq)

# Frankfurter API kurs dan mata uang

[2.109](https://github.com/lineofflight/frank

In [111]:
# Check funding-vehicle normalization where a fund name is not the investing firm's name.
normal_queries=['"FIDES 4" "Accurafy" BCN3D','"Rathbone Nominees" "Open Bionics"','"Pioneer" "E2B" investment Devbook','"Terrastruct" "Fathom Capital" "2021"']
normal_results=await asyncio.gather(*(web_search(q,num_results=3) for q in normal_queries))
for q1,rrr in zip(normal_queries,normal_results):
 print('\n',q1)
 for x in rrr:print(x['title'],x['url'],x.get('snippet','')[:950])
Path('audit_work/normalization_checks.json').write_text(json.dumps(list(zip(normal_queries,normal_results)),indent=2))


 "FIDES 4" "Accurafy" BCN3D
BCN3D spins off and announces $3 million in funding to boost its leadership in the IDEX 3D Printing technology - BCN3D Technologies https://bcn3d.com/bcn3d-spin-offs-announces-3-million-funding-idex-3d-printing-technology/ # BCN3D spins off and announces $3 million in funding to boost its leadership in the IDEX 3D Printing technology

 Mar 27, 2019 

## Barcelona, March 27, 2019 – The research group that created the disruptive IDEX 3D printing technology democratizing industry-grade features to the desktop market, now becomes a standalone company after raising $3M seed funding by VC firms Accurafy4, JME VC and Victoria. BCN3D currently leads the IDEX technology -Independent Dual EXtruder-, which doubles productivity compared to conventional desktop 3D printers. BCN3D recently achieved the 5,000 3D printers sold milestone through a network of 60 active distribution partners operating worldwide.

The round turns BCN3D into one of the top 10 desktop 3D printin

69483

In [112]:
t=src(resolved6['es_forbes_2023'])
for term in ['term sheet','Lux Capital','completed','will raise','raising','closed']:
 print('\n',term)
 for m in list(re.finditer(re.escape(term),t,re.I))[:3]:print(t[max(0,m.start()-220):m.start()+650].replace('\n',' '))
# Funding-source text for BCN3D follow-ons, to normalize FIDES4 rather than a fund vehicle.
for u in [resolved5['REF:extra:3'],resolved5['REF:lasturls:3']]:
 t=src(u)
 print('\nBCN3D follow-on',u)
 for term in ['Fides','FIDES','Mundi','Mondrag','Accurafy']:
  m=re.search(term,t,re.I)
  if m:print(t[max(0,m.start()-200):m.start()+500].replace('\n',' '))
# The English V1 Frankfurter documentation describes the data source separately from the V2 blended endpoint.
v1doc=await fetch_parsed_page('https://frankfurter.dev/v1/')
print('\nFX V1 docs',v1doc[:1100])
retain({'url':'https://frankfurter.dev/v1/','title':v1doc.splitlines()[0],'full_text':v1doc},'Frankfurter')


 term sheet

 Lux Capital
s/)  [Innovation](https://www.forbes.com/innovation/)[Venture Capital](https://www.forbes.com/venture-capital/)  [Editors' Pick](https://www.forbes.com/editors-picks/)  # Ex-Meta Researchers Have Raised $40 Million From Lux Capital For An AI Biotech Startup  By[Kenrick Cai](https://www.forbes.com/sites/kenrickcai/ "https://www.forbes.com/sites/kenrickcai/"),  Former Staff  and[Iain Martin](https://www.forbes.com/sites/iainmartin/ "https://www.forbes.com/sites/iainmartin/"),  Senior Editor.  Aug 25, 2023, 04:02pm EDTUpdated Aug 25, 2023, 06:11pm EDT  ![Chain of amino acid or bio molecules called protein - 3d illustration](https://imageio.forbes.com/specials-images/imageserve/64e8a8e250f468593cc6d0d5/Chain-of-amino-acid-or-bio-molecules-called-protein---3d-illustration/0x0.jpg?format=jpg&crop=1471,980,x254,y0,safe&width=1440)  Former Meta researchers who devel
strial chemicals.  By June, the startup was pitching venture capitalists for seed financing to advance


FX V1 docs Frankfurter | Документация API v1

[v2](https://frankfurter.dev/ru/) [v1](https://frankfurter.dev/ru/v1/) [Валюты](https://frankfurter.dev/ru/currencies/) [Источники](https://frankfurter.dev/ru/providers/) [Интеграции](https://frankfurter.dev/ru/integrations/) [Deploy](https://frankfurter.dev/ru/deploy/) [OpenAPI](https://api.frankfurter.dev/v2/openapi.json)

![Frankfurter](https://frankfurter.dev/images/logo.png?v=1795589442 "width=240 | height=128")

* [Последние курсы](https://frankfurter.dev/ru/v1/#latest)
* [Исторические курсы](https://frankfurter.dev/ru/v1/#historical)
* [Временные ряды](https://frankfurter.dev/ru/v1/#time-series)
* [Доступные валюты](https://frankfurter.dev/ru/v1/#currencies)
* [Конвертация валют](https://frankfurter.dev/ru/v1/#conversion)

# Frankfurter (v1)

API v1 имеет статус deprecated в пользу [v2](https://frankfurter.dev/ru/), но продолжит работать.

Получайте последние курсы, исторические курсы или временные ряды.

## [Последние курсы](https:

'auditf9fcc99d4d34'

In [113]:
# Download/fetch sources introduced during reconciliation and refresh the archive.
for a in ukri_awards:
 u=a['href'];source_cache[u]={'url':u,'title':'UKRI award '+a['projectTitle'],'text':json.dumps(ukri_details[u],indent=2)}
needed=list(dict.fromkeys(u for val in audit.source_url for u in val.split(' | ')))
await asyncio.gather(*(get_audit_source(u) for u in needed))
short=[]
for u in needed:
 t=src(u)
 if len(t)<1000:short.append((len(t),u,t[:130].replace('\n',' ')))
print('Short evidence pages',len(short))
for x in short:print(x)
# Normalize direct backers' usual firm names without expanding named people from memory.
name_map={'Walden Catalyst Ventures':'Walden Catalyst','SRIW':'Wallonie Entreprendre','Coatue Management':'Coatue','W. R. Hambrecht + Co.':'WR Hambrecht + Co','Transpose Platform Management':'Transpose Platform','A&E Investments':'A&E Investment','WTT Investment':'WTT Fubon Family','Browder Capital':'Browder Capital','Berkeley SkyDeck Fund':'Berkeley SkyDeck Fund'}
for col in ['lead','investors']:
 for i,v in audit[col].items():
  names=[]
  for name in v.split(' | '):
   name=name_map.get(name.strip(),name.strip())
   if name and name not in names:names.append(name)
  audit.at[i,col]=' | '.join(names)
# Standalone sponsor supporters are other funding, not equity holders; disclose channels and role.
note_additions['TecharoHQ'].append('I use type other for ongoing donations/sponsorship, not a priced round. The August 2025 update reports support through GitHub Sponsors, Patreon and Liberapay; recurring rates are not capital totals.')
note_additions['BruceDevices'].append('I use type other for the ongoing Patreon donation programme, not a bounded reward campaign. I leave total raised blank; monthly receipts are not a round total.')
Path('audit_work/source_cache.json').write_text(json.dumps(source_cache,indent=2))
print('Sources cached',len(source_cache))

Short evidence pages 5
(106, 'https://meta.farm.bot/assets/data/timeline/accounts/family_loan.csv', 'date,value 2017-12-31,0 2018-08-21,133000 2018-10-31,93000 2018-11-28,53000 2019-01-02,33000 2019-02-05,0 ')
(37, 'https://www.sec.gov/Archives/edgar/data/1096199/000109619912000004/gknt20111231-10k.htm', '10-K 1 gknt20111231-10k.htm FORM 10-K')
(33, 'https://www.sec.gov/Archives/edgar/data/1096199/000109619914000008/R9.htm', 'XML 33 R9.htm IDEA: XBRL DOCUMENT')
(37, 'https://www.sec.gov/Archives/edgar/data/1096199/000109619915000006/gknt20141231-10k.htm', '10-K 1 gknt20141231-10k.htm FORM 10-K')
(37, 'https://www.sec.gov/Archives/edgar/data/1096199/000144530513000421/gknt20121231-10k.htm', '10-K 1 gknt20121231-10k.htm FORM 10-K')
Sources cached 237


In [114]:
from bs4 import BeautifulSoup
for n,u,_ in short:
 if 'sec.gov' in u:
  async with httpx.AsyncClient(timeout=60,follow_redirects=True) as cl:
   r=await cl.get(u)
  if r.status_code==200:
   text=BeautifulSoup(r.text,'html.parser').get_text(' ',strip=True)
   source_cache[u]['direct_text']=text
   print(u,'status',r.status_code,'chars',len(text),'Wells Fargo',text.find('Wells Fargo'))
  else:print(u,'direct status',r.status_code)
Path('audit_work/source_cache.json').write_text(json.dumps(source_cache,indent=2))
# Check FX provider documentation and record data-source limitation precisely.
for term in ['ECB','European','Европ','16:00','central']:
 matches=list(re.finditer(re.escape(term),v1doc,re.I))
 if matches:print(term,v1doc[max(0,matches[0].start()-120):matches[0].start()+500].replace('\n',' '))
print('Current audit rows',len(audit))

https://www.sec.gov/Archives/edgar/data/1096199/000109619912000004/gknt20111231-10k.htm direct status 403
https://www.sec.gov/Archives/edgar/data/1096199/000109619914000008/R9.htm direct status 403


https://www.sec.gov/Archives/edgar/data/1096199/000109619915000006/gknt20141231-10k.htm direct status 403
https://www.sec.gov/Archives/edgar/data/1096199/000144530513000421/gknt20121231-10k.htm direct status 403
16:00 урсы](https://frankfurter.dev/ru/v1/#latest)  Получите курсы за последний рабочий день; они обновляются ежедневно около 16:00 CET.  ``` /* https://api.frankfurter.dev/v1/latest */ {   "base": "EUR",   "date": "2026-11-25",   "rates": {     "AUD": 1.6644,     "BRL": 6.4531,     "CAD": 1.5591,     "CHF": 0.9085,     "CNY": 7.2999,     "...": "..."   } } ```  Смените базовую валюту параметром `base`. По умолчанию используется EUR.  ``` /* https://api.frankfurter.dev/v1/latest?base=USD */ {   "base": "USD",   "date": "2026-11-25",   "rates": {     "AUD": 1.5221,     "BRL": 5.9013,     "CAD": 1.4258,     "CHF": 0.8308
Current audit rows 205


In [115]:
# Distinguish loan originations from ledger simulations or scheduled transactions.
for rec,t in zip(loan_records,loan_texts):
 records=list(csv.DictReader(io.StringIO(t)))
 if rec['account']=='Shopify Loan':
  funding=[z for z in records if z.get('Transaction Type')=='funding']
  print(Path(rec['file']).name,[(z.get('Date'),z.get('Amount'),z.get('Status')) for z in funding])
print('\nAll low-confidence final rows:')
print(audit[audit.confidence.eq('low')][['key','type','label','date','amount_usd']].to_string(index=False))
print('\nNotes count',len(allnotes),'incomplete keys',[k for k,v in allnotes.items() if v['incomplete']])

shopify_capital_loan_2018-11-21.csv [('2018-11-20', '$63,000.00 USD', 'completed')]
shopify_capital_loan_2020-01-03.csv [('2020-01-02', '$99,000.00 USD', 'completed')]
shopify_capital_loan_2020-07-30.csv [('2020-07-29', '$80,000.00 USD', 'completed')]
shopify_capital_loan_2022-08-26.csv [('2022-08-25', '$75,000.00 USD', 'completed')]
shopify_capital_loan_2023-03-15.csv [('2023-01-06', '$40,000.00 USD', 'completed')]
shopify_capital_loan_2023-06-08.csv [('2023-06-07', '$60,000.00 USD', 'completed')]
shopify_capital_loan_2023-12-20.csv [('2023-10-31', '$50,000.00 USD', 'completed')]
shopify_capital_loan_2024-06-03.csv [('2024-05-15', '$69,000.00 USD', 'completed')]
shopify_capital_loan_2025-02-20.csv [('2024-11-15', '$39,000.00 USD', 'pending')]
shopify_capital_loan_2025-06-20.csv [('2025-06-13', '$80,000.00 USD', 'pending')]

All low-confidence final rows:
                key                type                                                                                 label    dat

In [116]:
# Exclude pending loan originations rather than silently treating scheduled entries as money received.
pending_urls=[]
for rec,t in zip(loan_records,loan_texts):
 if rec['account']=='Shopify Loan':
  z=next(z for z in csv.DictReader(io.StringIO(t)) if z.get('Transaction Type')=='funding')
  if z.get('Status')!='completed':pending_urls.append(urljoin(farm_manifest_url,rec['file']))
drop(audit.source_url.isin(pending_urls),'Ledger funding record is pending, not completed; excluded from received outside funding.')
note_additions['farmbot'].append('I excluded the USD39000 and USD80000 Shopify entries marked pending; transaction dates, not inconsistent filenames, date completed loans. Family-loan and revolver balances are not amounts newly borrowed.')
# Check Odoo's first institutional lead and primary round naming.
u=audit[(audit.key=='odoo')&audit.type.eq('series_a')].iloc[0].source_url
t=src(u)
for term in ['Sofinnova','lead','2010','2009']:
 for m in list(re.finditer(term,t,re.I))[:3]:print(term,t[max(0,m.start()-100):m.start()+500].replace('\n',' '))
print('Working rows',len(audit))

Sofinnova Pinckaers (fp)  OpenERP announced in january 2010 that it raised 3 million euros. The investors are Sofinnova Partners, represented by Olivier Sichel, and the Iliad's managers, Xavier Niel and Olivier Rosenfeld. The funds raised will allow OpenERP to achieve its ambition to be one of the leading application business suite vendors worldwide.  OpenERP's strategy is based on a unique ecosystem which combines the resources of the community, of its partner’s network and of the editor. The community contributes to the ongoing enrichment of the OpenERP solution. The partners tailor the solution to cu
lead and Olivier Rosenfeld. The funds raised will allow OpenERP to achieve its ambition to be one of the leading application business suite vendors worldwide.  OpenERP's strategy is based on a unique ecosystem which combines the resources of the community, of its partner’s network and of the editor. The community contributes to the ongoing enrichment of the OpenERP solution. The partner

In [117]:
# Filing disclosure dates are preferable to first-sale dates for this announcement-date schema.
for key in ['anaconda','efabless','nicira']:
 for _,r0 in audit[(audit.key==key)&audit.source_url.str.contains('sec.gov')].iterrows():
  u=r0.source_url;t=src(u)
  print('\n',key,r0.label,'CURRENT',r0.date)
  for term in ['Signature Date','signatureDate','Date of First Sale','Date of Filing','Filing Date']:
   m=re.search(term,t,re.I)
   if m: print(t[max(0,m.start()-60):m.start()+350].replace('\n',' '))
# Grant start months are retained in labels but are not fabricated as announcement dates.
change((audit.key=='bitcraze')&audit.type.eq('grant_government'),'Source gives project start, not award announcement.',date='',label='Vinnova grant 2022-01270; efficient micro-drone propellers; project starts 2022-07')
for a in ukri_awards:
 mask=(audit.key=='open-bionics')&audit.source_url.eq(a['href'])
 current=audit.loc[mask,'label'].iloc[0]
 change(mask,'API gives project start, not announcement date; preserve start in label.',date='',label=current+'; starts '+a['startDate'][:7])
# Single event URL, with all additional corroborating sources kept in the venture note.
for i,r0 in audit.iterrows():
 sources=r0['source_url'].split(' | ')
 if len(sources)>1:
  audit.at[i,'source_url']=sources[0]
  note_additions[r0['key']].append(f"Additional sources for {r0['label']}: "+' ; '.join(sources[1:])+'.')
print('Pipe-separated source cells left',audit.source_url.str.contains(r' \| ').sum())


 anaconda Seed equity private placement; January 2012 notice, final August 2012 amended amount CURRENT 2012-01
          |  7\. Type of Filing  |   | New Notice |   |   | Date of First Sale | 2011-12-27 |   |   |   | First Sale Yet to Occur | | - | ---------- | - | - | ------------------ | ---------- | - | - | - | -------------------- | | X | Amendment  |   |   |                    |            |   |   |   |                      |  8\. Duration of Offering  | Does the Issuer intend this offering to last more than on

 anaconda Equity private placement; stage not specified in filing CURRENT 2016-08
--------- |  7\. Type of Filing  | X | New Notice |   |   | Date of First Sale | 2016-08-18 |   |   |   | First Sale Yet to Occur | | - | ---------- | - | - | ------------------ | ---------- | - | - | - | -------------------- | |   | Amendment  |   |   |                    |            |   |   |   |                      |  8\. Duration of Offering  | Does the Issuer intend this offering to l

In [118]:
print(datetime.datetime.now())
# Form-D signature dates near the end of parsed pages indicate disclosure timing.
for key in ['anaconda','efabless','nicira']:
 for _,r0 in audit[(audit.key==key)&audit.source_url.str.contains('sec.gov')].iterrows():
  t=src(r0.source_url)
  dates=re.findall(r'\b20\d{2}-\d{2}-\d{2}\b',t)
  print(key,r0.label[:45],r0.date,'dates tail',dates[-8:])
# Keep checked sources traceable when the final lead assignment differs among otherwise reliable reports.
note_additions['open-bionics'].append('The January 2019 source calls Foresight/Ananda/Downing co-leads; TechCrunch calls only Foresight lead. Counsel calls Foresight and Ananda leads and Downing existing investor. I retain the government co-lead classification; Rathbone Nominees is normalized to Rathbones. Supporting USD5.9m report: '+resolved6['ob_tc']+'; counsel: '+normal_results[1][0]['url']+'.')
# Round-specific evidence links Fathom to the December seed, resolving that participant but not older financing.
retain(normal_results[3][0],'PitchBook')
change((audit.key=='d2lang')&audit.type.eq('seed'),'Fathom investor-level deal table explicitly maps its Terrastruct investment to December 2021 seed.',investors='Haystack | Bloomberg Beta | Fathom Capital')
note_additions['d2lang'].append('I additionally verified Fathom Capital in the December2021 seed via its investor-level deal table: '+normal_results[3][0]['url']+'. Earlier April2021 backing may be an earlier close; no extra unproven event is invented.')

2026-11-26 06:25:29.049712
anaconda Seed equity private placement; January 2012 n 2012-01 dates tail ['2011-12-27', '2012-08-14']
anaconda Equity private placement; stage not specified 2016-08 dates tail ['2016-08-18', '2016-09-01']
anaconda Equity private placement; stage not specified 2017-06 dates tail ['2017-06-27', '2017-07-12']
anaconda Equity private placement; stage not specified 2021-05 dates tail ['2021-05-07', '2021-06-10']
efabless Private debt offering; first sale September 2 2024-09 dates tail ['2024-09-27', '2024-10-05']
nicira Series A preferred stock and unsecured conver 2008-07 dates tail []
nicira Series A preferred stock; first sale January  2009-01 dates tail []
nicira Series A preferred stock; first sale February 2009-02 dates tail ['2009-02-24', '2009-05-28']
nicira Series A preferred stock; December 2009 offer 2009-12 dates tail ['2009-12-18', '2010-01-28']
nicira Series B; USD 11349984 sold by September 2010 2009-12 dates tail ['2009-12-30', '2010-09-22']
nicir

In [119]:
# Apply source-disclosure months for regulatory notices rather than silently using first-sale months.
for old,new in [('2016-08','2016-09'),('2017-06','2017-07'),('2021-05','2021-06')]:
 change((audit.key=='anaconda')&audit.date.eq(old)&audit.source_url.str.contains('sec.gov'),'Date follows Form D signature/disclosure month; first-sale month remains in notes.',date=new)
change((audit.key=='efabless')&audit.type.eq('debt'),'October 2024 filing publicly discloses September first sale.',date='2024-10')
change((audit.key=='nicira')&audit.label.str.startswith('Series A preferred stock; first sale January'),'February 2009 filing disclosure, not January first sale.',date='2009-02')
change((audit.key=='nicira')&audit.label.str.startswith('Series A preferred stock; first sale February'),'May 2009 Form D signature/disclosure, not February first sale.',date='2009-05')
change((audit.key=='nicira')&audit.label.str.startswith('Series A preferred stock; December'),'January 2010 Form D signature/disclosure, not December 2009 first sale.',date='2010-01')
change((audit.key=='nicira')&audit.label.str.startswith('Series B; USD'),'Cited September 2010 amended disclosure supplies final sold amount; December2009 is first sale.',date='2010-09')
# Calendar-year-only disclosure: no made-up month in the cash column.
change((audit.key=='getzep')&audit.label.str.startswith('DV seed'),'Investor portfolio supports seed participation but January2026 SPV filing is not necessarily company financing date.',date='',type='venture_unspecified',label='DV disclosed seed participation; company financing date and structure unannounced')
# Unallocated historical investors explicitly disclosed in current round reporting are not silently lost.
# Rules permit a separate unallocated prior-backing row when their own event cannot be located.
# Here Fathom is now allocated; other unresolved relationships stay in notes rather than duplicate known capital.
note_additions['getzep'].append('I leave the DV participation date blank: its January2026 SPV filing is fund fundraising, not proof of when Zep received primary capital. I do not treat the SPV USD124000 as company proceeds.')
# Replace a generic ongoing-donations type label with the precise rule-supported class.
change((audit.key=='supermemoryai')&audit.type.eq('seed'),'Company and contemporary report call this pre-seed.',type='pre_seed')
# Mark undrawn facilities and subsequent extensions distinctly; these amounts are commitments, not receipts.
note_additions['va-linux'].append('Dates for private events use public announcement/disclosure where verified; Wells Fargo renewals known only by quarter remain undated. Revolver ceilings and increases are credit commitments, not drawn cash; do not sum them with equity.')
print('Table after final date adjudication',len(audit),'rows')
Path('audit_work/reconciliation_log.json').write_text(json.dumps(changes,indent=2))

Table after final date adjudication 203 rows


40877

In [120]:
# Check the few unallocated backer relationships explicitly mentioned in investor portfolios.
for url in ['https://firststar.vc/companies','https://www.stepfunction.vc','https://engineeringcapital.com']:
 if url not in source_cache:await get_audit_source(url)
 text=src(url)
 print('\n',url)
 for term in ['Zep','zep.ai','getzep']:
  m=re.search(re.escape(term),text,re.I)
  if m:print(text[max(0,m.start()-120):m.start()+420].replace('\n',' '));break
print('\nSource naming prior investor Root:')
for x in latest_checks:
 if x['key']=='getzep':
  for r0 in x['results']:
   if 'Root' in str(r0):print(r0['url'],r0.get('snippet','')[:900])


 https://firststar.vc/companies
ps://cdn.prod.website-files.com/68470b64774604bdd08c654e/68bd470c4b64e8c3dd5adde3_industry%20icon%2005.png)  dev tools  Zep  acquired  Memory layer for AI agents, using temporal knowledge graphs to deliver the right context at retrieval time.  ![](https://cdn.prod.website-files.com/68470b64774604bdd08c654e/68bd46f37d0dbcde53bc7305_industry%20icon%2001.png)  ai  ![](https://cdn.prod.website-files.com/68470b64774604bdd08c654e/68bd470c4b64e8c3dd5adde3_industry%20icon%2005.png)  dev tools  Brelle  acquired  Modular, ultra-fast-charging ba



 https://www.stepfunction.vc
ty Cybersecurity, AI Infra](https://www.latentdefense.ai)  [![](https://www.google.com/s2/favicons?sz=128&domain=www.getzep.com) ZepExited www.getzep.com Agent Memory & Context Dev / Eng Platforms, AI Infra](https://www.getzep.com)  [![](https://www.google.com/s2/favicons?sz=128&domain=www.unkey.com) UnkeyExited www.unkey.com Developer Platform for Modern APIs Dev / Eng Platforms, AI Infra](https://www.unkey.com)  [![](https://www.google.com/s2/favicons?sz=128&domain=www.pavoai.com) PavoExited www.pavoai.com Enterprise Superintelligen



 https://engineeringcapital.com
om/#)  [![](https://images.squarespace-cdn.com/content/v1/602fe8eef3f6185d8768e787/3730c78e-fffe-4215-b282-5b6922d7929d/Zep_840_560.png?format=300w)](https://www.engineeringcapital.com/#)  *The investments represented above and below are not representative of all investments in the portfolio of Engineering Capital I, II, III or IV (the “Funds”). Information regarding the full portfolio, including historical performance of the portfolio are available upon request. There is no guarantee that investors in the Funds will have access to th

Source naming prior investor Root:


In [121]:
append_event({'key':'getzep','type':'venture_unspecified','label':'Unallocated disclosed backers; round/date/primary-versus-secondary structure unknown','investors':'First Star Ventures | Step Function','source_url':'https://firststar.vc/companies','confidence':'medium'},'Rules permit a round-less prior-backer record when investor portfolios disclose backing but no original round can be located.')
note_additions['getzep'].append('The additional venture_unspecified row retains First Star Ventures and Step Function portfolio relationships without inventing a common new round; corroborating portfolio: https://www.stepfunction.vc . It must not be added as a separately quantified financing.')
# Any surviving duplicate between lead and participants is removed without adding names.
for i,r0 in audit.iterrows():
 leads=set(r0['lead'].split(' | '))- {''}
 audit.at[i,'investors']=' | '.join(x for x in r0['investors'].split(' | ') if x and x not in leads)
# Save immutable pre-final reconciliation output before final prose and validation.
audit.to_csv('audit_work/reconciled_events.csv',index=False)
print('Reconciled table:',len(audit),'rows;',audit.key.nunique(),'ventures;',sum(audit.type=='none_found'),'none_found')

Reconciled table: 204 rows; 40 ventures; 2 none_found


In [122]:
rnorm=await web_search('"FIDES 4" venture manager fund',num_results=4)
for x in rnorm:print(x['title'],x['url'],x.get('snippet','')[:1000])
# Pull first investor source for BCN3D 2021 if primary pages are bot shells.
u='https://www.webcapitalriesgo.com/el-cdti-aprueba-una-nueva-coinversion-en-bcn3d-por-e1m-a-traves-de-innvierte/'
await get_audit_source(u);t=src(u)
for term in ['FIDES','Fides','Mundi']:
 m=re.search(term,t,re.I)
 if m:print(t[max(0,m.start()-200):m.start()+600])
print('Sourcechars',len(t))

fides 4 - Software für Bürgschaftsbanken und Beteiligungsgesellschaften https://www.exec.de/produkte/fides-4/uebersicht # Software für Bürgschaftsbanken und Beteiligungsunternehmen

## fides 4 - für die komplette Engagementbetreuung

Besondere Merkmale im Überblick

- Umfangreiches Funktionsangebot
- Revisionssicherheit
- Vorgangsorientierte Bedienerführung
- Modularer Aufbau
- Individuelle Erweiterbarkeit
- Kombinationsmöglichkeit kundenindividueller Besonderheiten mit der Funktionalität des Standardpaketes

Zukunftssicherheit

- Moderne Systemarchitektur (browserbasierte Oberfläche, 3-Schichten-Architektur)
- Marktführerschaft mit Erfahrung aus drei Jahrzehnten

Nutzer & Zielgruppe

- Bürgschaftsbanken
- Mittelständische Beteiligungsgesellschaften
- Venture Capital Gesellschaften
- Förderbanken
- sonstige Finanzdienstleistungsunternehmen

Alle Geschäftsprozesse sicher steuern

fides 4 ist die Softwarekomplettlösung für Bürgschaftsverwaltung und Beteiligungsmanagement. Die webbasieren

soras de escritorio 3D con tecnología FFF (fabricación de filamentos fundidos). La inversión de seguimiento se realiza con ALMA MUNDI FUND II, FCRE, FOMENTO PARA LA INNOVACIÓN Y EL DESARROLLO CUATRO (FIDES 4), FCR, MONDRAGÓN FONDO DE PROMOCIÓN FCR y la aportación del CDTI es de 1.035.000 euros.

Desde enero de 2021 y hasta la fecha, a través de Innvierte el CDTI ha cerrado operaciones de coinversión para promover la innovación empresarial en 26 empresas con una aportación de 33.564.000 euros.

**El Programa Innvierte**

El programa INNVIERTE se enmarca dentro de la Estrategia Española de Ciencia, Tecnología e Innovación 2021-2027, aprobada por Acuerdo del Consejo de Ministros, en septiembre de 2020. Esta Estrategia contiene los objetivos, las reformas y las medidas que deben adoptarse en t
soras de escritorio 3D con tecnología FFF (fabricación de filamentos fundidos). La inversión de seguimiento se realiza con ALMA MUNDI FUND II, FCRE, FOMENTO PARA LA INNOVACIÓN Y EL DESARROLLO CUATRO 

In [123]:
rfides=await web_search('"FIDES" "CUATRO" gestora fondo',num_results=4)
for x in rfides:print(x['title'],x['url'],x.get('snippet','')[:1300])
change((audit.key=='bcn3d')&audit.label.str.startswith('Follow-on equity co-investment; CDTI contribution EUR 1.035'),'Use accessible event-specific source, not a bot shell.',source_url=u)
# Known grant/philanthropic funds use the agency name, not an invented equity lead.
for i,r0 in audit.iterrows():
 if r0['type'] in ['grant_government','grant_philanthropic','prize']:
  funders=list(dict.fromkeys([n for v in [r0['lead'],r0['investors']] for n in v.split(' | ') if n]))
  audit.at[i,'lead']='';audit.at[i,'investors']=' | '.join(funders)
print('Grant/prize lead fields normalized.')

FOLLETO INFORMATIVO FIDES CAPITAL SCR S.A. https://www.cnmv.es/webservices/verdocumento/ver?e=C%2FtThB8pvVBJAa%2FL4mVetE%2BGdzDW%2Br8xdxvYJ6h7QbpsMfXZZiXdwvL92%2BZz4Gks ## FOLLETO INFORMATIVO FIDES CAPITAL SCR S.A.

Fecha del folleto: Mayo 2024

De conformidad con lo dispuesto en el artículo 68 de la Ley 22/2014, de 12 de noviembre, reguladora de entidades de capital-riesgo, otras entidades de inversión colectiva de tipo cerrado y las sociedades gestoras de entidades de inversión colectiva de tipo cerrado, este Folleto informativo recoge la información necesaria para que el inversor pueda formarse un juicio fundado sobre la inversión que se le propone con carácter privado y estará a disposición de los inversores y accionistas en el domicilio de FIDES CAPITAL SCR S.A. No obstante, la información que contiene el presente Folleto informativo puede verse modificada en el futuro, y el presente Folleto actualizado en los términos previstos legalmente. Dichas modificaciones se acordarán y se 

In [124]:
rmanager=await web_search('"FOMENTO PARA LA INNOVACION Y EL DESARROLLO CUATRO" gestora',num_results=3)
for x in rmanager:print(x['title'],x['url'],x.get('snippet','')[:1800])

CNMV - Listado completo de fondos de capital-riesgo https://www.cnmv.es/portal/consultas/mostrarlistados?id=1&lang=es&page=19 CNMV - Listado completo de fondos de capital-riesgo 

# Listado
...
- - FOMENTO PARA LA INNOVACION Y EL DESARROLLO CUATRO, FCR
- Número y fecha de registro oficial: 197 - 01/07/2016 
- Gestora: JME VENTURE CAPITAL, SGEIC, S.A. (número de registro: 116).
CNMV - Información de fondos de capital-riesgo - FOMENTO PARA LA INNOVACION Y EL DESARROLLO CUATRO, FCR - Datos Generales https://www.cnmv.es/portal/consultas/ecr/fondo?lang=es&nif=V87586632 CNMV - Información de fondos de capital-riesgo - FOMENTO PARA LA INNOVACION Y EL DESARROLLO CUATRO, FCR - Datos Generales 

 FOMENTO PARA LA INNOVACION Y EL DESARROLLO CUATRO, FCR 

 Fecha de los datos 2026-10-06 Buscar 

 Fondo de Capital-Riesgo 

| Nº Registro Oficial | Fecha Registro Oficial | Fecha último folleto | Folleto (*) | DFI (*) |
| --- | --- | --- | --- | --- |
| 197 | 01/07/2016 | | | |

- Datos Generales
- Audi

In [125]:
# Resolve the identified FIDES4 vehicle to its usual manager name using Spain's securities regulator.
retain(rmanager[1],'CNMV')
change((audit.key=='bcn3d')&audit.investors.str.contains('FIDES 4',regex=False),'CNMV identifies JME Venture Capital as FIDES4 manager; do not output the fund vehicle.',investors='CDTI Innvierte | Mundi Ventures | JME Ventures | Mondragon Ventures')
note_additions['bcn3d'].append('FIDES4 is normalized to JME Ventures, using the CNMV manager record: '+rmanager[1]['url']+'. The EUR1035000 (2021) and EUR315000 (2024) amounts are CDTI tickets only, not full rounds, so amount_usd remains blank for those events.')
print('Named fund vehicle normalized to JME Ventures.')

Named fund vehicle normalized to JME Ventures.


In [126]:
# Inspect all person names and remaining source-backed firm forms once before locking the audit.
people=sorted({n for c in ['lead','investors'] for v in audit[c] for n in v.split(' | ') if '(person)' in n})
print('\n'.join(people))
print('\nPossible short names / generic investors:')
print(sorted({n for c in ['lead','investors'] for v in audit[c] for n in v.split(' | ') if n and (';' in n or n in ['DV','Intel','Amazon','European Commission','Buildspace'])}))
print('Incomplete',sum(n['incomplete'] for n in allnotes.values()))

Alain Rossmann (person)
Amber Atherton (person)
Amjad Masad (person)
Auren Hoffman (person)
Barry McCardel (person)
Bob Young (person)
Bryan Helmig (person)
Calvin French-Owen (person)
Charles Zedlewski (person)
Clem Delangue (person)
Cristobal Valenzuela (person)
Dane Knecht (person)
Daniel Gross (person)
David Cramer (person)
Dev Ittycheria (person)
Elad Gil (person)
Eric Zelikman (person)
Garry Tan (person)
Guillermo Rauch (person)
Hock Tan (person)
Igor Babuschkin (person)
Jakub Jurových (person)
Jeff Dean (person)
John Schulman (person)
Jordan Tigani (person)
Julian Weisser (person)
Juraj Masár (person)
Ken Goldman (person)
Lip-Bu Tan (person)
Liu Jiang (person)
Logan Kilpatrick (person)
Matt Mullenweg (person)
Matěj Turek (person)
Max Beauchemin (person)
Nat Friedman (person)
Naval Ravikant (person)
Olivier Pomel (person)
Olivier Rosenfeld (person)
Paul Copplestone (person)
Ram Shriram (person)
Robert Nishihara (person)
Sam Shah (person)
Scott Johnston (person)
Shay Banon (person

In [127]:
personqs=['Clem Delangue Clément Hugging Face CEO biography','DV deora.vc Deora Ventures venture Sharran','Koh Tuck Lye investor Speedinvest MindsDB 2018']
personres=await asyncio.gather(*(web_search(q,num_results=2) for q in personqs))
for q1,rrr in zip(personqs,personres):
 print('\n',q1)
 for x in rrr:print(x['title'],x['url'],x.get('snippet','')[:900])


 Clem Delangue Clément Hugging Face CEO biography
[TO SUBMIT] Congressional Hearing Biography and Testimony https://www.congress.gov/118/meeting/house/116078/witnesses/HHRG-118-SY00-Bio-DelangueC-20230622.pdf Biography 
Clement Delangue is co-founder and CEO of Hugging Face, a 100,000+ member community 
democratizing AI through open source and open science. Clement started his career in product 
at Moodstocks, a machine learning startup for computer vision (acquired by Google in 2016). He 
also co-founded several startups, including UniShared and VideoNot.es. He is passionate about 
building AI products.
Open-Sourcing the Future of AI https://sequoiacap.com/article/clem-delangue-spotlight Hugging Face’s Clement Delangue, the man behind the emoji, pushes AI to rewrite old rules.

Published June 29, 2023

by Sam Eifling

...

He was packed inside a lecture hall at the University College in Dublin, where Delangue was continuing a hopscotch of study abroad posts, from his full-time univer

In [128]:
retain(personres[0][1],'Sequoia Capital')
for col in ['lead','investors']:
 audit[col]=audit[col].str.replace('Clem Delangue (person)','Clement Delangue (person)',regex=False)
note_additions['letta-ai'].append('Clem Delangue is expanded to Clement Delangue using Sequoia’s biography: '+personres[0][1]['url']+'.')
base_notes={
'BruceDevices': 'I checked the official site/repository, donation links, Patreon, Ko-fi, Open Collective and funding/grant/accelerator searches. Graphtreon documents a Patreon campaign launched November2024 and paying supporters; I record one ongoing support programme, not each payment or each donation link. Hardware manufacturing/sponsor references do not establish cash investments. No conventional equity, debt or grant event was substantiated.',
'Cheshire-Labs': 'I searched Cheshire Labs, Orca and founder Michael Salmi for rounds, angels, grants/SBIR, accelerators, crowdfunding and bootstrapping; I checked the company site, LinkedIn identity and founder’s launch discussion. I found no disclosed outside funding. none_found is a search result, not evidence of no funding or a bootstrapping claim. Identity/launch corroboration: https://labautomation.io/t/open-source-scheduler/819?page=2 .',
'coqui-ai': 'I checked SEC filings/amendments, Forbes, investor ScaleX and funding databases. The October2021 notice reports USD1550000 sold, not its USD3000000 target; Mango Capital attribution and convertible classification come from CB Insights. The January2023 second note is database-only and may overlap March’s USD3.3m seed; I retain it at low confidence without imputing an amount. Shutdown is not funding. The early instrument/round linkage remains unresolved.',
'dbeaver': 'I checked the company’s first-round post, Headline/PTV, TechCrunch, databases and MassChallenge cohort/prize lists. April2023 USD6m is confirmed. I retain the reported 2021 PTV seed at low confidence, with no invented January month; it conflicts with the company calling April2023 its first round. I do not duplicate the March2023 database closing, and I exclude MassChallenge participation because no cash award was verified. The historical conflict remains unresolved.',
'e2b-dev': 'I checked E2B/Devbook, company and founder round announcements, financing counsel, CzechCrunch, investor Pioneer, databases and November2026 coverage. I record Pioneer’s undated investment, USD2.5m pre-seed, USD11.5m seed and USD21m SeriesA with event-specific funders. The USD406000 June2023 debt is PitchBook-only and uncorroborated. Conflicting lifetime totals/early predecessor capital remain unresolved; I do not subtract rounds from totals to invent another event.',
'Fincept-Corporation': 'I searched Fincept/Terminal, TLK Industries, Tilak Patel, grants and the named incubators; I checked company, FOSS United, GitHub Sponsors, PitchBook and Tracxn. PitchBook names three backers but neither maps them to its three undated incubation records nor establishes cash transfers; Tracxn calls the company unfunded. I preserve one low-confidence unallocated-backing row, not three invented cheques. Funding requests, a USD2000/month goal, private sponsors and First Wings participation do not establish discrete awards. Dates, amounts and cash/noncash status remain unresolved.',
'fosrl': 'I checked founder Milo Schwartz’s financing account, company announcement, YC, Chemistry and Tekedia. I consolidate the YC standard USD500000 package into one accelerator investment; its May2025 date is the acceptance/commitment month, not a payment receipt, so confidence is medium. The USD4.7m seed closed August but was publicly announced October2025; Tekedia’s December post joins that same seed. Conflicting USD125000/500000 database entries are not extra rounds. Nearly1000 donors are acknowledged without identifiable discrete payments; I do not infer a donor round.',
'GrapesJS': 'I checked creator/company posts, Open Core Ventures and the October2025 launch release, with seed/angel/grant/debt/follow-on searches. The June2024 posts substantiate OCV backing of Grapes Studio, but disclose neither stage nor amount; I use venture_unspecified and do not infer a lead. An unsupported USD2m directory estimate is withheld. The October2025 launch repeats the backing, not a second financing.',
'Kilo-Org': 'I checked the company announcement, CNBC, financing counsel Fenwick and early/follow-on/grant/debt searches. One USD8m December2025 seed is supported with Cota Capital as lead and four named firms. Sid Sijbrandij’s earlier founder capital is not an outside-investor angel event. I found no substantiated separate OCV or later round; acquisition consideration is excluded.',
'letta-ai': 'I checked Letta/MemGPT, Felicis, company press release/blog, TechCrunch and follow-on/grant/accelerator searches. September23/24 announcements describe one USD10m seed; the named angels belong to it, not separate invented rounds. I rejected an isolated alleged April2026 USD30m story: it has no reliable corroboration, inconsistent stage wording and an unresolvable purported source. Astasia Myers represents Felicis and is not a separate personal investor.',
'mindsdb': 'I checked SkyDeck, YC, investor portfolios, company releases, VentureBeat, Business Insider and 2025–2026 claims. I distinguish the2018 pre-seed,2020 seed,2021 incremental USD3.75m extension and three2023 announcements; I do not enter cumulative USD7.6m/41.5m/46.5m as rounds. The February2023 release calls its USD16.5m SeriesA, while later releases call the financing seed; I retain each source’s label. Early amounts/SkyDeck overlap remain unresolved. The alleged2025 USD47m SeriesC lacks credible corroboration and is withheld.',
'neuromore': 'I checked2015–2016 databases, SEC notices/amendments, company/grant searches and Republic. The two early seeds are low-confidence database disclosures with unnamed funders. I consolidate the debt/options filing and amendment to USD195000 sold rather than counting the initial USD40000 again. Republic’s USD30000 is explicitly reservations despite contradictory success wording, so I exclude it as unverified cash settlement. The company also describes USD750000 aggregate angel capital and unspecified grants; their individual dates/funders/overlap cannot be resolved and no cumulative extra round is invented.',
'odoo': 'I checked OpenERP/Odoo company releases, Summit, General Atlantic, AVP and contemporary reporting through2026. The2019–2026 transactions are secondary share sales, not all new company capital; sellers are not funders. The two January2026 events have different sellers/buyers: General Atlantic buying from Wallonie Entreprendre, and AVP/Alkeon buying management shares. Valuations are not amounts invested. I normalize SRIW to its current name Wallonie Entreprendre and retain public USD equivalents where reported. The first round is dated the company’s January2010 announcement.',
'openbq-org': 'I checked OpenBB/Gamestonk/Didier Lopes, the company-distributed March2022 seed release, OSS Capital, investor/follow-on searches and founder history. I record the USD8.5m announced seed and named participants once. A later history reports an August2021 founding seed of USD8.8m; whether this is a separate event, relabeling or revised amount remains unresolved, so I do not add both as confirmed independent rounds. No FOMO/valuation-derived capital or invented token sale is included.',
'payloadcms': 'I distinguished Payload CMS from the unrelated payments fintech; I checked Start Garden/Fox17, company YC/seed posts, Michigan awards and Figma acquisition. I record the2018 USD1000 finalist prize and USD20000 winner prize, explicit USD500000 YC investment, and USD4.7m seed. The conditional USD302500 Michigan award was cancelled in October2026 without disbursement, so it is excluded rather than counted as funding received. Figma’s acquisition/IPO is not a Payload round. Some seed participants are corroborated by Payload’s contemporaneous LinkedIn post.',
'redisson': 'I checked redisson.pro’s About page, founder interview, Crunchbase and targeted equity/seed/angel/accelerator/grant/debt/sponsorship searches. I found no disclosed outside funding; none_found is not proof of no funding and I found no sourced bootstrapping declaration. I excluded unrelated Redison music-device crowdfunding. Founder interview: https://www.alibabacloud.com/blog/interview-with-the-creator-of-redisson-building-an-open-source-enterprise-redis-client_593854 .',
'supermemoryai': 'I checked company/lead-investor announcements, TechCrunch, Buildspace’s grantee list, YC grant claims and follow-ons. I record the USD5000 Buildspace grant and one October2025 pre-seed: company says USD3m while Susa/TechCrunch report USD2.6m; I retain the company amount at medium confidence, not an extra USD400000 round. Co-leads/angels follow explicit reporting; employers are not investors. A founder interview says he got a YC Summer Fellows grant but chose company-building instead; actual acceptance/disbursement remains unclear, so I withhold that candidate and flag incompleteness.',
'TecharoHQ': 'I checked Techaro/Anubis funding documents, founder sustainability discussion, August2025 update, repository sponsors and January2026 interview. Cash donation/sponsorship support exists, so none_found would be wrong. The listed companies are disclosed sponsors, not necessarily equity investors; private/partial-name donors are not invented. I record one ongoing programme, not one fictional dated round per logo. Individual start dates, payment amounts and a finite capital total are undisclosed; software sales/consulting/subscription revenue are not financing.',
'd2lang': 'I checked Terrastruct/D2, founder posts, Bloomberg Beta, Fathom, Haystack and databases/follow-ons. One USD2m December2021 seed is reported. A founder post already describes funded operations in April2021; whether this is an earlier close of that seed or another financing remains unresolved. I do not invent a second amount or date. Lead designation is not disclosed. The venture remains incomplete for this early-round linkage, despite identifying the seed’s named firms.',
'getzep': 'I checked Zep company/YC, Engineering Capital, investor portfolios, funding databases and the2026 SPV filing. I record the roughly USD1.8m pre-seed at medium confidence; November/December2023 dates and convertible/seed descriptions conflict. The YC W24 investment is one event, not separate USD125000/500000 rounds; exact announcement month is unknown. DV backing is disclosed but its company financing date and primary/secondary structure are not. First Star/Step Function are retained as unallocated relationships, not an independently quantified new round. Early syndicate and2026 linkage remain incomplete.',
'anaconda': 'I checked Continuum/Anaconda, SEC filings, DARPA, SVB/PPP, Citi, Snowflake, BlackRock and2025–2026 releases. The2012 seed uses the final amended USD2250000 sold, not a second raise above the initial notice;2016/2017/2021 private placements are amounts sold, not offering ceilings. Their CSV months follow filing signatures/disclosures; first-sale months were August2016,June2017,May2021. BlackRock’s USD60m is an up-to financing commitment;2025 USD150m is a lower threshold including employee liquidity, not exact all-primary proceeds. Uncorroborated historical database rounds/secondary records remain gaps; I do not allocate unexplained cumulative totals.',
'antmicro': 'I checked company/NCBR/EU funding searches, CORDIS beneficiary records and the official EXCEED award sheet. I verified five distinct grants: AXIOM, X-MINE, VEDLIoT, EXCEED and TRISTAN. Amounts are Antmicro’s specific EU contributions, not consortium budgets; EXCEED’s EUR300000 is the grant-agreement maximum. Dates use grant-signature months, not later project starts. Consortium partners are not investors. No substantiated separate equity/debt/crowdfunding event was found.',
'bela': 'I checked Bela/Augmented Instruments, Kickstarter, original/archived Crowd Supply campaigns, maker profile and academic grant attribution. I record Bela,Trill,Gliss and Bela Gem campaigns; revised Trill sensors bundled with Gliss are not a new campaign. USD67951/64748 are disclosed campaign figures, not lifetime company financing. The2016 campaign predates/comes through the university project; legal company payees are not established by these pages, so the scope includes product/predecessor crowdfunding. Grants to Queen Mary/Andrew McPherson are not assigned to the company.',
'bcn3d': 'I checked company releases, CDTI/Innvierte records, Spanish reporting, ICEX and consortium disclosures. I record2019 seed,2020 round,2021/February2024 co-investments and November2024 funding once each; the January2021 EUR2.8m portal entry repeats2020. I add ICEX Next and VIVALDI/HIBRI4D public-project support without inventing award months or company allocations. The latter IDs show2021 call years only. I exclude bankruptcy/acquisition proceeds. Names are normalized from fund vehicles where established; unknown industrial investors remain unnamed.',
'bitcraze': 'I checked company history, Minc, PitchBook, preorders/crowdfunding and Vinnova. The Vinnova award names Bitcraze AB and SEK317289; July2022 is project start, not a verified announcement, so date is blank and start is in the label. I exclude Minc membership as unconfirmed cash support and ordinary preorder sales as not a disclosed crowdfunding campaign. Early reinvested sales do not prove absence of all outside funding.',
'efabless': 'I checked early notes,Dealroom/PitchBook,company releases/LinkedIn,S&P reporting and SEC. Three early notes and the2020 SeriesA are database-only;2020 amounts conflict, so its USD5m is low confidence. January/May2022 A-1 notices may be successive closes of one financing; I keep the explicitly disclosed closing labels and do not assume additivity. The2023 USD6.3m extension is distinct;2024 debt uses October filing disclosure. A HalfCourt lead could not be resolved. Google’s manufacturing programme sponsorship is not assumed unrestricted venture capital. Early history/closing linkage remain incomplete.',
'evolutionaryscale': 'I checked Forbes’ August2023 updated report, Amazon’s investor release,Reuters,TechCrunch and later financing searches. The Forbes body explicitly says the roughly/at-least USD40m Lux-led round was raised and names Nat Friedman/Daniel Gross; I distinguish it from the more-than USD142m June2024 seed. Some page metadata still says raising; confidence on2023 is medium. For2024 the two people are investors, not inferred firms; Amazon/NVentures are additional participants. No substantiated later round was found.',
'farmbot': 'I distinguished FarmBot Inc’s open-hardware robot from Australian Farmbot Monitoring. I checked capitalization,Shuttleworth,launch wrapup,public raw financial ledgers and vendor/lender identity. The USD125000 fellowship funded the predecessor project through founder Rory Aronson, not an equity cheque to the later corporation; I retain it with that recipient caveat. The bounded July2016 campaign raised USD813235 in preorder revenue, not equity. Completed Shopify/PayPal/Amex originations are separate debt records; refinancing gross proceeds are not net new money. Family debt/revolver origination details and broader historical account completeness remain unresolved.',
'open-bionics': 'I checked precursor crowdfunding,Intel/Disney/Dyson/Nominet/UAE awards,SBRI,UKRI current API,Crowdcube,NatWest,investors and original Companies House allotments. Low-confidence database entries may overlap other seed closes. Crowdcube totals include off-platform commitments and are not all verified cash received. UKRI figures are company-specific grant offers; source dates are project starts, so announcement dates remain blank. Four2025 cash-allotment tranches use share counts x GBP6.37, not nominal capital; duplicate/corrected filings are consolidated. Undated SBRI GBP697464 support may overlap phase reporting; I leave USD blank rather than invent a near-award exchange date. Historical share issues/round boundaries remain incomplete.',
'researchhub': 'I checked company2023 seed and2025 Boost VC posts,OSS Capital,Sora,SEC and token documentation. I record USD5m seed,August2023 Sora investment and additional USD2m Boost VC funding; company versus press seed/SeriesA labels do not create two rounds. Founder coin sales and research grants paid out by the platform are not outside funding into the venture. Token allocations/listing are not evidence of a token sale. Amountless/investorless additional database dates do not meet the event-source test and are not invented as extra rounds.',
'airbyte': 'I checked founder retrospectives,YC,Accel,company/investor round announcements,databases and follow-ons. The company explicitly describes USD600k January2020 pre-seed and USD1.2m May2020 seed; these are not inferred by subtracting later totals. YC W20 is retained separately but its exact investment month/amount and overlap with those early totals are unresolved, so both stay blank. I exclude Plug and Play participation because no cheque was established. Public2021 seed/A/B announcements supply event-specific funders; lifetime totals are not new events.',
'clearml': 'I checked Allegro AI predecessor and ClearML,company/investor announcements,Globes,Hyundai,Startup Nation Central and databases. I distinguish USD4m reported seed from USD7m2018 SeriesA; USD11m is cumulative, not another round. Hyundai’s2018 strategic extension is unquantified. Later records name the Israel Innovation Authority grant,July2024 backers and a November2025 SeriesA, but primary-round terms/investor allocations remain incompletely corroborated. I do not use unsupported USD15m/30m online claims or infer an accelerator cheque from a rebrand.',
'imply': 'I checked company posts,contemporary reporting,lead investors and later/debt/grant searches. I record USD2m seed,USD13.3mA,USD30mB,USD70mC and USD100mD with named round-specific participants. The first seed is unled in the source, not a fabricated Khosla lead. I exclude valuations,acquisition consideration and unnamed historical investors as new rounds. No additional substantiated financing was found in the reasonable search scope.',
'jfrog': 'I checked early Hebrew/English reports,company round releases,IPO closing/prospectus and2025–2026 filings. I distinguish the reported USD500k private-angel seed and USD1m Chief Scientist grant from SeriesA; exact announcement months for those earlier amounts are unknown. SeriesC’s source names five new investors but does not call all of them leads. IPO primary and selling-holder proceeds are separate rows computed from stated share counts x USD44; underwriters are not funders. Early angel attribution,grant timing and complete listed-company financing history remain unresolved.',
'meltano': 'I checked GitLab/company spinout release,GV,Venrock,TechCrunch and2025–2026 follow-ons/acquisition. The USD4.2m seed was announced June2021 despite April database financing dates. June2022 adds USD8.2m; USD12.4m is the combined total, not a new third round. The eventual Matatika acquisition is not funding into Meltano absent separately disclosed new capital. I retain only explicitly named outside participants, not GitLab incubation spend as a separate cheque.',
'nicira': 'I checked electronic FormD notices/amendments,two paper filings,contemporary investors and acquisition history. I consolidate amendments,notices and cents to whole dollars; the2008 USD1759441 event mixes preferred stock and USD200000 notes, not two additive copies. Dates follow public filing/signature months where verified,while labels identify earlier first sales. TheJanuary2009 scan reports positive sold capital but zero investor count,so confidence is medium. SeriesB investor attribution and overlapping early SeriesA closes remain unresolved; director names are not presumed investors.',
'pydantic': 'I checked Pydantic/Logfire company releases,Sequoia,TechCrunch and later/angel/grant/debt searches. I record one USD4.7m February2023 seed and one USD12.5m October2024 SeriesA. The personal seed investors are not copied into theA absent explicit participation. Event-specific firms follow contemporary announcements; framework downloads,product launches and valuations are not new funding. No other substantiated outside event was found.',
'radixark': 'I checked January2026 reporting,theMay company/investor announcement and named angels. I consolidate the January-reported Accel financing and formally announced May USD100m seed,not two USD100m rounds. Earlier Lip-Bu Tan angel backing is explicitly mentioned but its date/amount and overlap with the institutional financing remain uncertain. The May full syndicate names firms and people separately; SGLang’s academic origin/compute use is not automatically a company grant. The earlier angel/seed linkage keeps the history incomplete.',
'sysdig': 'I checked seed coverage,primary SeriesA–G releases,In-Q-Tel and post2021 follow-ons/debt/grant searches. I record the USD2.3m seed,seven pricedA–G events and the distinct May2019 strategic investment. SeriesF/G use series_f_plus while labels preserveF/G. Capital commitments,valuations and cumulative fundraising are not added as separate rounds; historical investors are included only when expressly named for that event. No later substantiated financing was found.',
'va-linux': 'I checked VA Research/VA Linux and successor VA Software/Geeknet,original announcements,prospectus/10-Ks,UMAX and credit facilities. I include same-issuer successor financings,not acquirer funding. The November1998 Sequoia andMarch1999 Intel announcements are separate SeriesA closings,followed by June/September SeriesB closes. Credit facility increases/renewals are distinctly labeled; amounts are commitments when undrawn. The IPO is gross primary proceeds including overallotment. UMAX timing,allocation among early closes and complete successor history remain unresolved; the values must not be treated as a verified summable lifetime total.'
}
assert set(base_notes)==set(identities)
print('Prepared concise research notes for',len(base_notes),'ventures.')

Prepared concise research notes for 40 ventures.


In [129]:
# Verify the headline amount source for each primary IPO includes the stated share-count calculation.
for _,r0 in audit[(audit.type.eq('other')&audit.label.str.contains('public offering|IPO',case=False))|((audit.key=='jfrog')&audit.type.eq('secondary'))].iterrows():
 print(r0['key'],r0['label'],r0['amount_usd'],r0['source_url'])
# Check source-only regulatory rows are not referencing an index without the funding data.
print('Index-only source rows:',audit[audit.source_url.str.contains('index.htm')][['key','label','source_url']].to_dict('records'))
# Export the specified schema in original venture order and chronological order within each venture.
order={key:n for n,key in enumerate(sample.key)}
audit['_order']=audit.key.map(order)
audit['_date_sort']=audit.date.replace('', '9999-99')
audit=audit.sort_values(['_order','_date_sort','label'],kind='stable').drop(columns=['_order','_date_sort']).reset_index(drop=True)
audit=audit[columns]
assert audit.key.nunique()==40
assert all(identities[k]==v for k,v in audit[['key','venture']].itertuples(index=False,name=None))
assert set(audit.type)<=allowed_types
assert audit.amount_usd.str.fullmatch(r'\d*').all()
assert audit.date.str.fullmatch(r'(?:\d{4}-(?:0[1-9]|1[0-2]))?').all()
assert audit.source_url.str.fullmatch(r'https?://\S+').all()
assert not audit.duplicated().any()
assert set(audit.confidence)<= {'high','medium','low'}
for r0 in audit.to_dict('records'):
 assert not ((set(r0['lead'].split(' | '))&set(r0['investors'].split(' | ')))-{''})
none=audit[audit.type.eq('none_found')]
assert all((audit.key==key).sum()==1 for key in none.key)
audit.to_csv('funders_audit.csv',index=False,encoding='utf-8',lineterminator='\n')
print('CSV validated:',audit.shape)
print(audit.groupby('key',sort=False).size().to_string())

jfrog IPO primary issuance including full overallotment; gross company proceeds 428350208 https://www.globenewswire.com/news-release/2020/09/18/2096064/0/en/JFrog-Announces-Closing-of-Its-Initial-Public-Offering-and-Exercise-in-Full-of-the-Underwriters-Option-to-Purchase-Additional-Shares.html
jfrog IPO selling-shareholder component; gross proceeds, none to company 157001592 https://www.globenewswire.com/news-release/2020/09/18/2096064/0/en/JFrog-Announces-Closing-of-Its-Initial-Public-Offering-and-Exercise-in-Full-of-the-Underwriters-Option-to-Purchase-Additional-Shares.html
va-linux Initial public offering including exercised overallotment; gross proceeds 151800000 https://www.sec.gov/Archives/edgar/data/1096199/000109581101505709/0001095811-01-505709.txt
Index-only source rows: []
CSV validated: (204, 10)
key
BruceDevices            1
Cheshire-Labs           1
coqui-ai                3
dbeaver                 2
e2b-dev                 5
Fincept-Corporation     1
fosrl               

In [130]:
# Every used source is cited through the mandated citation API; retain a local evidence archive for verification.
source_orgs={
 'sec.gov':'U.S. Securities and Exchange Commission','www.sec.gov':'U.S. Securities and Exchange Commission',
 'cordis.europa.eu':'European Commission','gtr.ukri.org':'UK Research and Innovation',
 'find-and-update.company-information.service.gov.uk':'Companies House',
 'www.globenewswire.com':'GlobeNewswire','www.prnewswire.com':'PR Newswire',
 'techcrunch.com':'TechCrunch','venturebeat.com':'VentureBeat','www.businesswire.com':'Business Wire',
 'pitchbook.com':'PitchBook','www.cbinsights.com':'CB Insights','app.dealroom.co':'Dealroom',
 'api.frankfurter.dev':'Frankfurter','meta.farm.bot':'FarmBot','farm.bot':'FarmBot',
 'www.cnmv.es':'CNMV','www.crowdsupply.com':'Crowd Supply','www.kickstarter.com':'Kickstarter',
 'www.crowdcube.com':'Crowdcube','www.gov.uk':'UK Government','www.vinnova.se':'Vinnova'
}
all_note_urls=set()
for k,n in base_notes.items():
 all_note_urls.update(re.findall(r'https?://[^\s<>]+',n))
for additions in note_additions.values():
 for n in additions:all_note_urls.update(u.rstrip('.,;)') for u in re.findall(r'https?://[^\s<>]+',n))
all_note_urls={u.rstrip('.,;)') for u in all_note_urls}
all_final_urls=set(audit.source_url)|all_note_urls
# Sources already registered by the principal agent retain their original metadata.
registered={d['url'] for d in own_evidence}
missing=all_final_urls-set(source_cache)
await asyncio.gather(*(get_audit_source(u) for u in missing))
for u in sorted(all_final_urls-registered):
 d=source_cache.get(u,{})
 text=d.get('text') or d.get('full_text') or d.get('fetched_text') or ''
 title=d.get('title') or (text.splitlines()[0] if text else u)
 if not title or title.startswith('{') or title in ['XML','10-K']:
  matching=audit[audit.source_url.eq(u)]
  title=(f"{matching.iloc[0]['venture']}: {matching.iloc[0]['label']}" if not matching.empty else title or u)
 domain=u.split('/')[2]
 author=source_orgs.get(domain,domain)
 retain({'url':u,'title':title,'full_text':text},author)
Path('audit_work/source_cache.json').write_text(json.dumps(source_cache,indent=2))
print('Final-used URLs:',len(all_final_urls),'registered total records:',len(own_evidence))
print('Unretrieved note sources:',[(u,source_cache.get(u,{}).get('error')) for u in all_final_urls if source_cache.get(u,{}).get('error')])

Final-used URLs: 235 registered total records: 267
Unretrieved note sources: []


In [131]:
# Final notes add explicit methodological limits without obscuring venture-specific research.
note_additions['open-bionics'].append('The first Crowdcube page is labelled December2022 while public company launch coverage is January2023; I count one campaign, using the platform’s month. The nominee/vehicle normalization does not identify underlying individual beneficial investors.')
note_additions['bela'].append('Campaign evidence is product/predecessor-level, not proof that Augmented Instruments Ltd legally received the2016 proceeds. The archived campaign figures are used rather than later shop sales.')
note_additions['payloadcms'].append('I retained the two cash competition awards as prize, not philanthropic grants. The cancelled Michigan commitment is excluded; its USD875000 planned project investment was never an outside funding amount.')
note_additions['nicira'].append('The cited amended SeriesB disclosure is dated September2010 even though first sale was December2009; the paperJanuary2009 sale was publicly filed in February. Round allocations are not reconstructed from cap-table director affiliations.')
note_additions['anaconda'].append('The January2012 seed date identifies its first public notice; the amount is final amended disclosure in August2012. The2022 BlackRock amount is an up-to commitment and the2025 SeriesC headline is over USD150m; numeric fields retain those stated thresholds.')
# These are substantive caveats, not newly invented events.
summary={
 'as_of':'2026-11-26','ventures_researched':40,'csv_rows':len(audit),
 'funding_records_including_provisional':int((audit.type!='none_found').sum()),
 'none_found_ventures':int(none.key.nunique()),
 'incomplete_ventures':sum(bool(allnotes[k]['incomplete']) for k in identities),
 'incomplete_keys':[k for k in identities if allnotes[k]['incomplete']],
 'confidence_counts':audit.confidence.value_counts().to_dict(),
 'blank_dates':int(audit.date.eq('').sum()),'blank_amounts':int(audit.amount_usd.eq('').sum()),
 'currency_conversions':len(fx_jobs)
}
header=f'''# Outside-funding audit notes

Research cutoff: **{summary['as_of']}**. I researched **40 ventures** and wrote **{len(audit)} CSV rows**, including **2 none_found rows**. **20 venture histories remain INCOMPLETE** because material early rounds, cash receipt, investor allocation or overlapping disclosures could not be settled; this does not mean those ventures were skipped. The other 20 received reasonable research coverage, not a guarantee that undisclosed funding does not exist.

I use only the requested ten CSV columns and the deal_rules.md event types. I preserve input keys/names exactly. Low confidence means a disclosed but weakly corroborated record, not confirmed receipt. Unallocated-backer rows preserve named funding relationships without inventing a round. Do not sum this file as verified lifetime capital: it includes possible overlapping closings, credit commitments, secondary sales, product/predecessor campaigns and ongoing-support programmes. I exclude unverified solicitations/reservations, ordinary sales, noncash incubator membership, acquisitions, founder capital and a cancelled unpaid award.

Dates are public announcement/disclosure months when established; blanks mean the month cannot be supported. Explicitly labelled campaign/commitment dates are distinguished below. Grants with only project-start dates retain those starts in labels, not fabricated announcement dates. Amounts are nominal whole USD, rounded half-up where necessary; blanks are not zero. For {len(fx_jobs)} non-USD amounts I use a historical Frankfurter v1 rate on the 15th of the disclosed month (or the returned preceding business day); project-start month is used only for conversion where award-announcement timing is unavailable. Original amounts, exact rate dates and endpoints are recorded below. This is a transparent near-date approximation, not an exact transaction-date conversion. Partial investor tickets are not substituted for an unknown whole-round total. USD headline lower/upper thresholds and campaign off-platform commitments remain qualified in labels/notes. Named sponsors are not necessarily equity holders; IPO underwriters are not treated as investors.

## One line per venture
'''
lines=[]
for key in sample.key:
 flag='INCOMPLETE' if allnotes[key]['incomplete'] else 'REASONABLE SEARCH COMPLETE'
 if key in set(none.key):flag='NONE_FOUND — REASONABLE SEARCH COMPLETE'
 adds=' '.join(note_additions[key])
 text=' '.join((base_notes[key]+' '+adds).split())
 lines.append(f'- **{key} — {identities[key]}** — **{flag}** — {text}')
notes_text=header+'\n'.join(lines)+'\n'
Path('funders_audit_notes.md').write_text(notes_text,encoding='utf-8')
Path('audit_work/audit_summary.json').write_text(json.dumps(summary,indent=2))
Path('audit_work/reconciliation_log.json').write_text(json.dumps(changes,indent=2))
print(json.dumps(summary,indent=2))
print('Notes length',len(notes_text),'chars; venture note lines',len(lines))

{
  "as_of": "2026-11-26",
  "ventures_researched": 40,
  "csv_rows": 204,
  "funding_records_including_provisional": 202,
  "none_found_ventures": 2,
  "incomplete_ventures": 20,
  "incomplete_keys": [
    "coqui-ai",
    "dbeaver",
    "e2b-dev",
    "Fincept-Corporation",
    "mindsdb",
    "neuromore",
    "openbq-org",
    "supermemoryai",
    "d2lang",
    "getzep",
    "anaconda",
    "efabless",
    "farmbot",
    "open-bionics",
    "airbyte",
    "clearml",
    "jfrog",
    "nicira",
    "radixark",
    "va-linux"
  ],
  "confidence_counts": {
    "high": 137,
    "medium": 53,
    "low": 14
  },
  "blank_dates": 28,
  "blank_amounts": 44,
  "currency_conversions": 31
}
Notes length 37428 chars; venture note lines 40


In [132]:
# Final read-back checks and required deliverable inventory.
final=pd.read_csv('funders_audit.csv',dtype=str,keep_default_na=False)
assert list(final.columns)==['key','venture','type','label','date','amount_usd','lead','investors','source_url','confidence']
assert len(final)==204 and final.key.nunique()==40
assert set(final.key)==set(sample.key)
assert all(identities[k]==v for k,v in zip(final.key,final.venture))
assert final.type.isin(allowed_types).all()
assert final.confidence.isin(['high','medium','low']).all()
assert final.amount_usd.str.fullmatch(r'[0-9]*').all()
assert final.date.str.fullmatch(r'(?:[0-9]{4}-(?:0[1-9]|1[0-2]))?').all()
assert final.source_url.str.match(r'^https?://').all()
assert not final.source_url.str.contains(' | ',regex=False).any()
assert not final.duplicated().any()
notes_lines=[line for line in Path('funders_audit_notes.md').read_text().splitlines() if line.startswith('- **')]
assert len(notes_lines)==40
assert sum('**INCOMPLETE**' in line for line in notes_lines)==20
assert all(any(line.startswith(f'- **{key} — {identities[key]}**') for line in notes_lines) for key in sample.key)
assert len(final[final.type.eq('none_found')])==2
# Every final source URL was retrieved and registered by the principal agent or a cited retrieval pass.
registered_urls={d['url'] for d in own_evidence}
assert set(final.source_url)<=registered_urls
manifest={'files':[
 {'path':'/workspace/funders_audit.csv','description':'Funding-event audit for all 40 sampled ventures; 204 rows with rule-compliant types, funder names, sources and confidence.','kind':'final'},
 {'path':'/workspace/funders_audit_notes.md','description':'One research note per venture, search scope, 20 incomplete-history flags, exclusions and original-currency conversion details.','kind':'final'},
 {'path':'/workspace/audit_work/reconciliation_log.json','description':'Supporting audit trail for source, classification, date and inclusion decisions.','kind':'intermediate'},
 {'path':'/workspace/audit_work/source_cache.json','description':'Supporting retrieved web-source text and metadata.','kind':'intermediate'},
 {'path':'/workspace/audit_work/audit_summary.json','description':'Supporting validation counts and incomplete venture keys.','kind':'intermediate'}
]}
Path('.deliverables.json').write_text(json.dumps(manifest,indent=2))
print('All final checks passed.')
print(final.groupby('confidence').size().to_string())
print('None found:',final.loc[final.type.eq('none_found'),'venture'].tolist())
print('Final files:',[(p,Path(p).stat().st_size) for p in ['funders_audit.csv','funders_audit_notes.md']])

All final checks passed.
confidence
high      137
low        14
medium     53
None found: ['Cheshire Labs', 'Redisson']
Final files: [('funders_audit.csv', 46074), ('funders_audit_notes.md', 37744)]
